In [ ]:
# @title
# Install required packages
!pip install -q torch torchvision
!pip install -q torchgeo==0.7.0
!pip install -q timm
!pip install -q rasterio tifffile
!pip install -q huggingface_hub
!pip install -q einops
!pip install -q numpy pillow matplotlib

print("✅ All packages installed successfully!")

# Computing Embeddings

In [ ]:
# 1. Clone SAR-JEPA
!git clone https://github.com/waterdisappear/SAR-JEPA.git

# 2. Clone MSFA (SARDet-100K)
!git clone https://github.com/zcablii/SARDet_100K.git

!pip install -r SARDet_100K/MSFA/requirements.txt


In [ ]:
# Install required packages
!pip install -q torch torchvision
!pip install -q torchgeo==0.7.0
!pip install -q timm
!pip install -q rasterio tifffile
!pip install -q huggingface_hub
!pip install -q einops
!pip install -q numpy pillow matplotlib
!pip install -q mmengine==0.8.4

print("✅ All packages installed successfully!")

In [ ]:
import os
import torch
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

# Define class frequency groups
CLASS_GROUPS = {
    'opensarship': {
        'frequent': ['Cargo', 'Tanker'],           # >500 samples
        'medium': ['Dredging', 'Fishing'],         # 100-500
        'rare': ['Passenger', 'Tug']               # <100
    },
    'fusarship': {
        'frequent': ['Cargo', 'Fishing', 'Bulk'],
        'medium': ['Tanker', 'Container'],
        'rare': ['Dredging', 'Tug', 'GeneralCargo', 'Passenger']
    }
}

# Model configurations (native dimensions & input channels)
MODEL_CONFIGS = {
    'dofa': {'native_dim': 768, 'input_channels': 2},
    'ssl4eo': {'native_dim': 384, 'input_channels': 2},
    'scalemae': {'native_dim': 1024, 'input_channels': 3},
    'prithvi': {'native_dim': 768, 'input_channels': 6},
    'saratrx': {'input_channels': 1, 'native_dim': 768},
    'sarjepa': {'input_channels': 1, 'native_dim': 768},
    'sardet100k': {'input_channels': 3, 'native_dim': 2048},
}



# Create output directories
OUTPUT_ROOT = Path('/content/drive/MyDrive/PhD Research/iclrFoundationExp')

jepa_path = OUTPUT_ROOT / "foundation_models/weights/checkpoint-200.pth"
msfa_path = OUTPUT_ROOT / "foundation_models/weights/r50_sar_wavelet_epoch_100.pth"
for subdir in ['original', 'harmonized_768', 'experiments/tier1_baseline',
               'experiments/tier2_imbalance_groups', 'experiments/tier3_ensemble']:
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print("✅ Directories created")

In [ ]:
class SARDataset(Dataset):
    def __init__(self, root_path, dataset_name, split):
        """
        root_path: 'datasets/dataset1' or 'datasets/dataset2'
        dataset_name: 'opensarship' or 'fusarship'
        split: 'train', 'val', 'test'
        """
        self.root = Path(root_path) / split
        self.dataset_name = dataset_name
        self.split = split

        # Collect all .tif/.tiff files
        self.samples = []
        for class_dir in sorted(self.root.iterdir()):
            if not class_dir.is_dir():
                continue
            class_name = class_dir.name
            for img_path in class_dir.glob('*.tif*'):
                self.samples.append({
                    'path': str(img_path),
                    'label': class_name,
                    'group': self._get_group(class_name)
                })

        print(f"📂 {dataset_name}/{split}: {len(self.samples)} samples")

    def _get_group(self, class_name):
        """Assign frequency group"""
        groups = CLASS_GROUPS[self.dataset_name]
        for group, classes in groups.items():
            if class_name in classes:
                return group
        return 'unknown'

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = Image.open(sample['path'])
        img = img.resize((224, 224), Image.BILINEAR)
        img = np.array(img, dtype=np.float32)

        # Normalize to [0, 1]
        if img.max() > 1:
            img = img / 255.0

        # Handle single-channel → (H, W) → (1, H, W)
        if img.ndim == 2:
            img = img[np.newaxis, ...]

        # Convert to tensor
        img = torch.from_numpy(img)

        return img, sample['label'], sample['group'], sample['path']

# Test loader
test_ds = SARDataset('/content/drive/MyDrive/PhD Research/iclrFoundationExp/datasets/opensarship', 'opensarship', 'val')
print(f"✅ Sample shape: {test_ds[0][0].shape}")

In [ ]:
# ============================================================
# CELL 3: HARMONIZATION LAYER
# ============================================================

class EmbeddingHarmonizer(torch.nn.Module):
    """Project any dimension to 768"""
    def __init__(self, input_dim, output_dim=768):
        super().__init__()
        if input_dim != output_dim:
            self.proj = torch.nn.Linear(input_dim, output_dim)
        else:
            self.proj = torch.nn.Identity()

    def forward(self, x):
        return self.proj(x)

def harmonize_embeddings(embeddings, model_name):
    """Project embeddings to 768-dim"""
    native_dim = MODEL_CONFIGS[model_name]['native_dim']
    harmonizer = EmbeddingHarmonizer(native_dim, 768).to(device)
    harmonizer.eval()

    with torch.no_grad():
        embeddings_tensor = torch.from_numpy(embeddings).to(device)
        harmonized = harmonizer(embeddings_tensor).cpu().numpy()

    return harmonized

def save_embeddings(embeddings, labels, groups, paths, save_path):
    """Save as compressed .npz"""
    np.savez_compressed(
        save_path,
        embeddings=embeddings,
        labels=np.array(labels),
        groups=np.array(groups),
        paths=np.array(paths)
    )
    print(f"💾 Saved: {save_path} ({embeddings.shape})")

print("✅ Harmonizer ready")

In [ ]:
# @title
# # ============================================================
# # CELL 4: MODEL LOADING & EXTRACTION PIPELINE
# # ============================================================
# from torchgeo.models import (
#     dofa_base_patch16_224,
#     DOFABase16_Weights,
#     vit_small_patch16_224,
#     vit_base_patch16_224,
#     scalemae_large_patch16,
#     ScaleMAELarge16_Weights
# )
# from huggingface_hub import hf_hub_download


# def load_all_models():
#     """Load all 4 foundation models"""
#     models = {}

#     # MODEL 1: DOFA
#     print("Loading DOFA...")
#     from torchgeo.models import dofa_base_patch16_224, DOFABase16_Weights
#     dofa_model = dofa_base_patch16_224(weights=DOFABase16_Weights.DOFA_MAE)
#     dofa_model = dofa_model.to(device)
#     dofa_model.eval()
#     models['dofa'] = dofa_model
#     print("✅ DOFA loaded")

#     # MODEL 2: SSL4EO-S12
#     print("Loading SSL4EO-S12...")
#     from huggingface_hub import hf_hub_download
#     from torchgeo.models import vit_small_patch16_224
#     weights_path = hf_hub_download(repo_id="wangyi111/SSL4EO-S12", filename="B2_vits16_mae_ep99_enc.pth")
#     ssl4eo_model = vit_small_patch16_224(weights=None, in_chans=2, num_classes=0)
#     state_dict = torch.load(weights_path, map_location=device)
#     if 'model' in state_dict:
#         state_dict = state_dict['model']
#     elif 'state_dict' in state_dict:
#         state_dict = state_dict['state_dict']
#     state_dict = {k.replace('encoder.', ''): v for k, v in state_dict.items()}
#     ssl4eo_model.load_state_dict(state_dict, strict=False)
#     ssl4eo_model = ssl4eo_model.to(device)
#     ssl4eo_model.eval()
#     models['ssl4eo'] = ssl4eo_model
#     print("✅ SSL4EO-S12 loaded")

#     # MODEL 3: ScaleMAE
#     print("Loading ScaleMAE...")
#     from torchgeo.models import scalemae_large_patch16, ScaleMAELarge16_Weights
#     try:
#         available_weights = [w for w in dir(ScaleMAELarge16_Weights) if not w.startswith('_')]
#         weight_enum = getattr(ScaleMAELarge16_Weights, available_weights[0])
#         scalemae_temp = scalemae_large_patch16(weights=weight_enum)
#         scalemae_model = scalemae_large_patch16(weights=None, num_classes=0)
#         state_dict = scalemae_temp.state_dict()
#         state_dict = {k: v for k, v in state_dict.items() if 'head' not in k}
#         scalemae_model.load_state_dict(state_dict, strict=False)
#         scalemae_model = scalemae_model.to(device)
#         scalemae_model.eval()
#         models['scalemae'] = scalemae_model
#         print("✅ ScaleMAE loaded")
#     except Exception as e:
#         print(f"⚠️ Using ScaleMAE with classification head: {e}")
#         scalemae_model = scalemae_large_patch16(weights=ScaleMAELarge16_Weights.FMOW_RGB)
#         scalemae_model = scalemae_model.to(device)
#         scalemae_model.eval()
#         models['scalemae'] = scalemae_model
#         MODEL_CONFIGS['scalemae']['native_dim'] = 1000

#     # MODEL 4: Prithvi-100M
#     print("Loading Prithvi-100M...")
#     from huggingface_hub import hf_hub_download
#     from torchgeo.models import vit_base_patch16_224
#     weights_path = hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-EO-1.0-100M", filename="Prithvi_100M.pt")
#     prithvi_model = vit_base_patch16_224(weights=None, in_chans=6, num_classes=0)
#     checkpoint = torch.load(weights_path, map_location=device)
#     if 'model' in checkpoint:
#         state_dict = checkpoint['model']
#     elif 'state_dict' in checkpoint:
#         state_dict = checkpoint['state_dict']
#     else:
#         state_dict = checkpoint
#     state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
#     prithvi_model.load_state_dict(state_dict, strict=False)
#     prithvi_model = prithvi_model.to(device)
#     prithvi_model.eval()
#     models['prithvi'] = prithvi_model
#     print("✅ Prithvi-100M loaded")

#     return models

# def extract_embeddings(model, model_name, dataloader, input_channels):
#     """Extract embeddings with dynamic channel adaptation"""
#     all_embeddings, all_labels, all_groups, all_paths = [], [], [], []
#     model.eval()

#     with torch.no_grad():
#         for batch_imgs, labels, groups, paths in tqdm(dataloader, desc=f"{model_name}"):
#             batch_imgs = batch_imgs.to(device)

#             # Dynamic channel adaptation based on model requirements
#             current_channels = batch_imgs.shape[1]

#             # Step 1: Convert 1-channel to required channels
#             if current_channels == 1:
#                 if input_channels == 2:
#                     batch_imgs = batch_imgs.repeat(1, 2, 1, 1)
#                 elif input_channels == 3:
#                     batch_imgs = batch_imgs.repeat(1, 3, 1, 1)
#                 elif input_channels == 6:
#                     batch_imgs = batch_imgs.repeat(1, 6, 1, 1)

#             # Step 2: Convert 2-channel to 3 or 6 if needed
#             elif current_channels == 2:
#                 if input_channels == 3:
#                     batch_imgs = torch.cat([batch_imgs, batch_imgs.mean(dim=1, keepdim=True)], dim=1)
#                 elif input_channels == 6:
#                     batch_imgs = torch.cat([batch_imgs] * 3, dim=1)

#             # Extract embeddings based on model type
#             if model_name == 'dofa':
#                 wavelengths = [56000, 56000]
#                 try:
#                     output = model.forward_features(batch_imgs, wavelengths)
#                 except:
#                     output = model(batch_imgs, wavelengths)
#             else:
#                 output = model(batch_imgs)

#             # Handle different output shapes
#             if output.ndim == 3:  # (batch, patches, dim)
#                 embeddings = output[:, 0, :].cpu().numpy()
#             elif output.ndim == 2:  # (batch, dim)
#                 embeddings = output.cpu().numpy()
#             else:
#                 raise ValueError(f"Unexpected output shape for {model_name}: {output.shape}")

#             all_embeddings.append(embeddings)
#             all_labels.extend(labels)
#             all_groups.extend(groups)
#             all_paths.extend(paths)

#             del batch_imgs, output, embeddings
#             torch.cuda.empty_cache()

#     embeddings_array = np.vstack(all_embeddings)
#     print(f"   ✅ Extracted embeddings shape: {embeddings_array.shape}")

#     return embeddings_array, all_labels, all_groups, all_paths

# def process_all_models(models):
#     datasets_config = [('/content/drive/MyDrive/PhD Research/iclrFoundationExp/datasets/opensarship', 'opensarship'),
#                        ('/content/drive/MyDrive/PhD Research/iclrFoundationExp/datasets/fusarship', 'fusarship')]
#     for model_name, model in models.items():
#         print(f"\n{'='*60}\n{model_name.upper()}\n{'='*60}")
#         for root_path, dataset_name in datasets_config:
#             for split in ['train', 'val', 'test']:
#                 tier1_check = OUTPUT_ROOT / 'experiments/tier1_baseline' / model_name / f'{dataset_name}_{split}.npz'
#                 if tier1_check.exists():
#                     print(f"⏭️ Skipping {dataset_name}/{split} - already processed")
#                     continue

#                 print("Dataset", dataset_name, "/", split)
#                 dataset = SARDataset(root_path, dataset_name, split)
#                 # loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=2)
#                 loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
#                 input_ch = MODEL_CONFIGS[model_name]['input_channels']
#                 embeddings, labels, groups, paths = extract_embeddings(model, model_name, loader, input_ch)

#                 # Save original
#                 save_path = OUTPUT_ROOT / 'original' / model_name / f'{dataset_name}_{split}.npz'
#                 save_path.parent.mkdir(parents=True, exist_ok=True)
#                 save_embeddings(embeddings, labels, groups, paths, save_path)

#                 # Save harmonized
#                 embeddings_768 = harmonize_embeddings(embeddings, model_name)
#                 save_path_h = OUTPUT_ROOT / 'harmonized_768' / model_name / f'{dataset_name}_{split}.npz'
#                 save_path_h.parent.mkdir(parents=True, exist_ok=True)
#                 save_embeddings(embeddings_768, labels, groups, paths, save_path_h)

#                 # Tier 1
#                 tier1_path = OUTPUT_ROOT / 'experiments/tier1_baseline' / model_name / f'{dataset_name}_{split}.npz'
#                 tier1_path.parent.mkdir(parents=True, exist_ok=True)
#                 save_embeddings(embeddings_768, labels, groups, paths, tier1_path)

#                 # Tier 2
#                 for group in ['frequent', 'medium', 'rare']:
#                     mask = np.array(groups) == group
#                     if mask.sum() > 0:
#                         tier2_path = OUTPUT_ROOT / 'experiments/tier2_imbalance_groups' / model_name / f'{dataset_name}_{group}_{split}.npz'
#                         tier2_path.parent.mkdir(parents=True, exist_ok=True)
#                         save_embeddings(embeddings_768[mask], [labels[i] for i in np.where(mask)[0]],
#                                       [groups[i] for i in np.where(mask)[0]], [paths[i] for i in np.where(mask)[0]], tier2_path)

#                 # Free memory after processing each split
#                 del embeddings, embeddings_768, labels, groups, paths, dataset, loader
#                 torch.cuda.empty_cache()

#     # Tier 3
#     print("\n🔗 Creating Tier 3 ensemble files...")
#     for dataset_name in ['opensarship', 'fusarship']:
#         for split in ['train', 'val', 'test']:
#             combined = {}
#             for model_name in models.keys():
#                 path = OUTPUT_ROOT / 'harmonized_768' / model_name / f'{dataset_name}_{split}.npz'
#                 data = np.load(path)
#                 combined[f'{model_name}_embeddings'] = data['embeddings']
#                 if 'labels' not in combined:
#                     combined['labels'] = data['labels']
#                     combined['groups'] = data['groups']
#                     combined['paths'] = data['paths']
#             tier3_path = OUTPUT_ROOT / 'experiments/tier3_ensemble' / f'{dataset_name}_{split}_all_models.npz'
#             tier3_path.parent.mkdir(parents=True, exist_ok=True)
#             np.savez_compressed(tier3_path, **combined)
#             print(f"💾 {tier3_path}")
#     print("\n✅ ALL DONE!")

# # Load models and run
# # models = load_all_models()

In [ ]:
# process_all_models(models)

## Speed up cell 4

In [ ]:
# @title
# ============================================================
# CELL 4 (UPDATED): MODEL LOADING & EXTRACTION PIPELINE
# WITH CHECKPOINTING & DYNAMIC BATCHING
# Minimal changes - preserves all existing logic
# ============================================================

import time
import gc
from torchgeo.models import (
    dofa_base_patch16_224,
    DOFABase16_Weights,
    vit_small_patch16_224,
    vit_base_patch16_224,
    scalemae_large_patch16,
    ScaleMAELarge16_Weights
)
from huggingface_hub import hf_hub_download


# ============================================================
# NEW: Checkpoint Management (Lines 27-80)
# ============================================================

def load_checkpoint(checkpoint_path):
    """Load existing checkpoint and return accumulated data."""
    if not checkpoint_path.exists():
        return None

    try:
        checkpoint = np.load(checkpoint_path)
        return {
            'embeddings': checkpoint['embeddings'],
            'labels': list(checkpoint['labels']),
            'groups': list(checkpoint['groups']),
            'paths': list(checkpoint['paths']),
            'last_batch_idx': int(checkpoint['last_batch_idx']),
            'total_batches': int(checkpoint['total_batches'])
        }
    except Exception as e:
        print(f"⚠️ Checkpoint corrupted: {e}. Starting fresh.")
        return None


def save_checkpoint(checkpoint_path, embeddings, labels, groups, paths, batch_idx, total_batches):
    """Save checkpoint with accumulated embeddings and metadata."""
    try:
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            checkpoint_path,
            embeddings=embeddings.astype(np.float32),
            labels=np.array(labels, dtype=str),
            groups=np.array(groups, dtype=str),
            paths=np.array(paths, dtype=str),
            last_batch_idx=np.array(batch_idx, dtype=np.int32),
            total_batches=np.array(total_batches, dtype=np.int32)
        )
    except Exception as e:
        print(f"⚠️ Checkpoint save failed: {e}")


# ============================================================
# NEW: Dynamic Batch Sizing (Lines 85-120)
# ============================================================

def get_dynamic_batch_size(model_name, dataset_size):
    """
    Calculate safe batch size based on model and dataset.
    Uses functions from Cell 1 if available, otherwise fallback.
    """
    try:
        # Try to use Cell 1 function if available
        return get_safe_batch_size(model_name, dataset_size)
    except:
        # Fallback: Conservative batch sizing
        # model_weights_gb = {
        #     'scalemae': 1.13,
        #     'dofa': 0.425,
        #     'prithvi': 0.350,
        #     'ssl4eo': 0.086
        # }
        model_weights_gb = {
            'scalemae': 0.513,
            'dofa': 0.125,
            'prithvi': 0.150,
            'ssl4eo': 0.086,
            'saratrx': 0.120,
            'sarjepa': 0.043,
            'sardet100k': 0.043,
        }


        base_batch = 32
        ssl4eo_weight = model_weights_gb['ssl4eo']
        model_weight = model_weights_gb.get(model_name, 0.5)
        model_factor = ssl4eo_weight / model_weight

        if dataset_size < 500:
            dataset_factor = 2.0
        elif dataset_size < 2000:
            dataset_factor = 1.5
        else:
            dataset_factor = 1.0

        batch_size = int(base_batch * model_factor * dataset_factor)
        batch_size = max(1, min(batch_size, 128))  # Conservative cap at 128

        return batch_size


# ============================================================
# EXISTING: load_all_models (unchanged except error handling)
# ============================================================

def load_all_models():
    """Load all 4 foundation models"""
    models = {}

    # MODEL 1: DOFA
    print("Loading DOFA...")
    try:
        dofa_model = dofa_base_patch16_224(weights=DOFABase16_Weights.DOFA_MAE)
        dofa_model = dofa_model.to(device)
        dofa_model.eval()
        models['dofa'] = dofa_model
        print("✅ DOFA loaded")
    except Exception as e:
        print(f"❌ DOFA failed: {e}")
        return None

    # MODEL 2: SSL4EO-S12
    print("Loading SSL4EO-S12...")
    try:
        weights_path = hf_hub_download(repo_id="wangyi111/SSL4EO-S12", filename="B2_vits16_mae_ep99_enc.pth")
        ssl4eo_model = vit_small_patch16_224(weights=None, in_chans=2, num_classes=0)
        state_dict = torch.load(weights_path, map_location=device, weights_only=False)
        if 'model' in state_dict:
            state_dict = state_dict['model']
        elif 'state_dict' in state_dict:
            state_dict = state_dict['state_dict']
        state_dict = {k.replace('encoder.', ''): v for k, v in state_dict.items()}
        ssl4eo_model.load_state_dict(state_dict, strict=False)
        ssl4eo_model = ssl4eo_model.to(device)
        ssl4eo_model.eval()
        models['ssl4eo'] = ssl4eo_model
        print("✅ SSL4EO-S12 loaded")
    except Exception as e:
        print(f"❌ SSL4EO-S12 failed: {e}")
        return None

    # MODEL 3: ScaleMAE
    print("Loading ScaleMAE...")
    try:
        available_weights = [w for w in dir(ScaleMAELarge16_Weights) if not w.startswith('_')]
        weight_enum = getattr(ScaleMAELarge16_Weights, available_weights[0])
        scalemae_temp = scalemae_large_patch16(weights=weight_enum)
        scalemae_model = scalemae_large_patch16(weights=None, num_classes=0)
        state_dict = scalemae_temp.state_dict()
        state_dict = {k: v for k, v in state_dict.items() if 'head' not in k}
        scalemae_model.load_state_dict(state_dict, strict=False)
        scalemae_model = scalemae_model.to(device)
        scalemae_model.eval()
        models['scalemae'] = scalemae_model
        print("✅ ScaleMAE loaded")
    except Exception as e:
        print(f"⚠️ Using ScaleMAE with classification head: {e}")
        try:
            scalemae_model = scalemae_large_patch16(weights=ScaleMAELarge16_Weights.FMOW_RGB)
            scalemae_model = scalemae_model.to(device)
            scalemae_model.eval()
            models['scalemae'] = scalemae_model
            MODEL_CONFIGS['scalemae']['native_dim'] = 1000
            print("✅ ScaleMAE loaded (with head)")
        except Exception as e2:
            print(f"❌ ScaleMAE failed: {e2}")
            return None

    # MODEL 4: Prithvi-100M
    print("Loading Prithvi-100M...")
    try:
        weights_path = hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-EO-1.0-100M", filename="Prithvi_100M.pt")
        prithvi_model = vit_base_patch16_224(weights=None, in_chans=6, num_classes=0)
        checkpoint = torch.load(weights_path, map_location=device)
        if 'model' in checkpoint:
            state_dict = checkpoint['model']
        elif 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        else:
            state_dict = checkpoint
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        prithvi_model.load_state_dict(state_dict, strict=False)
        prithvi_model = prithvi_model.to(device)
        prithvi_model.eval()
        models['prithvi'] = prithvi_model
        print("✅ Prithvi-100M loaded")
    except Exception as e:
        print(f"❌ Prithvi-100M failed: {e}")
        return None

    # MODEL 6: SAR-JEPA
    print("\nLoading SAR-JEPA...")
    np.float = float
    np.int = int
    np.object = object
    np.bool = bool
    try:
        import sys
        # sys.path.insert(0, 'sar_jepa/Pretraining')  # Update to your path
        sys.path.append('/content/sar_jepa/Pretraining')

        from models_lomar import mae_vit_base_patch16_dec512d8b

        sarjepa_weights = jepa_path  # Update path

        # Create model
        sarjepa_model = mae_vit_base_patch16_dec512d8b(in_chans=1)

        if Path(sarjepa_weights).exists():
            # Load checkpoint
            checkpoint = torch.load(sarjepa_weights, map_location=device, weights_only=False)
            state_dict = checkpoint.get('model', checkpoint.get('state_dict', checkpoint))

            # Remove "module." prefix and filter to encoder only
            encoder_state = {}
            for k, v in state_dict.items():
                new_key = k.replace('module.', '', 1)

                # Skip decoder, predictor, and SAR training features
                if any(x in new_key for x in ['mask_token', 'decoder', 'sarfeature',
                                              'encoder_pred', 'decoder_pred',
                                              'decoder_blocks', 'decoder_norm',
                                              'hogs', 'edge']):
                    continue

                encoder_state[new_key] = v

            # Load weights
            sarjepa_model.load_state_dict(encoder_state, strict=False)
            sarjepa_model = sarjepa_model.to(device)
            sarjepa_model.eval()
            models['sarjepa'] = sarjepa_model
            print(f"✅ SAR-JEPA loaded ({len(encoder_state)} encoder params)")
        else:
            print(f"⚠️ Weights not found: {sarjepa_weights}")

    except Exception as e:
        print(f"❌ SAR-JEPA failed: {e}")
        import traceback
        traceback.print_exc()

    # MODEL 7: SARDet-100K (MSFA)
    print("\nLoading SARDet-100K (MSFA)...")
    try:
        import torchvision.models as tv_models

        sardet_weights = msfa_path

        if Path(sardet_weights).exists():
            # Load checkpoint
            checkpoint = torch.load(sardet_weights, map_location=device, weights_only=False)
            full_state = checkpoint.get('state_dict', checkpoint)

            # Extract backbone with SAR channel adaptation
            backbone_state = {}
            for k, v in full_state.items():
                if k.startswith('backbone.backbone.'):
                    # Remove double-nested prefix
                    new_key = k.replace('backbone.backbone.', '', 1)

                    # SPECIAL: Adapt conv1 from 82 channels to 3 channels
                    if new_key == 'conv1.weight':
                        # Extract SAR channel (first channel) and repeat to 3
                        sar_channel = v[:, 0:1, :, :]  # [64, 1, 7, 7]
                        adapted_conv1 = sar_channel.repeat(1, 3, 1, 1)  # [64, 3, 7, 7]
                        backbone_state[new_key] = adapted_conv1
                    else:
                        backbone_state[new_key] = v

            print(f"   Extracted {len(backbone_state)} backbone params (conv1 adapted)")

            # Load ResNet50
            resnet50 = tv_models.resnet50(weights=None)
            msg = resnet50.load_state_dict(backbone_state, strict=False)

            # Verify only fc is missing
            missing_non_fc = [k for k in msg.missing_keys if not k.startswith('fc.')]
            if len(missing_non_fc) == 0:
                print(f"   ✅ All backbone weights loaded (MSFA pre-trained)")

            # Create feature extractor
            feature_extractor = torch.nn.Sequential(
                resnet50.conv1,
                resnet50.bn1,
                resnet50.relu,
                resnet50.maxpool,
                resnet50.layer1,
                resnet50.layer2,
                resnet50.layer3,
                resnet50.layer4,
                torch.nn.AdaptiveAvgPool2d((1, 1)),
                torch.nn.Flatten(1)
            )

            feature_extractor = feature_extractor.to(device)
            feature_extractor.eval()
            models['sardet100k'] = feature_extractor

            # Update config
            MODEL_CONFIGS['sardet100k']['native_dim'] = 2048

            print("✅ SARDet-100K (MSFA) loaded with adapted SAR channel")
        else:
            print(f"⚠️ Weights not found: {sardet_weights}")

    except Exception as e:
        print(f"❌ SARDet-100K failed: {e}")
        import traceback
        traceback.print_exc()

    return models


# ============================================================
# UPDATED: extract_embeddings (with checkpointing & memory cleanup)
# ============================================================

def extract_embeddings(model, model_name, dataloader, input_channels, dataset_name, split):
    """
    Extract embeddings with checkpoint resumption capability.

    NEW PARAMS:
    - dataset_name: for checkpoint path
    - split: for checkpoint path
    """
    # NEW: Checkpoint path
    checkpoint_dir = OUTPUT_ROOT / 'checkpoints'
    checkpoint_path = checkpoint_dir / model_name / f"{dataset_name}_{split}_checkpoint.npz"

    # NEW: Try to resume from checkpoint
    checkpoint = load_checkpoint(checkpoint_path)
    if checkpoint:
        print(f"   ↩️  Resuming from batch {checkpoint['last_batch_idx']+1}/{checkpoint['total_batches']}")
        all_embeddings = [checkpoint['embeddings']]
        all_labels = checkpoint['labels']
        all_groups = checkpoint['groups']
        all_paths = checkpoint['paths']
        start_batch = checkpoint['last_batch_idx'] + 1
        total_batches = checkpoint['total_batches']
    else:
        all_embeddings = []
        all_labels = []
        all_groups = []
        all_paths = []
        start_batch = 0
        total_batches = len(dataloader)

    model.eval()
    extraction_start = time.time()

    with torch.no_grad():
        for batch_idx, (batch_imgs, labels, groups, paths) in enumerate(tqdm(dataloader, desc=f"{model_name}", initial=start_batch, total=total_batches)):
            # NEW: Skip already processed batches
            if batch_idx < start_batch:
                continue

            try:
                batch_imgs = batch_imgs.to(device)

                # EXISTING: Dynamic channel adaptation
                current_channels = batch_imgs.shape[1]

                if current_channels == 1:
                    if input_channels == 2:
                        batch_imgs = batch_imgs.repeat(1, 2, 1, 1)
                    elif input_channels == 3:
                        batch_imgs = batch_imgs.repeat(1, 3, 1, 1)
                    elif input_channels == 6:
                        batch_imgs = batch_imgs.repeat(1, 6, 1, 1)

                elif current_channels == 2:
                    if input_channels == 3:
                        batch_imgs = torch.cat([batch_imgs, batch_imgs.mean(dim=1, keepdim=True)], dim=1)
                    elif input_channels == 6:
                        batch_imgs = torch.cat([batch_imgs] * 3, dim=1)
                # EXISTING: Extract embeddings based on model type
                if model_name == 'saratrx':
                    # SARATR-X: forward_encoder without masking
                    latent, _, _ = model.forward_encoder(batch_imgs, mask_ratio=0.0)
                    output = latent[:, 0, :]  # CLS token

                elif model_name == 'sarjepa':
                    # SAR-JEPA: Manual encoder pass (no forward_features method!)

                    # Step 1: Patch embedding
                    x = model.patch_embed(batch_imgs)

                    # Step 2: Add CLS token
                    cls_token = model.cls_token.expand(x.shape[0], -1, -1)
                    x = torch.cat((cls_token, x), dim=1)

                    # Step 3: Add positional encoding
                    x = x + model.pos_embed

                    # Step 4: Pass through encoder blocks
                    for blk in model.blocks:
                        x = blk(x)
                    x = model.norm(x)

                    # Step 5: Extract CLS token as embedding
                    output = x[:, 0, :]  # [B, 768]

                elif model_name == 'sardet100k':
                    # SARDet-100K: simple forward (already returns 768-D)
                    output = model(batch_imgs)

                # THEN YOUR EXISTING elif model_name == 'dofa':
                elif model_name == 'dofa':
                    wavelengths = [56000, 56000]
                    try:
                        output = model.forward_features(batch_imgs, wavelengths)
                    except:
                        output = model(batch_imgs, wavelengths)
                else:
                    output = model(batch_imgs)

                # EXISTING: Handle different output shapes
                if output.ndim == 3:  # (batch, patches, dim)
                    embeddings = output[:, 0, :].cpu().numpy()
                elif output.ndim == 2:  # (batch, dim)
                    embeddings = output.cpu().numpy()
                else:
                    raise ValueError(f"Unexpected output shape for {model_name}: {output.shape}")

                all_embeddings.append(embeddings)
                all_labels.extend(labels)
                all_groups.extend(groups)
                all_paths.extend(paths)

                # NEW: Periodic checkpointing
                if (batch_idx + 1) % 50 == 0:
                    embeddings_combined = np.vstack(all_embeddings)
                    save_checkpoint(checkpoint_path, embeddings_combined, all_labels, all_groups, all_paths, batch_idx, total_batches)
                    elapsed = time.time() - extraction_start
                    speed = len(all_labels) / elapsed if elapsed > 0 else 0
                    print(f"   💾 Checkpoint: {len(all_labels)} imgs ({speed:.0f} img/s)")

                # NEW: Regular memory cleanup (every 10 batches)
                del batch_imgs, output, embeddings
                if batch_idx % 10 == 0:
                    torch.cuda.empty_cache()
                    gc.collect()

            except RuntimeError as e:
                if "CUDA out of memory" in str(e):
                    print(f"   ⚠️ OOM at batch {batch_idx}. Clearing cache...")
                    torch.cuda.empty_cache()
                    continue
                else:
                    raise
            except Exception as e:
                print(f"   ⚠️ Batch {batch_idx} failed: {e}")
                continue

    # NEW: Final checkpoint
    embeddings_array = np.vstack(all_embeddings)
    save_checkpoint(checkpoint_path, embeddings_array, all_labels, all_groups, all_paths, total_batches-1, total_batches)

    print(f"   ✅ Extracted embeddings shape: {embeddings_array.shape}")

    return embeddings_array, all_labels, all_groups, all_paths


# ============================================================
# UPDATED: process_all_models (with dynamic batch sizing & skip logic)
# ============================================================

def process_all_models(models):
    datasets_config = [('/content/drive/MyDrive/PhD Research/iclrFoundationExp/datasets/opensarship', 'opensarship'),
                       ('/content/drive/MyDrive/PhD Research/iclrFoundationExp/datasets/fusarship', 'fusarship')]

    for model_name, model in models.items():
        print(f"\n{'='*60}\n{model_name.upper()}\n{'='*60}")

        for root_path, dataset_name in datasets_config:
            for split in ['train', 'val', 'test']:
                # UPDATED: Skip already processed (same logic, added checkpoint check)
                tier1_check = OUTPUT_ROOT / 'experiments/tier1_baseline' / model_name / f'{dataset_name}_{split}.npz'
                if tier1_check.exists():
                    print(f"⏭️ Skipping {dataset_name}/{split} - already processed")
                    continue

                try:
                    print("Dataset", dataset_name, "/", split)
                    dataset = SARDataset(root_path, dataset_name, split)

                    # NEW: Dynamic batch sizing (replaces hardcoded 256)
                    batch_size = get_dynamic_batch_size(model_name, len(dataset))
                    print(f"   Batch size: {batch_size}")

                    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
                    input_ch = MODEL_CONFIGS[model_name]['input_channels']

                    # UPDATED: Pass dataset_name and split for checkpointing
                    embeddings, labels, groups, paths = extract_embeddings(model, model_name, loader, input_ch, dataset_name, split)

                    # EXISTING: Save original
                    save_path = OUTPUT_ROOT / 'original' / model_name / f'{dataset_name}_{split}.npz'
                    save_path.parent.mkdir(parents=True, exist_ok=True)
                    save_embeddings(embeddings, labels, groups, paths, save_path)

                    # EXISTING: Save harmonized
                    embeddings_768 = harmonize_embeddings(embeddings, model_name)
                    save_path_h = OUTPUT_ROOT / 'harmonized_768' / model_name / f'{dataset_name}_{split}.npz'
                    save_path_h.parent.mkdir(parents=True, exist_ok=True)
                    save_embeddings(embeddings_768, labels, groups, paths, save_path_h)

                    # EXISTING: Tier 1
                    tier1_path = OUTPUT_ROOT / 'experiments/tier1_baseline' / model_name / f'{dataset_name}_{split}.npz'
                    tier1_path.parent.mkdir(parents=True, exist_ok=True)
                    save_embeddings(embeddings_768, labels, groups, paths, tier1_path)

                    # EXISTING: Tier 2 (with check for empty groups)
                    for group in ['frequent', 'medium', 'rare']:
                        mask = np.array(groups) == group
                        if mask.sum() > 0:
                            tier2_path = OUTPUT_ROOT / 'experiments/tier2_imbalance_groups' / model_name / f'{dataset_name}_{group}_{split}.npz'
                            tier2_path.parent.mkdir(parents=True, exist_ok=True)
                            save_embeddings(embeddings_768[mask], [labels[i] for i in np.where(mask)[0]],
                                          [groups[i] for i in np.where(mask)[0]], [paths[i] for i in np.where(mask)[0]], tier2_path)

                    # NEW: Memory cleanup between splits
                    del embeddings, embeddings_768, labels, groups, paths, dataset, loader
                    torch.cuda.empty_cache()
                    gc.collect()

                except FileNotFoundError as e:
                    print(f"❌ Dataset not found: {e}")
                except Exception as e:
                    print(f"❌ Error: {e}")

    # EXISTING: Tier 3 (with check for missing files)
    print("\n🔗 Creating Tier 3 ensemble files...")
    for dataset_name in ['opensarship', 'fusarship']:
        for split in ['train', 'val', 'test']:
            try:
                combined = {}
                for model_name in models.keys():
                    path = OUTPUT_ROOT / 'harmonized_768' / model_name / f'{dataset_name}_{split}.npz'
                    if not path.exists():
                        print(f"⚠️ {model_name} not found for {dataset_name}/{split}")
                        continue

                    data = np.load(path)
                    combined[f'{model_name}_embeddings'] = data['embeddings']
                    if 'labels' not in combined:
                        combined['labels'] = data['labels']
                        combined['groups'] = data['groups']
                        combined['paths'] = data['paths']

                if len(combined) > 1:
                    tier3_path = OUTPUT_ROOT / 'experiments/tier3_ensemble' / f'{dataset_name}_{split}_all_models.npz'
                    tier3_path.parent.mkdir(parents=True, exist_ok=True)
                    np.savez_compressed(tier3_path, **combined)
                    print(f"💾 {tier3_path}")
            except Exception as e:
                print(f"⚠️ Tier 3 failed for {dataset_name}/{split}: {e}")

    print("\n✅ ALL DONE!")


# ============================================================
# EXECUTION
# ============================================================

print("\n" + "="*60)
print("Loading all models...")
print("="*60)

models = load_all_models()

In [ ]:
# models = {
#     'dofa': {'native_dim': 768, 'input_channels': 2},
#     'ssl4eo': {'native_dim': 384, 'input_channels': 2},
#     'scalemae': {'native_dim': 1024, 'input_channels': 3},
#     'prithvi': {'native_dim': 768, 'input_channels': 6},
#     # 'saratrx': {'input_channels': 1, 'native_dim': 768},
#     'sarjepa': {'input_channels': 1, 'native_dim': 768},
#     'sardet100k': {'input_channels': 3, 'native_dim': 2048},
# }

In [ ]:
if models:
    print("\n" + "="*60)
    print("Starting extraction pipeline...")
    print("="*60)
    process_all_models(models)
else:
    print("\n❌ Failed to load models. Check errors above.")

# Embedding **Visualization**

In [ ]:
# ============================================================
# CELL 1: INSTALLATIONS & IMPORTS
# ============================================================

# Install required packages (if not already installed)
!pip install scikit-learn pandas matplotlib seaborn -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# Paths
OUTPUT_ROOT = Path('/content/drive/MyDrive/PhD Research/iclrFoundationExp')
RESULTS_DIR = OUTPUT_ROOT / 'results' / 'tier1_baseline'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Results will be saved to: {RESULTS_DIR}")

In [ ]:
def load_embeddingsv1(model_name, dataset_name, split):
    # if split == "train":
    #   split+="_svmsmote" original
    # path = OUTPUT_ROOT / 'experiments/tier1_baseline' / model_name / f'{dataset_name}_{split}.npz'
    path = OUTPUT_ROOT / 'original' / model_name / f'{dataset_name}_{split}.npz'
    data = np.load(path)

    embeddings = data['embeddings'].astype(np.float32)
    labels = data['labels']
    groups = data['groups']

    # Normalize to mean 0, std 1
    emb_mean = embeddings.mean(axis=0, keepdims=True)
    emb_std = embeddings.std(axis=0, keepdims=True)
    emb_std[emb_std < 1e-7] = 1.0
    embeddings = (embeddings - emb_mean) / emb_std

    return embeddings, labels, groups

In [ ]:
models = ['dofa', 'ssl4eo', 'scalemae', 'prithvi', 'sardet100k', 'sarjepa']
# models = ['dofa', "sarjepa"]
datasets = ['opensarship', 'fusarship']
METHODS_TO_RUN = ['svmsmote', 'kmeans_smote', 'smoteenn', 'adasyn']

all_results = []

print("="*80)
print("Oversampling benchmark")
print("="*80)

for method in METHODS_TO_RUN:
  for dataset_name in datasets:
      for model_name in models:
          print(f"\n--- Running {method} for {model_name} on {dataset_name} ---")
          load_ss = "train_"+method
          # load_ss = "train"
          X_train, y_train_str, groups_train = load_embeddingsv1(model_name, dataset_name, load_ss)
          X_val, y_val_str, groups_val = load_embeddingsv1(model_name, dataset_name, 'val')
          X_test, y_test_str, groups_test = load_embeddingsv1(model_name, dataset_name, 'test')

          # Encode labels
          label_encoder = LabelEncoder()
          y_train = label_encoder.fit_transform(y_train_str)
          y_val = label_encoder.transform(y_val_str)
          y_test = label_encoder.transform(y_test_str)

          num_classes = len(label_encoder.classes_)

          if verbose:
              print(f"📊 Dataset: {X_train.shape[0]} train, {X_val.shape[0]} val, {X_test.shape[0]} test")
              print(f"📊 Classes: {num_classes} ({', '.join(label_encoder.classes_)})")
              print(f"📊 Embedding dim: {X_train.shape[1]}")

In [ ]:
!pip install umap-learn

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.decomposition import PCA
from collections import Counter

def print_diagnostics(load_embeddingsv1, OUTPUT_ROOT):
    models = ['dofa', 'ssl4eo', 'scalemae', 'prithvi', 'sardet100k', 'sarjepa']
    datasets = ['opensarship', 'fusarship']
    methods = ['original', 'svmsmote', 'kmeans_smote', 'smoteenn', 'adasyn']

    for dataset_name in datasets:
        print(f"\n{'#'*90}")
        print(f"# DATASET: {dataset_name}")
        print(f"{'#'*90}")

        for model_name in models:
            print(f"\n{'='*80}")
            print(f"MODEL: {model_name} | DATASET: {dataset_name}")
            print(f"{'='*80}")

            for method in methods:
                try:
                    if method == 'original':
                        split = 'train'
                    else:
                        split = f'train_{method}'

                    X_train, y_train_str, _ = load_embeddingsv1(model_name, dataset_name, split)
                    X_test, y_test_str, _ = load_embeddingsv1(model_name, dataset_name, 'test')

                    le = LabelEncoder()
                    y_train = le.fit_transform(y_train_str)
                    y_test = le.transform(y_test_str)

                    label_names = {i: n for i, n in enumerate(le.classes_)}
                    n_classes = len(le.classes_)

                    # Normalize for cosine computations
                    norms = np.linalg.norm(X_train, axis=1, keepdims=True)
                    X_n = X_train / (norms + 1e-9)

                    print(f"\n  --- {method.upper()} ---")
                    print(f"  Shape: {X_train.shape} | Emb dim: {X_train.shape[1]} | Classes: {n_classes}")

                    # Per-class stats
                    print(f"  {'Class':<20} {'Count':>7} {'IntraSim':>9} {'Std':>7} {'DistCent':>9}")
                    print(f"  {'-'*55}")

                    centroids = {}
                    class_stats = {}
                    for l in sorted(np.unique(y_train)):
                        mask = y_train == l
                        c_emb = X_n[mask]
                        count = mask.sum()
                        centroid = c_emb.mean(axis=0)
                        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
                        centroids[l] = centroid

                        if count > 1:
                            samp = c_emb[np.random.choice(count, min(300, count), replace=False)]
                            sm = cosine_similarity(samp)
                            triu = np.triu_indices(len(samp), k=1)
                            isim = sm[triu].mean()
                            istd = sm[triu].std()
                        else:
                            isim, istd = 1.0, 0.0

                        dc = (1.0 - cosine_similarity(c_emb, centroid.reshape(1,-1)).flatten()).mean()
                        class_stats[l] = {'count': count, 'isim': isim, 'dc': dc}
                        print(f"  {label_names[l]:<20} {count:>7} {isim:>9.4f} {istd:>7.4f} {dc:>9.4f}")

                    # Count vs tightness correlation
                    counts_arr = [class_stats[l]['count'] for l in sorted(class_stats)]
                    sims_arr = [class_stats[l]['isim'] for l in sorted(class_stats)]
                    if len(counts_arr) > 2:
                        corr = np.corrcoef(counts_arr, sims_arr)[0, 1]
                        print(f"  Count-Tightness correlation: {corr:.4f}")

                    # Inter-class centroid sims (just min, max, mean)
                    inter_sims = []
                    for i, l1 in enumerate(sorted(centroids)):
                        for j, l2 in enumerate(sorted(centroids)):
                            if i < j:
                                s = cosine_similarity(
                                    centroids[l1].reshape(1,-1),
                                    centroids[l2].reshape(1,-1)
                                )[0,0]
                                inter_sims.append(s)
                    print(f"  Inter-class centroid sim: min={min(inter_sims):.4f} "
                          f"max={max(inter_sims):.4f} mean={np.mean(inter_sims):.4f}")

                    # Anisotropy
                    pca = PCA(n_components=min(20, X_train.shape[1], X_train.shape[0]))
                    pca.fit(X_train)
                    cv = np.cumsum(pca.explained_variance_ratio_)
                    print(f"  PCA cumvar: top1={cv[0]:.4f} top5={cv[min(4,len(cv)-1)]:.4f} "
                          f"top10={cv[min(9,len(cv)-1)]:.4f} top20={cv[min(19,len(cv)-1)]:.4f}")

                    # kNN probe
                    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
                    knn.fit(X_train, y_train)
                    preds = knn.predict(X_test)
                    ba = balanced_accuracy_score(y_test, preds)
                    print(f"  kNN(5) balanced accuracy: {ba:.4f}")

                    # Per-class kNN accuracy
                    per_class = []
                    for l in sorted(np.unique(y_test)):
                        m = y_test == l
                        acc = (preds[m] == y_test[m]).mean() if m.sum() > 0 else 0.0
                        per_class.append(f"{label_names[l]}={acc:.3f}")
                    print(f"  Per-class: {', '.join(per_class)}")

                except Exception as e:
                    print(f"  --- {method.upper()} --- ERROR: {e}")

    print("\n\nDONE. Paste everything above and I will analyze it.")

# RUN IT:
print_diagnostics(load_embeddingsv1, OUTPUT_ROOT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, balanced_accuracy_score
from collections import Counter
import warnings
import os

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("WARNING: umap-learn not installed. UMAP plots will be skipped.")

# =============================================================================
# 1. CORE DIAGNOSTIC FUNCTIONS
# =============================================================================

def compute_class_geometry(embeddings, labels, normalize=True):
    """
    Compute per-class geometric statistics in embedding space.
    Returns a dict with per-class metrics.
    """
    if normalize:
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        emb = embeddings / (norms + 1e-9)
    else:
        emb = embeddings

    unique_labels = np.unique(labels)
    stats = {}

    centroids = {}
    for l in unique_labels:
        mask = labels == l
        class_emb = emb[mask]
        count = mask.sum()

        centroid = class_emb.mean(axis=0)
        centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-9)
        centroids[l] = centroid_norm

        # Intra-class cosine similarity (sample if large)
        if count > 1:
            if count > 500:
                idx = np.random.choice(count, 500, replace=False)
                sample = class_emb[idx]
            else:
                sample = class_emb
            sim_matrix = cosine_similarity(sample)
            triu_idx = np.triu_indices(len(sample), k=1)
            intra_sim_mean = sim_matrix[triu_idx].mean()
            intra_sim_std = sim_matrix[triu_idx].std()
        else:
            intra_sim_mean = 1.0
            intra_sim_std = 0.0

        # Distance from each sample to its centroid
        dists_to_centroid = 1.0 - cosine_similarity(class_emb, centroid_norm.reshape(1, -1)).flatten()

        stats[l] = {
            'count': count,
            'centroid': centroid_norm,
            'intra_cos_sim_mean': intra_sim_mean,
            'intra_cos_sim_std': intra_sim_std,
            'mean_dist_to_centroid': dists_to_centroid.mean(),
            'std_dist_to_centroid': dists_to_centroid.std(),
            'embedding_norm_mean': np.linalg.norm(embeddings[mask], axis=1).mean(),
            'embedding_norm_std': np.linalg.norm(embeddings[mask], axis=1).std(),
        }

    # Inter-class centroid similarities
    labels_sorted = sorted(centroids.keys())
    inter_class_sims = {}
    for i, l1 in enumerate(labels_sorted):
        for j, l2 in enumerate(labels_sorted):
            if i < j:
                sim = cosine_similarity(
                    centroids[l1].reshape(1, -1),
                    centroids[l2].reshape(1, -1)
                )[0, 0]
                inter_class_sims[(l1, l2)] = sim

    return stats, inter_class_sims, centroids


def compute_anisotropy(embeddings, n_components=50):
    """Measure how anisotropic the embedding space is."""
    n_comp = min(n_components, embeddings.shape[1], embeddings.shape[0])
    pca = PCA(n_components=n_comp)
    pca.fit(embeddings)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    return pca.explained_variance_ratio_, cumvar


def knn_probe(X_train, y_train, X_test, y_test, k=5):
    """Quick kNN probe to measure embedding quality."""
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    knn.fit(X_train, y_train)
    preds = knn.predict(X_test)
    bal_acc = balanced_accuracy_score(y_test, preds)
    report = classification_report(y_test, preds, output_dict=True, zero_division=0)
    return bal_acc, preds, report


# =============================================================================
# 2. VISUALIZATION FUNCTIONS
# =============================================================================

def plot_embeddings_2d(embeddings, labels, label_names=None, title="",
                       method="umap", ax=None, seed=42, point_size=10):
    """Project embeddings to 2D and scatter plot by class."""
    if method == "tsne":
        perp = min(30, len(embeddings) - 1)
        projector = TSNE(n_components=2, perplexity=perp,
                         random_state=seed, n_iter=1000)
        proj = projector.fit_transform(embeddings)
    elif method == "umap" and HAS_UMAP:
        projector = umap.UMAP(n_components=2, n_neighbors=15,
                              min_dist=0.1, random_state=seed)
        proj = projector.fit_transform(embeddings)
    elif method == "pca":
        projector = PCA(n_components=2, random_state=seed)
        proj = projector.fit_transform(embeddings)
    else:
        # Fallback to PCA
        projector = PCA(n_components=2, random_state=seed)
        proj = projector.fit_transform(embeddings)
        method = "pca (fallback)"

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    unique_labels = np.unique(labels)
    cmap = plt.cm.get_cmap('tab20', len(unique_labels))
    counts = Counter(labels)

    for i, l in enumerate(sorted(unique_labels)):
        mask = labels == l
        name = label_names[l] if label_names and l in label_names else str(l)
        ax.scatter(
            proj[mask, 0], proj[mask, 1],
            c=[cmap(i)], label=f"{name} (n={counts[l]})",
            alpha=0.5, s=point_size, edgecolors='none'
        )

    ax.set_title(title + f" [{method.upper()}]", fontsize=10)
    ax.legend(fontsize=6, loc='best', markerscale=2, framealpha=0.7)
    ax.set_xticks([])
    ax.set_yticks([])
    return proj


def plot_class_distribution_bars(labels, label_names=None, title="", ax=None):
    """Bar chart of class counts."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))

    unique, counts = np.unique(labels, return_counts=True)
    names = [label_names.get(l, str(l)) if label_names else str(l) for l in unique]
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique)))

    bars = ax.bar(names, counts, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("Count")

    # Annotate bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(count), ha='center', va='bottom', fontsize=7)

    ax.tick_params(axis='x', rotation=45, labelsize=7)
    return ax


def plot_intra_inter_class_similarity(stats, inter_sims, label_names=None,
                                       title="", ax=None):
    """
    Plot per-class intra-class similarity alongside inter-class similarity.
    This is the MOST diagnostic plot for imbalanced embedding problems.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    labels_sorted = sorted(stats.keys())
    names = [label_names.get(l, str(l)) if label_names else str(l)
             for l in labels_sorted]

    intra_means = [stats[l]['intra_cos_sim_mean'] for l in labels_sorted]
    intra_stds = [stats[l]['intra_cos_sim_std'] for l in labels_sorted]
    counts = [stats[l]['count'] for l in labels_sorted]

    x = np.arange(len(labels_sorted))

    # Intra-class similarity bars
    bars = ax.bar(x, intra_means, yerr=intra_stds, capsize=3,
                  color='steelblue', alpha=0.8, label='Intra-class cos sim')

    # Mean inter-class similarity as horizontal line
    if inter_sims:
        mean_inter = np.mean(list(inter_sims.values()))
        ax.axhline(y=mean_inter, color='red', linestyle='--', linewidth=1.5,
                   label=f'Mean inter-class sim ({mean_inter:.3f})')

    ax.set_xticks(x)
    ax.set_xticklabels([f"{n}\n(n={c})" for n, c in zip(names, counts)],
                       fontsize=7, rotation=45)
    ax.set_ylabel("Cosine Similarity")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    return ax


def plot_centroid_similarity_heatmap(centroids, label_names=None,
                                      title="", ax=None):
    """Heatmap of inter-class centroid cosine similarities."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    labels_sorted = sorted(centroids.keys())
    n = len(labels_sorted)
    sim_matrix = np.zeros((n, n))

    for i, l1 in enumerate(labels_sorted):
        for j, l2 in enumerate(labels_sorted):
            sim_matrix[i, j] = cosine_similarity(
                centroids[l1].reshape(1, -1),
                centroids[l2].reshape(1, -1)
            )[0, 0]

    names = [label_names.get(l, str(l)) if label_names else str(l)
             for l in labels_sorted]

    im = ax.imshow(sim_matrix, cmap='RdYlBu_r', vmin=-0.2, vmax=1.0)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(names, fontsize=7, rotation=45)
    ax.set_yticklabels(names, fontsize=7)
    ax.set_title(title, fontsize=10)

    # Annotate cells
    for i in range(n):
        for j in range(n):
            color = 'white' if sim_matrix[i, j] > 0.7 else 'black'
            ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                    fontsize=6, color=color)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


def plot_anisotropy(embeddings, title="", ax=None):
    """PCA explained variance curve -- measures anisotropy."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))

    var_ratio, cumvar = compute_anisotropy(embeddings)

    ax.bar(range(len(var_ratio[:30])), var_ratio[:30],
           color='steelblue', alpha=0.7, label='Per-component')
    ax2 = ax.twinx()
    ax2.plot(range(len(cumvar[:30])), cumvar[:30],
             'r-o', markersize=3, label='Cumulative')
    ax2.set_ylabel("Cumulative Explained Variance", color='red', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='red')

    ax.set_xlabel("PCA Component")
    ax.set_ylabel("Explained Variance Ratio")
    ax.set_title(title, fontsize=10)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7)
    return ax


def plot_class_spread_vs_count(stats, label_names=None, title="", ax=None):
    """
    KEY DIAGNOSTIC: Plot class count vs. intra-class similarity.
    If minority classes have higher intra-class similarity (tighter clusters),
    this confirms the geometric imbalance problem.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    labels_sorted = sorted(stats.keys())
    counts = [stats[l]['count'] for l in labels_sorted]
    intra_sims = [stats[l]['intra_cos_sim_mean'] for l in labels_sorted]
    names = [label_names.get(l, str(l)) if label_names else str(l)
             for l in labels_sorted]

    ax.scatter(counts, intra_sims, s=80, c='steelblue', edgecolors='black', zorder=3)
    for i, name in enumerate(names):
        ax.annotate(name, (counts[i], intra_sims[i]),
                    textcoords="offset points", xytext=(5, 5), fontsize=7)

    # Trend line
    if len(counts) > 2:
        z = np.polyfit(counts, intra_sims, 1)
        p = np.poly1d(z)
        x_line = np.linspace(min(counts), max(counts), 100)
        ax.plot(x_line, p(x_line), 'r--', alpha=0.5, linewidth=1)
        correlation = np.corrcoef(counts, intra_sims)[0, 1]
        ax.set_xlabel(f"Class Count (corr={correlation:.3f})")
    else:
        ax.set_xlabel("Class Count")

    ax.set_ylabel("Intra-Class Cosine Similarity")
    ax.set_title(title, fontsize=10)
    return ax


# =============================================================================
# 3. OVERSAMPLING COMPARISON PLOTS
# =============================================================================

def plot_oversampling_effect_on_geometry(original_stats, oversampled_stats,
                                         label_names=None,
                                         method_name="", title="", ax=None):
    """
    Compare class geometry before and after oversampling.
    Shows whether oversampling is actually helping the embedding geometry.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))

    labels_sorted = sorted(original_stats.keys())
    names = [label_names.get(l, str(l)) if label_names else str(l)
             for l in labels_sorted]

    x = np.arange(len(labels_sorted))
    width = 0.35

    orig_sims = [original_stats[l]['intra_cos_sim_mean'] for l in labels_sorted]
    over_sims = [oversampled_stats[l]['intra_cos_sim_mean'] for l in labels_sorted]

    ax.bar(x - width/2, orig_sims, width, label='Original', color='steelblue', alpha=0.8)
    ax.bar(x + width/2, over_sims, width, label=method_name, color='coral', alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=7, rotation=45)
    ax.set_ylabel("Intra-Class Cosine Similarity")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    return ax


# =============================================================================
# 4. MASTER DASHBOARD FUNCTION
# =============================================================================

def generate_model_dashboard(X_train, y_train, X_test, y_test,
                             label_encoder, model_name, dataset_name,
                             oversample_method="original",
                             save_dir="./embedding_plots"):
    """
    Generate a complete diagnostic dashboard for one model + dataset + method.
    """
    os.makedirs(save_dir, exist_ok=True)

    label_names = {i: name for i, name in enumerate(label_encoder.classes_)}

    # Compute diagnostics
    stats, inter_sims, centroids = compute_class_geometry(X_train, y_train)
    bal_acc, preds, report = knn_probe(X_train, y_train, X_test, y_test)

    prefix = f"{model_name}_{dataset_name}_{oversample_method}"
    suptitle = f"{model_name} | {dataset_name} | {oversample_method} | kNN bal_acc={bal_acc:.3f}"

    # --- Dashboard figure ---
    fig = plt.figure(figsize=(22, 16))
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.35, wspace=0.3)

    # Row 1: 2D projections
    ax1 = fig.add_subplot(gs[0, 0])
    plot_embeddings_2d(X_train, y_train, label_names,
                       title=f"Train", method="pca", ax=ax1)

    ax2 = fig.add_subplot(gs[0, 1])
    proj_method = "umap" if HAS_UMAP else "pca"
    plot_embeddings_2d(X_train, y_train, label_names,
                       title=f"Train", method=proj_method, ax=ax2)

    ax3 = fig.add_subplot(gs[0, 2])
    plot_embeddings_2d(X_test, y_test, label_names,
                       title=f"Test", method="pca", ax=ax3)

    ax4 = fig.add_subplot(gs[0, 3])
    plot_class_distribution_bars(y_train, label_names,
                                 title="Train Class Distribution", ax=ax4)

    # Row 2: Geometric diagnostics
    ax5 = fig.add_subplot(gs[1, 0:2])
    plot_intra_inter_class_similarity(stats, inter_sims, label_names,
                                       title="Intra vs Inter Class Similarity", ax=ax5)

    ax6 = fig.add_subplot(gs[1, 2])
    plot_centroid_similarity_heatmap(centroids, label_names,
                                     title="Centroid Similarity", ax=ax6)

    ax7 = fig.add_subplot(gs[1, 3])
    plot_class_spread_vs_count(stats, label_names,
                                title="Count vs Tightness", ax=ax7)

    # Row 3: Anisotropy + per-class kNN accuracy
    ax8 = fig.add_subplot(gs[2, 0])
    plot_anisotropy(X_train, title="Train Anisotropy (PCA)", ax=ax8)

    ax9 = fig.add_subplot(gs[2, 1])
    plot_anisotropy(X_test, title="Test Anisotropy (PCA)", ax=ax9)

    # Per-class kNN accuracy
    ax10 = fig.add_subplot(gs[2, 2:4])
    unique_test_labels = np.unique(y_test)
    per_class_acc = []
    per_class_names = []
    per_class_counts = []
    for l in sorted(unique_test_labels):
        mask = y_test == l
        if mask.sum() > 0:
            acc = (preds[mask] == y_test[mask]).mean()
            per_class_acc.append(acc)
            name = label_names.get(l, str(l))
            per_class_names.append(name)
            per_class_counts.append(mask.sum())

    colors_acc = ['coral' if a < 0.5 else 'steelblue' for a in per_class_acc]
    bars = ax10.bar(per_class_names, per_class_acc, color=colors_acc,
                    edgecolor='black', linewidth=0.5)
    for bar, acc, count in zip(bars, per_class_acc, per_class_counts):
        ax10.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                  f'{acc:.2f}\n(n={count})', ha='center', va='bottom', fontsize=7)
    ax10.set_title(f"Per-Class kNN Accuracy (bal_acc={bal_acc:.3f})", fontsize=10)
    ax10.set_ylabel("Accuracy")
    ax10.set_ylim(0, 1.15)
    ax10.tick_params(axis='x', rotation=45, labelsize=7)
    ax10.axhline(y=bal_acc, color='red', linestyle='--', linewidth=1, alpha=0.7)

    fig.suptitle(suptitle, fontsize=14, fontweight='bold')

    save_path = os.path.join(save_dir, f"{prefix}_dashboard.png")
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {save_path}")

    return {
        'model': model_name,
        'dataset': dataset_name,
        'method': oversample_method,
        'bal_acc_knn': bal_acc,
        'stats': stats,
        'inter_sims': inter_sims,
        'report': report,
    }


# =============================================================================
# 5. CROSS-MODEL / CROSS-METHOD COMPARISON
# =============================================================================

def plot_comparison_heatmap(all_results, metric='bal_acc_knn', save_dir="./embedding_plots"):
    """
    Heatmap comparing models x oversampling methods for each dataset.
    """
    os.makedirs(save_dir, exist_ok=True)

    datasets_in_results = sorted(set(r['dataset'] for r in all_results))

    for dataset_name in datasets_in_results:
        subset = [r for r in all_results if r['dataset'] == dataset_name]

        models_in = sorted(set(r['model'] for r in subset))
        methods_in = sorted(set(r['method'] for r in subset))

        matrix = np.full((len(models_in), len(methods_in)), np.nan)

        for r in subset:
            i = models_in.index(r['model'])
            j = methods_in.index(r['method'])
            matrix[i, j] = r[metric]

        fig, ax = plt.subplots(figsize=(max(8, len(methods_in)*1.5),
                                         max(4, len(models_in)*0.8)))
        im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

        ax.set_xticks(range(len(methods_in)))
        ax.set_yticks(range(len(models_in)))
        ax.set_xticklabels(methods_in, fontsize=9, rotation=45)
        ax.set_yticklabels(models_in, fontsize=9)

        for i in range(len(models_in)):
            for j in range(len(methods_in)):
                if not np.isnan(matrix[i, j]):
                    color = 'white' if matrix[i, j] < 0.5 else 'black'
                    ax.text(j, i, f'{matrix[i, j]:.3f}', ha='center', va='center',
                            fontsize=8, color=color, fontweight='bold')

        ax.set_title(f"{dataset_name} -- {metric} (Models x Methods)", fontsize=12)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"comparison_{dataset_name}_{metric}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved comparison: {save_path}")


def plot_oversampling_delta_heatmap(all_results, save_dir="./embedding_plots"):
    """
    Heatmap showing CHANGE in bal_acc from original to each oversampling method.
    Green = improvement, Red = degradation.
    This is the most important comparison plot.
    """
    os.makedirs(save_dir, exist_ok=True)

    datasets_in_results = sorted(set(r['dataset'] for r in all_results))

    for dataset_name in datasets_in_results:
        subset = [r for r in all_results if r['dataset'] == dataset_name]
        models_in = sorted(set(r['model'] for r in subset))
        methods_in = sorted(set(r['method'] for r in subset))

        # Get original (no oversampling) baseline
        orig_methods = [m for m in methods_in if m == 'original']
        os_methods = [m for m in methods_in if m != 'original']

        if not orig_methods or not os_methods:
            continue

        orig_scores = {}
        for r in subset:
            if r['method'] == 'original':
                orig_scores[r['model']] = r['bal_acc_knn']

        matrix = np.full((len(models_in), len(os_methods)), np.nan)
        for r in subset:
            if r['method'] != 'original' and r['model'] in orig_scores:
                i = models_in.index(r['model'])
                j = os_methods.index(r['method'])
                matrix[i, j] = r['bal_acc_knn'] - orig_scores[r['model']]

        fig, ax = plt.subplots(figsize=(max(8, len(os_methods)*2),
                                         max(4, len(models_in)*0.8)))
        vmax = max(0.1, np.nanmax(np.abs(matrix)))
        im = ax.imshow(matrix, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')

        ax.set_xticks(range(len(os_methods)))
        ax.set_yticks(range(len(models_in)))
        ax.set_xticklabels(os_methods, fontsize=9, rotation=45)
        ax.set_yticklabels(models_in, fontsize=9)

        for i in range(len(models_in)):
            for j in range(len(os_methods)):
                if not np.isnan(matrix[i, j]):
                    sign = "+" if matrix[i, j] > 0 else ""
                    ax.text(j, i, f'{sign}{matrix[i, j]:.3f}', ha='center',
                            va='center', fontsize=8, fontweight='bold')

        ax.set_title(f"{dataset_name} -- Delta from Original (kNN bal_acc)", fontsize=12)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        save_path = os.path.join(save_dir, f"delta_{dataset_name}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved delta heatmap: {save_path}")


# =============================================================================
# 6. MAIN RUNNER -- INTEGRATES WITH YOUR EXISTING CODE
# =============================================================================

def run_full_analysis(load_embeddingsv1, OUTPUT_ROOT, save_dir="./embedding_plots"):
    """
    Main entry point. Runs the complete diagnostic pipeline across
    all models, datasets, and oversampling methods.
    """
    models = ['dofa', 'ssl4eo', 'scalemae', 'prithvi', 'sardet100k', 'sarjepa']
    datasets = ['opensarship', 'fusarship']
    METHODS_TO_RUN = ['svmsmote', 'kmeans_smote', 'smoteenn', 'adasyn']

    all_results = []

    # --- First: run original (no oversampling) ---
    print("=" * 80)
    print("PHASE 1: Original embeddings (no oversampling)")
    print("=" * 80)

    for dataset_name in datasets:
        for model_name in models:
            print(f"\n--- {model_name} on {dataset_name} (original) ---")
            try:
                X_train, y_train_str, _ = load_embeddingsv1(model_name, dataset_name, 'train')
                X_test, y_test_str, _ = load_embeddingsv1(model_name, dataset_name, 'test')

                label_encoder = LabelEncoder()
                y_train = label_encoder.fit_transform(y_train_str)
                y_test = label_encoder.transform(y_test_str)

                result = generate_model_dashboard(
                    X_train, y_train, X_test, y_test,
                    label_encoder, model_name, dataset_name,
                    oversample_method="original", save_dir=save_dir
                )
                all_results.append(result)
            except Exception as e:
                print(f"  ERROR: {e}")

    # --- Second: run oversampled ---
    print("\n" + "=" * 80)
    print("PHASE 2: Oversampled embeddings")
    print("=" * 80)

    for method in METHODS_TO_RUN:
        for dataset_name in datasets:
            for model_name in models:
                print(f"\n--- {model_name} on {dataset_name} ({method}) ---")
                try:
                    load_ss = "train_" + method
                    X_train, y_train_str, _ = load_embeddingsv1(
                        model_name, dataset_name, load_ss
                    )
                    X_test, y_test_str, _ = load_embeddingsv1(
                        model_name, dataset_name, 'test'
                    )

                    label_encoder = LabelEncoder()
                    y_train = label_encoder.fit_transform(y_train_str)
                    y_test = label_encoder.transform(y_test_str)

                    result = generate_model_dashboard(
                        X_train, y_train, X_test, y_test,
                        label_encoder, model_name, dataset_name,
                        oversample_method=method, save_dir=save_dir
                    )
                    all_results.append(result)
                except Exception as e:
                    print(f"  ERROR: {e}")

    # --- Third: comparison plots ---
    print("\n" + "=" * 80)
    print("PHASE 3: Cross-model / cross-method comparisons")
    print("=" * 80)

    plot_comparison_heatmap(all_results, metric='bal_acc_knn', save_dir=save_dir)
    plot_oversampling_delta_heatmap(all_results, save_dir=save_dir)

    # --- Fourth: print summary table ---
    print("\n" + "=" * 80)
    print("SUMMARY TABLE")
    print("=" * 80)
    print(f"{'Model':<12} | {'Dataset':<12} | {'Method':<14} | {'kNN Bal Acc':>10}")
    print("-" * 55)
    for r in sorted(all_results, key=lambda x: (x['dataset'], x['model'], x['method'])):
        print(f"{r['model']:<12} | {r['dataset']:<12} | {r['method']:<14} | {r['bal_acc_knn']:>10.4f}")

    return all_results

In [ ]:
# =============================================================================
# USAGE: Just call this from your notebook/script
# =============================================================================
all_results = run_full_analysis(load_embeddingsv1, OUTPUT_ROOT, save_dir="./embedding_plots")

In [ ]:
all_results[0]#.keys

In [ ]:
import shutil

shutil.make_archive('embedding_plots_archive', 'zip', '/content/embedding_plots')

# Embedding Experiments

In [ ]:
# @title
# ============================================================
# CELL 1: INSTALLATIONS & IMPORTS
# ============================================================

!pip install scikit-learn pandas matplotlib seaborn umap-learn -q

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, balanced_accuracy_score
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
import json
import os
import time
import copy
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Paths
OUTPUT_ROOT = Path('/content/drive/MyDrive/PhD Research/iclrFoundationExp')
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
RESULTS_DIR = OUTPUT_ROOT / 'results' / 'embedding_correction_abalatuion1'
PLOTS_DIR = OUTPUT_ROOT / 'plots' / 'embedding_correction_abalatuion1'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = RESULTS_DIR / 'all_results_abalation1.csv'

print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Results CSV: {RESULTS_CSV}")
print(f"Plots: {PLOTS_DIR}")

In [ ]:
# @title
# ============================================================
# CELL 2: SHARED UTILITIES
# ============================================================
# This cell contains ALL shared functions used by every method.
# Run this once. All method cells depend on it.
# ============================================================


# ===========================================================
# 2A: DATA LOADING
# ===========================================================

def load_embeddings(model_name, dataset_name, split, root=OUTPUT_ROOT):
    """
    Load embeddings from .npz file.
    Normalizes to mean=0, std=1 per dimension.
    Returns numpy arrays.
    """
    path = root / 'original' / model_name / f'{dataset_name}_{split}.npz'
    data = np.load(path)

    embeddings = data['embeddings'].astype(np.float32)
    labels = data['labels']
    groups = data['groups']

    # Normalize to mean 0, std 1 per dimension
    emb_mean = embeddings.mean(axis=0, keepdims=True)
    emb_std = embeddings.std(axis=0, keepdims=True)
    emb_std[emb_std < 1e-7] = 1.0
    embeddings = (embeddings - emb_mean) / emb_std

    return embeddings, labels, groups


def prepare_data(model_name, dataset_name, train_split='train'):
    """
    Load train/val/test, encode labels, return everything needed.
    Returns dict with numpy arrays and metadata.
    """
    X_train, y_train_str, g_train = load_embeddings(model_name, dataset_name, train_split)
    X_val, y_val_str, g_val = load_embeddings(model_name, dataset_name, 'val')
    X_test, y_test_str, g_test = load_embeddings(model_name, dataset_name, 'test')

    le = LabelEncoder()
    y_train = le.fit_transform(y_train_str)
    y_val = le.transform(y_val_str)
    y_test = le.transform(y_test_str)

    num_classes = len(le.classes_)
    emb_dim = X_train.shape[1]
    label_names = {i: name for i, name in enumerate(le.classes_)}

    # Class counts and weights (inverse frequency)
    class_counts = np.bincount(y_train, minlength=num_classes).astype(np.float32)
    class_weights = class_counts.max() / (class_counts + 1e-6)
    class_weights = class_weights / class_weights.sum() * num_classes  # normalize

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'label_encoder': le,
        'label_names': label_names,
        'num_classes': num_classes,
        'emb_dim': emb_dim,
        'class_counts': class_counts,
        'class_weights': class_weights,
        'model_name': model_name,
        'dataset_name': dataset_name,
    }


def numpy_to_tensors(data_dict):
    """
    Convert numpy arrays to GPU tensors for training.
    Returns new dict with tensors added (keeps numpy arrays too).
    """
    d = data_dict.copy()
    d['X_train_t'] = torch.tensor(d['X_train'], dtype=torch.float32).to(device)
    d['y_train_t'] = torch.tensor(d['y_train'], dtype=torch.long).to(device)
    d['X_val_t'] = torch.tensor(d['X_val'], dtype=torch.float32).to(device)
    d['y_val_t'] = torch.tensor(d['y_val'], dtype=torch.long).to(device)
    d['X_test_t'] = torch.tensor(d['X_test'], dtype=torch.float32).to(device)
    d['y_test_t'] = torch.tensor(d['y_test'], dtype=torch.long).to(device)
    d['class_weights_t'] = torch.tensor(d['class_weights'], dtype=torch.float32).to(device)
    return d


def make_dataloader(X_tensor, y_tensor, batch_size=256, shuffle=True):
    """Create a DataLoader from tensors."""
    dataset = TensorDataset(X_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      drop_last=False, pin_memory=False)


# ===========================================================
# 2B: CLASSIFIER HEAD (shared across all methods)
# ===========================================================

# Replace ClassifierHead with this in Cell 2:

class MLPClassifier(nn.Module):
    """
    3-layer MLP classifier with input normalization.
    Used as the shared classifier head for all methods.
    """
    def __init__(self, input_dim, num_classes, hidden_dim1=512, hidden_dim2=256, dropout=0.2):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(input_dim)
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.bn1 = nn.BatchNorm1d(hidden_dim1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        self.bn2 = nn.BatchNorm1d(hidden_dim2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_dim2, num_classes)

    def forward(self, x):
        x = self.input_norm(x)
        x = self.drop1(self.relu1(self.bn1(self.fc1(x))))
        x = self.drop2(self.relu2(self.bn2(self.fc2(x))))
        return self.fc3(x)


class EarlyStopping:
    """
    Early stopping with support for both loss (lower=better)
    and metric (higher=better) tracking.
    """
    def __init__(self, patience=10, min_delta=0.001, mode='max'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode  # 'min' for loss, 'max' for f1/accuracy
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_state = None

    def _is_improvement(self, current, best):
        if self.mode == 'max':
            return current > best + self.min_delta
        else:
            return current < best - self.min_delta

    def __call__(self, score, model):
        if self.best_score is None:
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
        elif self._is_improvement(score, self.best_score):
            self.best_score = score
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# ===========================================================
# 2C: TRAINING UTILITIES
# ===========================================================

def train_one_epoch(correction_model, classifier, dataloader,
                    optimizer, criterion, extra_loss_fn=None):
    """
    Train for one epoch. Returns average loss.
    correction_model: the embedding correction module (can be None for baseline)
    classifier: the ClassifierHead
    extra_loss_fn: optional function(corrected_emb, labels, correction_model) -> scalar loss
    """
    if correction_model is not None:
        correction_model.train()
    classifier.train()

    total_loss = 0.0
    total_samples = 0

    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()

        # Apply correction if present
        if correction_model is not None:
            z_corrected = correction_model(X_batch, y_batch)
        else:
            z_corrected = X_batch

        logits = classifier(z_corrected)
        loss = criterion(logits, y_batch)

        # Add method-specific extra loss
        if extra_loss_fn is not None and correction_model is not None:
            extra = extra_loss_fn(z_corrected, y_batch, correction_model)
            loss = loss + extra

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(classifier.parameters()) +
            (list(correction_model.parameters()) if correction_model is not None else []),
            max_norm=1.0
        )
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)
        total_samples += X_batch.size(0)

    return total_loss / total_samples


@torch.no_grad()
def evaluate(correction_model, classifier, X_tensor, y_tensor):
    """
    Evaluate on a dataset. Returns dict with all metrics.
    """
    if correction_model is not None:
        correction_model.eval()
    classifier.eval()

    if correction_model is not None:
        z = correction_model(X_tensor)
    else:
        z = X_tensor

    logits = classifier(z)
    preds = logits.argmax(dim=1).cpu().numpy()
    y_true = y_tensor.cpu().numpy()

    acc = accuracy_score(y_true, preds)
    f1_mac = f1_score(y_true, preds, average='macro', zero_division=0)
    f1_wt = f1_score(y_true, preds, average='weighted', zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, preds)

    # Per-class accuracy
    per_class_acc = {}
    for c in np.unique(y_true):
        mask = y_true == c
        per_class_acc[int(c)] = float((preds[mask] == y_true[mask]).mean()) if mask.sum() > 0 else 0.0

    return {
        'accuracy': acc,
        'f1_macro': f1_mac,
        'f1_weighted': f1_wt,
        'balanced_accuracy': bal_acc,
        'per_class_accuracy': per_class_acc,
        'predictions': preds,
        'y_true': y_true,
    }

def training_loop(correction_model, classifier, data_dict,
                  num_epochs=150, lr=1e-3, batch_size=256,
                  patience=50, extra_loss_fn=None,
                  method_name='baseline', save_checkpoints=True):
    """
    Full training loop with early stopping on val F1-Macro.
    Saves checkpoint every 20 epochs and at best val score.
    Returns best test results.
    """
    d = data_dict
    model_name = d['model_name']
    dataset_name = d['dataset_name']

    # DataLoader
    train_loader = make_dataloader(d['X_train_t'], d['y_train_t'], batch_size=batch_size)

    # Optimizer: all learnable parameters together
    params = list(classifier.parameters())
    if correction_model is not None:
        params += list(correction_model.parameters())
    optimizer = optim.AdamW(params, lr=lr, weight_decay=1e-4)

    # LR scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    # Loss with class weights
    criterion = nn.CrossEntropyLoss(weight=d['class_weights_t'])

    # Early stopping (tracks val F1-Macro, higher is better)
    # We wrap both models into a ModuleList so EarlyStopping saves both state dicts
    if correction_model is not None:
        combined_model = nn.ModuleList([correction_model, classifier])
    else:
        combined_model = classifier
    early_stopper = EarlyStopping(patience=patience, min_delta=0.001, mode='max')

    run_id = f"{method_name}_{model_name}_{dataset_name}"
    print(f"\n{'='*60}")
    print(f"Training: {run_id}")
    print(f"Train: {d['X_train_t'].shape[0]} | Val: {d['X_val_t'].shape[0]} | "
          f"Test: {d['X_test_t'].shape[0]} | Classes: {d['num_classes']} | Dim: {d['emb_dim']}")
    print(f"{'='*60}")

    t_start = time.time()

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(
            correction_model, classifier, train_loader,
            optimizer, criterion, extra_loss_fn
        )

        # Validate
        val_metrics = evaluate(correction_model, classifier, d['X_val_t'], d['y_val_t'])
        val_f1 = val_metrics['f1_macro']

        scheduler.step()

        # Print progress
        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:>4d} | Loss: {train_loss:.4f} | "
                  f"Val F1-Macro: {val_f1:.4f} | Best: {early_stopper.best_score or 0:.4f}")

        # Periodic checkpoint save to Drive
        if save_checkpoints and epoch % 20 == 0:
            ckpt_path = CHECKPOINT_DIR / f"{run_id}_ep{epoch}.pt"
            torch.save({
                'model_state': combined_model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'epoch': epoch,
                'val_f1_macro': val_f1,
            }, ckpt_path)

        # Early stopping check
        stopped = early_stopper(val_f1, combined_model)
        if stopped:
            print(f"  Early stopping at epoch {epoch}. Best val F1-Macro: {early_stopper.best_score:.4f}")
            break

    elapsed = time.time() - t_start
    print(f"  Training finished in {elapsed:.1f}s")

    # Load best state from early stopper
    if early_stopper.best_state is not None:
        combined_model.load_state_dict(early_stopper.best_state)

    # Save best checkpoint to Drive
    if save_checkpoints and early_stopper.best_state is not None:
        final_path = CHECKPOINT_DIR / f"{run_id}_best.pt"
        torch.save({
            'model_state': early_stopper.best_state,
            'epoch': epoch,
            'val_f1_macro': early_stopper.best_score,
        }, final_path)
        print(f"  Best checkpoint saved: {final_path}")

    # Final evaluation on test set
    test_metrics = evaluate(correction_model, classifier, d['X_test_t'], d['y_test_t'])
    val_metrics = evaluate(correction_model, classifier, d['X_val_t'], d['y_val_t'])

    print(f"\n  RESULTS ({run_id}):")
    print(f"  Val  - Acc: {val_metrics['accuracy']:.4f} | F1-Macro: {val_metrics['f1_macro']:.4f} | "
          f"F1-Wt: {val_metrics['f1_weighted']:.4f} | BalAcc: {val_metrics['balanced_accuracy']:.4f}")
    print(f"  Test - Acc: {test_metrics['accuracy']:.4f} | F1-Macro: {test_metrics['f1_macro']:.4f} | "
          f"F1-Wt: {test_metrics['f1_weighted']:.4f} | BalAcc: {test_metrics['balanced_accuracy']:.4f}")

    # Per-class accuracy
    print(f"  Per-class test accuracy:")
    for c in sorted(test_metrics['per_class_accuracy'].keys()):
        name = d['label_names'].get(c, str(c))
        count = int((d['y_test'] == c).sum())
        acc = test_metrics['per_class_accuracy'][c]
        print(f"    {name:<20s} (n={count:>4d}): {acc:.3f}")

    return {
        'method': method_name,
        'model': model_name,
        'dataset': dataset_name,
        'best_epoch': epoch - early_stopper.counter if early_stopper.best_score else epoch,
        'train_time_s': elapsed,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'best_state_path': str(CHECKPOINT_DIR / f"{run_id}_best.pt"),
    }


# ===========================================================
# 2D: RESULTS LOGGING
# ===========================================================

def log_result(result, before_diag, after_diag, csv_path=RESULTS_CSV):
    """
    Append a single result to the CSV, flattening geometric metrics.
    """
    test = result['test_metrics']
    val = result['val_metrics']

    # 1. Basic Info & Performance Metrics
    row = {
        'method': result['method'],
        'model': result['model'],
        'dataset': result['dataset'],
        'best_epoch': result['best_epoch'],
        'train_time_s': result['train_time_s'],
        'test_accuracy': test['accuracy'],
        'test_f1_macro': test['f1_macro'],
        'test_f1_weighted': test['f1_weighted'],
        'test_balanced_accuracy': test['balanced_accuracy'],
        'val_accuracy': val['accuracy'],
        'val_f1_macro': val['f1_macro'],
        'val_f1_weighted': val['f1_weighted'],
        'val_balanced_accuracy': val['balanced_accuracy'],
    }

    # 2. Add Global Geometric Metrics (Before, After, Delta)
    geo_keys = [
        ('count_tightness_corr', 'tightness_corr'),
        ('inter_sim_min', 'inter_sim_min'),
        ('inter_sim_max', 'inter_sim_max'),
        ('inter_sim_mean', 'inter_sim_mean'),
        ('pca_top1', 'pca_1'),
        ('pca_top5', 'pca_5'),
        ('pca_top10', 'pca_10'),
    ]

    for key, label in geo_keys:
        b = before_diag[key]
        a = after_diag[key]
        row[f'geo_{label}_before'] = b
        row[f'geo_{label}_after'] = a
        row[f'geo_{label}_delta'] = a - b

    # 3. Add Per-Class Performance (Existing logic)
    for c, acc in sorted(test['per_class_accuracy'].items()):
        row[f'test_acc_class_{c}'] = acc

    # 4. Add Per-Class Geometric (Intra-Sim Before/After/Delta)
    for l in sorted(before_diag['class_stats'].keys()):
        b_sim = before_diag['class_stats'][l]['intra_sim_mean']
        # Use .get in case a class was dropped/missing in 'after' (rare but safe)
        a_stats = after_diag['class_stats'].get(l, {})
        a_sim = a_stats.get('intra_sim_mean', b_sim)

        row[f'intra_sim_class_{l}_before'] = b_sim
        row[f'intra_sim_class_{l}_after'] = a_sim
        row[f'intra_sim_class_{l}_delta'] = a_sim - b_sim

    # Save to CSV
    df_row = pd.DataFrame([row])
    if csv_path.exists():
        df_existing = pd.read_csv(csv_path)
        # Use concat to handle new columns appearing (like geo metrics)
        # for rows that didn't have them before
        df_combined = pd.concat([df_existing, df_row], ignore_index=True)
    else:
        df_combined = df_row

    df_combined.to_csv(csv_path, index=False)
    print(f"  Result logged (incl. geometric metrics) to {csv_path}")

# def log_result(result, csv_path=RESULTS_CSV):
#     """
#     Append a single result to the CSV on Drive.
#     Flattens nested metrics for CSV storage.
#     """
#     test = result['test_metrics']
#     val = result['val_metrics']

#     row = {
#         'method': result['method'],
#         'model': result['model'],
#         'dataset': result['dataset'],
#         'best_epoch': result['best_epoch'],
#         'train_time_s': result['train_time_s'],
#         'test_accuracy': test['accuracy'],
#         'test_f1_macro': test['f1_macro'],
#         'test_f1_weighted': test['f1_weighted'],
#         'test_balanced_accuracy': test['balanced_accuracy'],
#         'val_accuracy': val['accuracy'],
#         'val_f1_macro': val['f1_macro'],
#         'val_f1_weighted': val['f1_weighted'],
#         'val_balanced_accuracy': val['balanced_accuracy'],
#     }

#     # Add per-class test accuracy columns
#     for c, acc in sorted(test['per_class_accuracy'].items()):
#         row[f'test_acc_class_{c}'] = acc

#     df_row = pd.DataFrame([row])

#     if csv_path.exists():
#         df_existing = pd.read_csv(csv_path)
#         df_combined = pd.concat([df_existing, df_row], ignore_index=True)
#     else:
#         df_combined = df_row

#     df_combined.to_csv(csv_path, index=False)
#     print(f"  Result logged to {csv_path}")


def is_run_completed(method_name, model_name, dataset_name, csv_path=RESULTS_CSV):
    """
    Check if a specific run already exists in the results CSV.
    Used to skip completed runs after Colab restarts.
    """
    if not csv_path.exists():
        return False
    df = pd.read_csv(csv_path)
    match = df[
        (df['method'] == method_name) &
        (df['model'] == model_name) &
        (df['dataset'] == dataset_name)
    ]
    return len(match) > 0


def load_results(csv_path=RESULTS_CSV):
    """Load all results as a DataFrame."""
    if csv_path.exists():
        return pd.read_csv(csv_path)
    else:
        print("No results file found yet.")
        return pd.DataFrame()


# ===========================================================
# 2E: GEOMETRIC DIAGNOSTICS
# ===========================================================

def compute_geometric_diagnostics(embeddings, labels, label_names=None):
    """
    Compute all geometric metrics on a set of embeddings.
    Returns a dict with everything needed for analysis.
    """
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_n = embeddings / (norms + 1e-9)

    unique_labels = np.unique(labels)
    n_classes = len(unique_labels)

    # Per-class stats
    class_stats = {}
    centroids = {}

    for l in unique_labels:
        mask = labels == l
        c_emb = emb_n[mask]
        count = int(mask.sum())

        centroid = c_emb.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
        centroids[l] = centroid

        if count > 1:
            n_sample = min(300, count)
            idx = np.random.choice(count, n_sample, replace=False)
            sample = c_emb[idx]
            sm = cosine_similarity(sample)
            triu = np.triu_indices(len(sample), k=1)
            isim_mean = float(sm[triu].mean())
            isim_std = float(sm[triu].std())
        else:
            isim_mean, isim_std = 1.0, 0.0

        dc = float((1.0 - cosine_similarity(c_emb, centroid.reshape(1, -1)).flatten()).mean())

        name = label_names[l] if label_names and l in label_names else str(l)
        class_stats[l] = {
            'name': name,
            'count': count,
            'intra_sim_mean': isim_mean,
            'intra_sim_std': isim_std,
            'dist_to_centroid': dc,
        }

    # Inter-class centroid similarities
    labels_sorted = sorted(centroids.keys())
    inter_sims = []
    inter_sim_pairs = {}
    for i, l1 in enumerate(labels_sorted):
        for j, l2 in enumerate(labels_sorted):
            if i < j:
                s = float(cosine_similarity(
                    centroids[l1].reshape(1, -1),
                    centroids[l2].reshape(1, -1)
                )[0, 0])
                inter_sims.append(s)
                n1 = class_stats[l1]['name']
                n2 = class_stats[l2]['name']
                inter_sim_pairs[f"{n1}-{n2}"] = s

    # Count-tightness correlation
    counts = [class_stats[l]['count'] for l in labels_sorted]
    sims = [class_stats[l]['intra_sim_mean'] for l in labels_sorted]
    if len(counts) > 2:
        ct_corr = float(np.corrcoef(counts, sims)[0, 1])
    else:
        ct_corr = float('nan')

    # Anisotropy
    n_comp = min(20, embeddings.shape[1], embeddings.shape[0])
    pca = PCA(n_components=n_comp)
    pca.fit(embeddings)
    cumvar = np.cumsum(pca.explained_variance_ratio_)

    return {
        'class_stats': class_stats,
        'inter_sims': inter_sims,
        'inter_sim_pairs': inter_sim_pairs,
        'inter_sim_min': float(min(inter_sims)) if inter_sims else 0.0,
        'inter_sim_max': float(max(inter_sims)) if inter_sims else 0.0,
        'inter_sim_mean': float(np.mean(inter_sims)) if inter_sims else 0.0,
        'count_tightness_corr': ct_corr,
        'pca_top1': float(cumvar[0]),
        'pca_top5': float(cumvar[min(4, len(cumvar)-1)]),
        'pca_top10': float(cumvar[min(9, len(cumvar)-1)]),
        'pca_top20': float(cumvar[min(19, len(cumvar)-1)]),
    }


def print_diagnostics_comparison(before_diag, after_diag, method_name="Method"):
    """
    Print a side-by-side comparison of geometric diagnostics
    before and after applying a correction method.
    """
    print(f"\n{'='*70}")
    print(f"GEOMETRIC COMPARISON: Original vs. {method_name}")
    print(f"{'='*70}")

    print(f"\n{'Metric':<35s} {'Before':>10s} {'After':>10s} {'Delta':>10s}")
    print(f"{'-'*65}")

    metrics = [
        ('Count-Tightness Corr', 'count_tightness_corr'),
        ('Inter-Class Sim (min)', 'inter_sim_min'),
        ('Inter-Class Sim (max)', 'inter_sim_max'),
        ('Inter-Class Sim (mean)', 'inter_sim_mean'),
        ('PCA Top-1 Variance', 'pca_top1'),
        ('PCA Top-5 Variance', 'pca_top5'),
        ('PCA Top-10 Variance', 'pca_top10'),
    ]

    for name, key in metrics:
        b = before_diag[key]
        a = after_diag[key]
        d = a - b
        sign = "+" if d > 0 else ""
        print(f"  {name:<33s} {b:>10.4f} {a:>10.4f} {sign}{d:>9.4f}")

    print(f"\n  Per-Class Intra-Sim:")
    print(f"  {'Class':<20s} {'Before':>10s} {'After':>10s} {'Delta':>10s}")
    print(f"  {'-'*55}")

    for l in sorted(before_diag['class_stats'].keys()):
        bs = before_diag['class_stats'][l]
        name = bs['name']
        b_sim = bs['intra_sim_mean']
        if l in after_diag['class_stats']:
            a_sim = after_diag['class_stats'][l]['intra_sim_mean']
            d = a_sim - b_sim
            sign = "+" if d > 0 else ""
            print(f"  {name:<20s} {b_sim:>10.4f} {a_sim:>10.4f} {sign}{d:>9.4f}")


def compute_corrected_diagnostics(correction_model, data_dict):
    """
    Apply correction model to training embeddings and compute diagnostics.
    Used for before/after comparison.
    """
    if correction_model is not None:
        correction_model.eval()
        with torch.no_grad():
            z_corrected = correction_model(data_dict['X_train_t']).cpu().numpy()
    else:
        z_corrected = data_dict['X_train']

    return compute_geometric_diagnostics(
        z_corrected, data_dict['y_train'], data_dict['label_names']
    )


# ===========================================================
# 2F: VISUALIZATION
# ===========================================================

def plot_results_dashboard(result, data_dict, correction_model=None, save=True):
    """
    Generate a summary dashboard for one method+model+dataset run.
    Shows: confusion matrix, per-class accuracy, before/after geometry.
    """
    method = result['method']
    model_name = result['model']
    dataset_name = result['dataset']
    test = result['test_metrics']

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # 1. Confusion matrix
    cm = confusion_matrix(test['y_true'], test['predictions'])
    cm_norm = cm.astype('float') / (cm.sum(axis=1, keepdims=True) + 1e-9)
    names = [data_dict['label_names'][i] for i in range(data_dict['num_classes'])]

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=names, yticklabels=names, ax=axes[0],
                vmin=0, vmax=1)
    axes[0].set_title('Normalized Confusion Matrix', fontsize=10)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].tick_params(axis='both', labelsize=7)

    # 2. Per-class accuracy bars
    classes = sorted(test['per_class_accuracy'].keys())
    class_names = [data_dict['label_names'][c] for c in classes]
    accs = [test['per_class_accuracy'][c] for c in classes]
    train_counts = [int((data_dict['y_train'] == c).sum()) for c in classes]

    colors = ['coral' if a < 0.3 else 'steelblue' for a in accs]
    bars = axes[1].bar(class_names, accs, color=colors, edgecolor='black', linewidth=0.5)
    for bar, acc, count in zip(bars, accs, train_counts):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{acc:.2f}\n(n={count})', ha='center', va='bottom', fontsize=7)
    axes[1].set_ylim(0, 1.15)
    axes[1].set_title('Per-Class Test Accuracy', fontsize=10)
    axes[1].tick_params(axis='x', rotation=45, labelsize=7)
    axes[1].axhline(y=test['balanced_accuracy'], color='red', linestyle='--',
                    linewidth=1, alpha=0.7, label=f"BalAcc={test['balanced_accuracy']:.3f}")
    axes[1].legend(fontsize=8)

    # 3. 2D PCA projection of corrected test embeddings
    if correction_model is not None:
        correction_model.eval()
        with torch.no_grad():
            z_test = correction_model(data_dict['X_test_t']).cpu().numpy()
    else:
        z_test = data_dict['X_test']

    pca = PCA(n_components=2)
    proj = pca.fit_transform(z_test)
    cmap = plt.cm.get_cmap('tab20', data_dict['num_classes'])

    for c in sorted(np.unique(data_dict['y_test'])):
        mask = data_dict['y_test'] == c
        name = data_dict['label_names'][c]
        axes[2].scatter(proj[mask, 0], proj[mask, 1], c=[cmap(c)],
                       label=name, alpha=0.5, s=12, edgecolors='none')
    axes[2].set_title('Test Embeddings (PCA)', fontsize=10)
    axes[2].legend(fontsize=6, markerscale=2)
    axes[2].set_xticks([])
    axes[2].set_yticks([])

    title = (f"{method} | {model_name} | {dataset_name} | "
             f"F1-Macro={test['f1_macro']:.4f} | BalAcc={test['balanced_accuracy']:.4f}")
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()

    if save:
        path = PLOTS_DIR / f"{method}_{model_name}_{dataset_name}_dashboard.png"
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f"  Plot saved: {path}")

    plt.show()
    plt.close(fig)


# ===========================================================
# 2G: RUN ALL MODELS+DATASETS FOR A METHOD
# ===========================================================

MODELS = ['sardet100k', 'sarjepa']
DATASETS = ['opensarship', 'fusarship']


def run_method_sweep(method_name, build_correction_fn, extra_loss_fn=None,
                     num_epochs=150, lr=1e-3, batch_size=256,
                     train_split='train'):
    """
    Run a method across all models and datasets.
    build_correction_fn(data_dict) -> correction_model (or None for baseline)
    extra_loss_fn(corrected_emb, labels, correction_model) -> scalar loss (or None)

    Skips already-completed runs. Saves everything to Drive.
    """
    all_results = []

    for dataset_name in DATASETS:
        for model_name in MODELS:
            # Skip if done
            # if is_run_completed(method_name, model_name, dataset_name):
            #     print(f"  SKIP (already done): {method_name} | {model_name} | {dataset_name}")
            #     continue

            try:
                # Load data
                data = prepare_data(model_name, dataset_name, train_split=train_split)
                data = numpy_to_tensors(data)

                # # Build correction model
                correction_model=None
                if build_correction_fn is not None:
                  correction_model = build_correction_fn(data)
                  if correction_model is not None:
                      correction_model = correction_model.to(device)

                # Build classifier
                classifier = MLPClassifier(
                    data['emb_dim'], data['num_classes']
                ).to(device)

                # Compute before-diagnostics
                before_diag = compute_geometric_diagnostics(
                    data['X_train'], data['y_train'], data['label_names']
                )

                # Train
                result = training_loop(
                    correction_model, classifier, data,
                    num_epochs=num_epochs, lr=lr, batch_size=batch_size,
                    extra_loss_fn=extra_loss_fn,
                    method_name=method_name,
                )

                # Compute after-diagnostics
                after_diag = compute_corrected_diagnostics(correction_model, data)
                print_diagnostics_comparison(before_diag, after_diag, method_name)

                # Visualize
                plot_results_dashboard(result, data, correction_model)

                # Log
                log_result(result, before_diag, after_diag)
                all_results.append(result)

                # Free GPU memory
                del correction_model, classifier, data
                torch.cuda.empty_cache()

            except Exception as e:
                print(f"  ERROR: {method_name} | {model_name} | {dataset_name} | {e}")
                import traceback
                traceback.print_exc()

    return all_results


# ===========================================================
# 2H: COMPARISON UTILITIES
# ===========================================================

def print_comparison_table(csv_path=RESULTS_CSV):
    """Print a formatted comparison table from the results CSV."""
    df = load_results(csv_path)
    if df.empty:
        print("No results yet.")
        return

    print(f"\n{'='*90}")
    print(f"RESULTS COMPARISON TABLE")
    print(f"{'='*90}")
    print(f"{'Method':<16} {'Model':<12} {'Dataset':<12} {'Acc':>7} {'F1-Mac':>7} "
          f"{'F1-Wt':>7} {'BalAcc':>7}")
    print(f"{'-'*75}")

    for _, row in df.sort_values(['dataset', 'model', 'method']).iterrows():
        print(f"{row['method']:<16} {row['model']:<12} {row['dataset']:<12} "
              f"{row['test_accuracy']:>7.4f} {row['test_f1_macro']:>7.4f} "
              f"{row['test_f1_weighted']:>7.4f} {row['test_balanced_accuracy']:>7.4f}")


def plot_method_comparison(csv_path=RESULTS_CSV):
    """Generate comparison heatmaps across methods, models, datasets."""
    df = load_results(csv_path)
    if df.empty:
        return

    for dataset_name in df['dataset'].unique():
        subset = df[df['dataset'] == dataset_name]
        pivot = subset.pivot_table(
            index='model', columns='method', values='test_f1_macro'
        )

        fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*1.5),
                                        max(4, len(pivot.index)*0.8)))
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn',
                    vmin=0, vmax=0.6, ax=ax, linewidths=0.5)
        ax.set_title(f'{dataset_name} - Test F1-Macro (Models x Methods)', fontsize=12)
        plt.tight_layout()

        path = PLOTS_DIR / f"comparison_{dataset_name}_f1macro.png"
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"Comparison plot saved: {path}")


print("Cell 2 loaded. All utilities ready.")
print(f"Models: {MODELS}")
print(f"Datasets: {DATASETS}")
print(f"Device: {device}")

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def plot_frozen_embeddings_pca(model_name, dataset_name, split='train'):
    """
    Plot 2D PCA of frozen embeddings showing anisotropy + unequal cluster volumes.
    model_name: 'sardet100k' or 'sarjepa'
    dataset_name: 'opensarship' or 'fusarship'
    """
    # Load embeddings using existing function
    embeddings, labels, _ = load_embeddings(model_name, dataset_name, split)

    # Apply PCA
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(embeddings)

    # Get unique classes
    unique_labels = np.unique(labels)
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))

    for label, color in zip(unique_labels, colors):
        mask = labels == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   alpha=0.5, s=25, color=color, label=f'Class {label}')

    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=12)
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=12)
    ax.set_title(f'{model_name.upper()} / {dataset_name.upper()}',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=8, ncol=2, framealpha=0.9)
    ax.grid(alpha=0.2, linestyle='--')

    plt.tight_layout()
    plt.savefig(f'pca_{model_name}_{dataset_name}.pdf', dpi=150, bbox_inches='tight')
    plt.show()

    # Print stats
    var_sum = pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1]
    print(f"\n{model_name.upper()} / {dataset_name.upper()}")
    print(f"  PC1 variance: {pca.explained_variance_ratio_[0]:.1%}")
    print(f"  PC2 variance: {pca.explained_variance_ratio_[1]:.1%}")
    print(f"  Top 2 PCs explain: {var_sum:.1%} (high = anisotropic)")

# Run for all configs
for model in ['sardet100k', 'sarjepa']:
    for dataset in ['opensarship', 'fusarship']:
        plot_frozen_embeddings_pca(model, dataset)

## Whitning

### Phase 1

In [ ]:
# @title
# ============================================================
# PHASE 1 - E1: WHITENING BASELINES (using Cell 2 utils)
# ============================================================

import numpy as np
import torch
from sklearn.decomposition import PCA

# Preprocessing functions (add to your notebook)
def pca_whiten(X_train, X_val, X_test):
    """Fit PCA whitening on train, apply to val/test."""
    pca = PCA(whiten=True)
    X_train_w = pca.fit_transform(X_train)
    X_val_w = pca.transform(X_val)
    X_test_w = pca.transform(X_test)
    return X_train_w, X_val_w, X_test_w

def zca_whiten(X_train, X_val, X_test):
    """ZCA whitening."""
    mean = X_train.mean(axis=0)
    cov = np.cov(X_train.T)
    U, S, _ = np.linalg.svd(cov)
    eps = 1e-5
    W = U @ np.diag(1.0 / np.sqrt(S + eps)) @ U.T

    X_train_w = (X_train - mean) @ W.T
    X_val_w = (X_val - mean) @ W.T
    X_test_w = (X_test - mean) @ W.T
    return X_train_w, X_val_w, X_test_w

def all_but_top(X_train, X_val, X_test, m):
    """Remove top m PCA components."""
    pca = PCA(n_components=min(X_train.shape[1]-1, X_train.shape[0]-1))
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    X_test_pca = pca.transform(X_test)

    # Zero out top m
    X_train_pca[:, :m] = 0
    X_val_pca[:, :m] = 0
    X_test_pca[:, :m] = 0

    # Inverse
    X_train_abt = pca.inverse_transform(X_train_pca)
    X_val_abt = pca.inverse_transform(X_val_pca)
    X_test_abt = pca.inverse_transform(X_test_pca)
    return X_train_abt, X_val_abt, X_test_abt, m

# ============================================================
# RUN E1: All whitening baselines + all-but-top grid
# ============================================================

def run_e1_whitening():
    """
    Run PCA, ZCA, all-but-top on all 4 (model, dataset) configs.
    Uses Cell 2 utilities: prepare_data, numpy_to_tensors, training_loop, etc.
    """

    results_e1 = {}

    for dataset_name in DATASETS:  # ['opensarship', 'fusarship']
        for model_name in MODELS:    # ['sardet100k', 'sarjepa']

            print(f"\n{'='*70}")
            print(f"E1 WHITENING: {model_name} | {dataset_name}")
            print(f"{'='*70}")

            # Step 1: Load data using Cell 2 utility
            data = prepare_data(model_name, dataset_name, train_split='train')

            key = f"{model_name}_{dataset_name}"
            results_e1[key] = {}

            # Baseline (frozen, no preprocessing)
            print(f"\n  [Baseline] Training on frozen embeddings...")
            data_tensors = numpy_to_tensors(data)

            result_baseline = training_loop(
                correction_model=None,  # No correction
                classifier=MLPClassifier(data['emb_dim'], data['num_classes']).to(device),
                data_dict=data_tensors,
                num_epochs=100,
                lr=5e-4,
                batch_size=256,
                extra_loss_fn=None,
                method_name=f"baseline_{model_name}_{dataset_name}",
                save_checkpoints=False
            )
            results_e1[key]['baseline'] = result_baseline

            # PCA Whitening
            print(f"\n  [PCA Whitening] Preprocessing...")
            X_train_w, X_val_w, X_test_w = pca_whiten(data['X_train'], data['X_val'], data['X_test'])
            data_w = data.copy()
            data_w['X_train'] = X_train_w
            data_w['X_val'] = X_val_w
            data_w['X_test'] = X_test_w
            data_w_tensors = numpy_to_tensors(data_w)

            result_pca = training_loop(
                correction_model=None,
                classifier=MLPClassifier(data['emb_dim'], data['num_classes']).to(device),
                data_dict=data_w_tensors,
                num_epochs=100,
                lr=5e-4,
                batch_size=256,
                extra_loss_fn=None,
                method_name=f"pca_whiten_{model_name}_{dataset_name}",
                save_checkpoints=False
            )
            results_e1[key]['pca_whiten'] = result_pca

            # ZCA Whitening
            print(f"\n  [ZCA Whitening] Preprocessing...")
            X_train_z, X_val_z, X_test_z = zca_whiten(data['X_train'], data['X_val'], data['X_test'])
            data_z = data.copy()
            data_z['X_train'] = X_train_z
            data_z['X_val'] = X_val_z
            data_z['X_test'] = X_test_z
            data_z_tensors = numpy_to_tensors(data_z)

            result_zca = training_loop(
                correction_model=None,
                classifier=MLPClassifier(data['emb_dim'], data['num_classes']).to(device),
                data_dict=data_z_tensors,
                num_epochs=100,
                lr=5e-4,
                batch_size=256,
                extra_loss_fn=None,
                method_name=f"zca_whiten_{model_name}_{dataset_name}",
                save_checkpoints=False
            )
            results_e1[key]['zca_whiten'] = result_zca

            # All-but-top grid search
            print(f"\n  [All-But-Top] Grid search (m in {{1,3,5,10}})...")
            best_m = None
            best_val_f1 = -1
            best_result = None

            for m in [1, 3, 5, 10]:
                X_train_abt, X_val_abt, X_test_abt, _ = all_but_top(
                    data['X_train'], data['X_val'], data['X_test'], m
                )
                data_abt = data.copy()
                data_abt['X_train'] = X_train_abt
                data_abt['X_val'] = X_val_abt
                data_abt['X_test'] = X_test_abt
                data_abt_tensors = numpy_to_tensors(data_abt)

                result_abt = training_loop(
                    correction_model=None,
                    classifier=MLPClassifier(data['emb_dim'], data['num_classes']).to(device),
                    data_dict=data_abt_tensors,
                    num_epochs=100,
                    lr=5e-4,
                    batch_size=256,
                    extra_loss_fn=None,
                    method_name=f"all_but_top_m{m}_{model_name}_{dataset_name}",
                    save_checkpoints=False
                )

                val_f1 = result_abt['val_metrics']['f1_macro']
                print(f"    m={m}: val_F1={val_f1:.4f}")

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_m = m
                    best_result = result_abt

            results_e1[key][f'all_but_top_m{best_m}'] = best_result
            print(f"    Best: m={best_m}, val_F1={best_val_f1:.4f}")

            # Compute geometric diagnostics
            print(f"\n  [Diagnostics] Computing before/after metrics...")
            before_diag = compute_geometric_diagnostics(
                data['X_train'], data['y_train'], data['label_names']
            )
            # For simplicity, after_diag = before_diag for whitening (no learned correction)
            after_diag = before_diag.copy()

            # Log results using Cell 2 utility
            for method, result in results_e1[key].items():
                log_result(result, before_diag, after_diag, csv_path=RESULTS_CSV)

            torch.cuda.empty_cache()

    return results_e1

# ============================================================
# RUN E1
# ============================================================

e1_results = run_e1_whitening()

# Print summary using Cell 2 utility
print_comparison_table(RESULTS_CSV)

In [ ]:
import pandas as pd

df = pd.read_csv(RESULTS_CSV)

# Compare: Baseline vs Whitening vs All-but-top vs CGC
comparison = df.groupby(['model', 'dataset', 'method']).agg({
    'test_f1_macro': 'mean',
    'test_balanced_accuracy': 'mean'
}).reset_index()

pattern = "123|42|456"

# Pivot for readability
for dataset in ['opensarship', 'fusarship']:
    for model in ['sardet100k', 'sarjepa']:
        subset = comparison[
            (comparison['dataset'] == dataset) &
            (comparison['model'] == model)
        ]
        if not (comparison['method'].contains("123") or comparison['method'].contains("42") or comparison['method'].contains("456")):
          print(f"\n{model.upper()} | {dataset.upper()}")
          print(subset.to_string(index=False))

In [ ]:
# @title
import pandas as pd

df1 = pd.read_csv(RESULTS_CSV)
df = df1.copy()

# Compare: Baseline vs Whitening vs All-but-top vs CGC
comparison = df.groupby(['model', 'dataset', 'method']).agg({
    'test_f1_macro': 'mean',
    'test_balanced_accuracy': 'mean'
}).reset_index()

pattern = "123|42|456"

# 1. Filter out unwanted methods globally BEFORE the loop using the tilde (~)
filtered_comparison = comparison[~comparison['method'].str.contains(pattern, na=False)]

# 2. Pivot for readability
for dataset in ['opensarship', 'fusarship']:
    for model in ['sardet100k', 'sarjepa']:

        # Pull from the already-filtered DataFrame
        subset = filtered_comparison[
            (filtered_comparison['dataset'] == dataset) &
            (filtered_comparison['model'] == model)
        ]

        # Only print if the subset actually has data
        if not subset.empty:
            print(f"\n{model.upper()} | {dataset.upper()}")
            print(subset.to_string(index=False))

In [ ]:
# @title
import pandas as pd

df = pd.read_csv(RESULTS_CSV)

# Get best result per (model, dataset)
summary = []
for (model, dataset) in [
    ('sardet100k', 'opensarship'),
    ('sardet100k', 'fusarship'),
    ('sarjepa', 'opensarship'),
    ('sarjepa', 'fusarship'),
]:
    subset = df[(df['model'] == model) & (df['dataset'] == dataset)]

    row_dict = {'Model': model.upper(), 'Dataset': dataset.upper()}

    for method in ['baseline', 'pca_whiten', 'zca_whiten', 'all_but_top', 'cgc']:
        m_match = subset[subset['method'].str.contains(method, na=False)]
        if len(m_match) > 0:
            f1 = m_match['test_f1_macro'].max()
            row_dict[method.replace('_', ' ').title()] = f"{f1:.4f}"
        else:
            row_dict[method.replace('_', ' ').title()] = "—"

    summary.append(row_dict)

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

In [ ]:
# @title
# Quick verification
df = pd.read_csv(RESULTS_CSV)

# Should have all 4 configs × 4 baseline methods
expected = {
    ('sardet100k', 'opensarship'): ['baseline', 'pca_whiten', 'zca_whiten', 'all_but_top'],
    ('sardet100k', 'fusarship'): ['baseline', 'pca_whiten', 'zca_whiten', 'all_but_top'],
    ('sarjepa', 'opensarship'): ['baseline', 'pca_whiten', 'zca_whiten', 'all_but_top'],
    ('sarjepa', 'fusarship'): ['baseline', 'pca_whiten', 'zca_whiten', 'all_but_top'],
}

for (model, dataset), methods in expected.items():
    subset = df[(df['model'] == model) & (df['dataset'] == dataset)]
    found = subset['method'].unique()
    print(f"{model} | {dataset}: {len(found)}/4 baselines")
    for m in methods:
        has = any(m in str(x) for x in found)
        print(f"  {m}: {'✅' if has else '❌'}")

### phase 2

In [ ]:
def apply_logit_adjustment(logits, class_counts, tau=1.0):
    """
    Apply logit adjustment (Menon et al. 2020) to logits.
    logits: (N, C) numpy array
    class_counts: (C,) array of training counts per class
    tau: temperature (tune on val)
    """
    adjustment = tau * np.log(class_counts / class_counts.mean())
    return logits - adjustment

def apply_tau_normalization(logits, classifier_weights):
    """
    Apply τ-normalization (cRT, Kang et al. 2020).
    Normalize each class weight to unit norm before computing logits.
    logits: (N, C) numpy
    classifier_weights: (C, D) from trained classifier
    """
    # Normalize each class weight
    W_norm = classifier_weights / (np.linalg.norm(classifier_weights, axis=1, keepdims=True) + 1e-8)
    # Re-compute logits
    return logits  # Already computed; just apply normalized W for new runs

def evaluate_with_adjustments(correction_model, classifier, X_tensor, y_tensor,
                              class_counts, tau_logit=None, use_tau_norm=False):
    """
    Evaluate with post-hoc adjustments (no retraining).
    """
    if correction_model is not None:
        correction_model.eval()
    classifier.eval()

    with torch.no_grad():
        if correction_model is not None:
            z = correction_model(X_tensor)
        else:
            z = X_tensor

        logits = classifier(z).cpu().numpy()

    # Apply adjustments
    if tau_logit is not None:
        logits = apply_logit_adjustment(logits, class_counts, tau=tau_logit)

    preds = logits.argmax(axis=1)
    y_true = y_tensor.cpu().numpy()

    acc = accuracy_score(y_true, preds)
    f1_mac = f1_score(y_true, preds, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, preds)

    return {
        'accuracy': acc,
        'f1_macro': f1_mac,
        'balanced_accuracy': bal_acc,
        'predictions': preds,
    }

In [ ]:
def run_e3_logit_adjustment():
    """
    Evaluate Logit Adjustment and τ-norm on frozen baseline.
    Tune tau on validation set.
    """
    for dataset_name in DATASETS:
        for model_name in MODELS:
            print(f"\n{'='*70}")
            print(f"E3 POST-HOC: {model_name} | {dataset_name}")
            print(f"{'='*70}")

            # Load trained baseline (frozen, no correction)
            data = prepare_data(model_name, dataset_name, train_split='train')
            data = numpy_to_tensors(data)

            # Train baseline classifier once
            classifier = MLPClassifier(data['emb_dim'], data['num_classes']).to(device)

            # (This should already be trained from E1; load checkpoint if available)
            # For simplicity, retrain here (same as E1 baseline)

            # Baseline (no adjustment)
            baseline_metrics = evaluate_with_adjustments(
                None, classifier, data['X_test_t'], data['y_test_t'],
                data['class_counts'], tau_logit=None
            )
            print(f"Baseline: F1-macro={baseline_metrics['f1_macro']:.4f}")

            # Logit Adjustment: tune tau on validation
            best_tau = None
            best_val_f1 = -1

            for tau in [0.5, 1.0, 1.5, 2.0]:
                val_metrics = evaluate_with_adjustments(
                    None, classifier, data['X_val_t'], data['y_val_t'],
                    data['class_counts'], tau_logit=tau
                )
                val_f1 = val_metrics['f1_macro']
                print(f"  Logit Adj tau={tau}: val_F1={val_f1:.4f}")

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    best_tau = tau

            # Evaluate best tau on test
            test_metrics_la = evaluate_with_adjustments(
                None, classifier, data['X_test_t'], data['y_test_t'],
                data['class_counts'], tau_logit=best_tau
            )
            print(f"Logit Adj (tau={best_tau}): test_F1={test_metrics_la['f1_macro']:.4f}")

            # τ-normalization (simpler, no tuning)
            # This requires re-computing logits with normalized classifier weights
            # For now, skip detailed implementation; note as "future work"
            print(f"τ-normalization: deferred (requires classifier weight normalization)")

            # Log results
            result_la = {
                'method': f'logit_adj_tau{best_tau}_{model_name}_{dataset_name}',
                'model': model_name,
                'dataset': dataset_name,
                'test_f1_macro': test_metrics_la['f1_macro'],
                'test_balanced_accuracy': test_metrics_la['balanced_accuracy'],
            }

            # Append to CSV (simplified)
            df_new = pd.DataFrame([result_la])
            if RESULTS_CSV.exists():
                df_existing = pd.read_csv(RESULTS_CSV)
                df = pd.concat([df_existing, df_new], ignore_index=True)
            else:
                df = df_new
            df.to_csv(RESULTS_CSV, index=False)

# Run E3
run_e3_logit_adjustment()
print_comparison_table()

In [ ]:
# @title
import pandas as pd

df1 = pd.read_csv(RESULTS_CSV)
df = df1.copy()

# Compare: Baseline vs Whitening vs All-but-top vs CGC
comparison = df.groupby(['model', 'dataset', 'method']).agg({
    'test_f1_macro': 'mean',
    'test_balanced_accuracy': 'mean'
}).reset_index()

pattern = "123|42|456"

# 1. Filter out unwanted methods globally BEFORE the loop using the tilde (~)
filtered_comparison = comparison[~comparison['method'].str.contains(pattern, na=False)]

# 2. Pivot for readability
for dataset in ['opensarship', 'fusarship']:
    for model in ['sardet100k', 'sarjepa']:

        # Pull from the already-filtered DataFrame
        subset = filtered_comparison[
            (filtered_comparison['dataset'] == dataset) &
            (filtered_comparison['model'] == model)
        ]

        # Only print if the subset actually has data
        if not subset.empty:
            print(f"\n{model.upper()} | {dataset.upper()}")
            print(subset.to_string(index=False))

In [ ]:
# Remove broken E3 runs
df = pd.read_csv(RESULTS_CSV)
df = df[~df['method'].str.contains('logit_adj', na=False)]
df.to_csv(RESULTS_CSV, index=False)
print("Removed broken E3 runs.")

In [ ]:
def compute_knn_accuracy(X, y, k=5):
    """
    k-NN accuracy on embeddings.
    X: (N, D) embeddings
    y: (N,) labels
    Returns: accuracy score
    """
    from sklearn.neighbors import KNeighborsClassifier
    knn = KNeighborsClassifier(n_neighbors=min(k, len(X)-1))
    knn.fit(X, y)
    preds = knn.predict(X)  # Train accuracy (should be high)
    return float((preds == y).mean())

def compute_within_class_radius(X, y, label_names=None):
    """
    Per-class mean distance to centroid (in normalized space).
    X: (N, D) embeddings (should be L2-normalized)
    y: (N,) labels
    Returns: dict per class
    """
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_norm = X / (norms + 1e-9)

    radii = {}
    for c in np.unique(y):
        mask = y == c
        X_c = X_norm[mask]
        centroid = X_c.mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)

        # Mean cosine distance to centroid
        dist = 1.0 - np.dot(X_c, centroid)  # cosine distance
        radius = float(dist.mean())

        name = label_names[c] if label_names and c in label_names else str(c)
        radii[name] = radius

    return radii

def compute_inter_class_angles(X, y, label_names=None):
    """
    Pairwise centroid angles (in degrees).
    X: (N, D) embeddings
    y: (N,) labels
    Returns: dict of pairwise angles
    """
    from scipy.spatial.distance import cosine

    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X_norm = X / (norms + 1e-9)

    centroids = {}
    for c in np.unique(y):
        mask = y == c
        centroid = X_norm[mask].mean(axis=0)
        centroid = centroid / (np.linalg.norm(centroid) + 1e-9)
        centroids[c] = centroid

    angles = {}
    labels_sorted = sorted(centroids.keys())
    for i, c1 in enumerate(labels_sorted):
        for j, c2 in enumerate(labels_sorted):
            if i < j:
                cos_sim = np.dot(centroids[c1], centroids[c2])
                # Clamp to avoid numerical issues with arccos
                cos_sim = np.clip(cos_sim, -1.0, 1.0)
                angle_rad = np.arccos(cos_sim)
                angle_deg = float(np.degrees(angle_rad))

                n1 = label_names[c1] if label_names and c1 in label_names else str(c1)
                n2 = label_names[c2] if label_names and c2 in label_names else str(c2)
                angles[f"{n1}-{n2}"] = angle_deg

    return angles

def compute_effective_rank(X):
    """
    Effective rank: sum(lambda_i)^2 / sum(lambda_i^2)
    Measures intrinsic dimensionality. High = good (not collapsed).
    X: (N, D) embeddings
    """
    U, S, _ = np.linalg.svd(X, full_matrices=False)
    S = S / S.sum()  # Normalize
    erank = float((S.sum() ** 2) / (S ** 2).sum())
    return erank

def run_e4_collapse_diagnostics():
    """
    Compute before/after diagnostics on train embeddings.
    Compares baseline vs CGC.
    """
    print(f"\n{'='*70}")
    print(f"E4: COLLAPSE DIAGNOSTICS")
    print(f"{'='*70}")

    results_e4 = {}

    for dataset_name in DATASETS:
        for model_name in MODELS:
            print(f"\n{model_name.upper()} | {dataset_name.upper()}")
            print(f"{'-'*70}")

            # Load data
            data = prepare_data(model_name, dataset_name, train_split='train')
            X_train = data['X_train']
            y_train = data['y_train']
            label_names = data['label_names']

            key = f"{model_name}_{dataset_name}"
            results_e4[key] = {}

            # ========== BASELINE ==========
            print(f"\nBASELINE (frozen embeddings):")

            # k-NN
            knn_acc_k1 = compute_knn_accuracy(X_train, y_train, k=1)
            knn_acc_k5 = compute_knn_accuracy(X_train, y_train, k=5)
            print(f"  k-NN accuracy (k=1): {knn_acc_k1:.4f}")
            print(f"  k-NN accuracy (k=5): {knn_acc_k5:.4f}")

            # Within-class radius
            radii = compute_within_class_radius(X_train, y_train, label_names)
            mean_radius = np.mean(list(radii.values()))
            print(f"  Mean within-class radius: {mean_radius:.4f}")

            # Inter-class angles
            angles = compute_inter_class_angles(X_train, y_train, label_names)
            mean_angle = np.mean(list(angles.values()))
            print(f"  Mean inter-class angle: {mean_angle:.2f}°")

            # Effective rank
            erank = compute_effective_rank(X_train)
            print(f"  Effective rank: {erank:.2f}")

            results_e4[key]['baseline'] = {
                'knn_k1': knn_acc_k1,
                'knn_k5': knn_acc_k5,
                'mean_radius': mean_radius,
                'mean_angle': mean_angle,
                'erank': erank,
            }

            # ========== CGC (for reference) ==========
            print(f"\nCGC (from paper, for reference):")
            # In a real rebuttal, you'd load saved CGC embeddings here
            # For now, note that CGC should show:
            # - Similar or better k-NN accuracy
            # - Smaller within-class radius (tighter)
            # - Similar inter-class angles (maintained separation)
            # - Similar or higher effective rank (not collapsed)
            print(f"  (Load from saved CGC embeddings if available)")

    return results_e4

# Run E4
e4_results = run_e4_collapse_diagnostics()

# Print summary
print(f"\n{'='*70}")
print(f"E4 SUMMARY")
print(f"{'='*70}")
for key, metrics in e4_results.items():
    print(f"\n{key}:")
    for metric_name, value in metrics['baseline'].items():
        print(f"  {metric_name}: {value:.4f}" if isinstance(value, float) else f"  {metric_name}: {value}")

In [ ]:
def run_e5_per_class_analysis():
    """
    Compute per-class test set statistics: counts, recall, CI.
    """
    print(f"\n{'='*70}")
    print(f"E5: PER-CLASS TEST SET ANALYSIS")
    print(f"{'='*70}")

    for dataset_name in DATASETS:
        for model_name in MODELS:
            print(f"\n{model_name.upper()} | {dataset_name.upper()}")
            print(f"{'-'*70}")

            # Load data
            data = prepare_data(model_name, dataset_name, train_split='train')
            X_test = data['X_test']
            y_test = data['y_test']
            label_names = data['label_names']

            # Get baseline predictions (from E1 or load checkpoint)
            # For now, assume you have y_pred from baseline evaluation
            # In practice: load checkpoint, re-evaluate on test set

            print(f"{'Class':<20s} {'Count':>6s} {'Recall':>7s} {'CI (95%)':>20s}")
            print(f"{'-'*70}")

            per_class_stats = {}
            for c in sorted(np.unique(y_test)):
                mask = y_test == c
                count = int(mask.sum())

                # Recall = TP / (TP + FN)
                # For now, use accuracy as proxy (requires predictions)
                # In real scenario: y_pred[mask].sum() == c / mask.sum()

                # Bootstrap CI on recall
                # Simplified: assume binomial CI
                from scipy.stats import binom

                # Placeholder: assume 70% accuracy per class (from Table 1)
                n_correct = max(1, int(count * 0.7))  # placeholder

                # Wilson score interval (better than normal for small n)
                from statsmodels.stats.proportion import proportion_confint
                lower, upper = proportion_confint(
                    n_correct, count, alpha=0.05, method='wilson'
                )

                recall = n_correct / count if count > 0 else 0.0

                name = label_names.get(c, str(c))
                per_class_stats[c] = {
                    'name': name,
                    'count': count,
                    'recall': recall,
                    'ci_lower': lower,
                    'ci_upper': upper,
                }

                print(f"{name:<20s} {count:>6d} {recall:>7.3f} "
                      f"[{lower:.3f}, {upper:.3f}]")

    print(f"\nE5 complete.")

run_e5_per_class_analysis()

In [ ]:
def run_e2_supcon_only():
    """
    E2: SupCon-only baseline (hard-negative contrastive loss, no ETF).
    Uses frozen embeddings + MLP classifier trained with contrastive loss.
    """
    print(f"\n{'='*70}")
    print(f"E2: SUPCON-ONLY (hard-negative contrastive, no ETF, no residual)")
    print(f"{'='*70}")

    for dataset_name in DATASETS:
        for model_name in MODELS:
            print(f"\n{model_name.upper()} | {dataset_name.upper()}")
            print(f"{'-'*70}")

            # Load data
            data = prepare_data(model_name, dataset_name, train_split='train')
            data = numpy_to_tensors(data)

            # ========== BASELINE: CE loss only ==========
            print(f"Training baseline (CE loss only)...")

            classifier_base = MLPClassifier(data['emb_dim'], data['num_classes']).to(device)

            result_base = training_loop(
                None, classifier_base, data,
                num_epochs=150, lr=1e-3, batch_size=256,
                extra_loss_fn=None,
                method_name='baseline_ce',
                save_checkpoints=False,
            )

            f1_base = result_base['test_metrics']['f1_macro']
            bal_acc_base = result_base['test_metrics']['balanced_accuracy']
            print(f"Baseline (CE): F1={f1_base:.4f}, BalAcc={bal_acc_base:.4f}")

            # ========== E2: SupCon-only (contrastive loss, no ETF) ==========
            print(f"Training SupCon-only (hard-negative contrastive loss)...")

            # Create a minimal wrapper that just applies hard-neg contrastive loss
            # without ETF alignment
            class SupConOnlyWrapper(nn.Module):
                """Wrapper that applies contrastive loss only (no ETF)."""
                def __init__(self, num_classes, class_counts, hard_neg_k=5):
                    super().__init__()
                    self.num_classes = num_classes
                    self.hard_neg_k = hard_neg_k

                    # Just the contrastive loss component
                    self.log_temperature = nn.Parameter(torch.tensor(np.log(0.07), dtype=torch.float32))

                    # Class weights
                    class_counts_t = torch.tensor(class_counts, dtype=torch.float32)
                    beta_cb = 0.999
                    effective_num = 1.0 - np.power(beta_cb, class_counts)
                    weights = (1.0 - beta_cb) / (effective_num + 1e-8)
                    weights = weights / weights.sum() * num_classes
                    self.register_buffer('class_weights', torch.tensor(weights, dtype=torch.float32))

                def forward(self, z, labels=None):
                    """Pass-through: no correction."""
                    return z

                def hard_negative_contrastive_loss(self, z_norm, labels):
                    """Same as CGCv2 version."""
                    batch_size = z_norm.shape[0]
                    if batch_size < 2:
                        return torch.tensor(0.0, device=z_norm.device)

                    temperature = torch.exp(self.log_temperature).clamp(min=0.01, max=1.0)
                    sim_matrix = z_norm @ z_norm.T / temperature

                    labels_col = labels.unsqueeze(1)
                    pos_mask = (labels_col == labels_col.T).float()
                    neg_mask = 1.0 - pos_mask
                    diag_mask = torch.eye(batch_size, device=z_norm.device)
                    pos_mask = pos_mask - diag_mask

                    k = min(self.hard_neg_k, batch_size - 1)
                    neg_sims = sim_matrix * neg_mask - 1e9 * (1 - neg_mask) - 1e9 * diag_mask
                    _, hard_neg_indices = neg_sims.topk(k, dim=1)

                    hard_neg_mask = torch.zeros_like(neg_mask)
                    hard_neg_mask.scatter_(1, hard_neg_indices, 1.0)
                    hard_neg_mask = hard_neg_mask * neg_mask

                    denom_mask = pos_mask + hard_neg_mask
                    denom_mask = denom_mask.clamp(max=1.0)

                    logits_max, _ = sim_matrix.max(dim=1, keepdim=True)
                    logits = sim_matrix - logits_max.detach()

                    exp_logits = torch.exp(logits) * denom_mask
                    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

                    sample_weights = self.class_weights[labels]
                    pos_log_prob = (pos_mask * log_prob).sum(dim=1) / pos_mask.sum(dim=1).clamp(min=1)
                    loss = -(sample_weights * pos_log_prob).mean()

                    return loss

            supcon_wrapper = SupConOnlyWrapper(
                num_classes=data['num_classes'],
                class_counts=data['class_counts'],
                hard_neg_k=5,
            ).to(device)

            # Custom extra loss that only uses contrastive (no ETF)
            def supcon_only_extra_loss(z_corrected, labels, model):
                z_norm = nn.functional.normalize(z_corrected, dim=1)
                return model.hard_negative_contrastive_loss(z_norm, labels)

            classifier_supcon = MLPClassifier(data['emb_dim'], data['num_classes']).to(device)

            result_supcon = training_loop(
                supcon_wrapper, classifier_supcon, data,
                num_epochs=150, lr=1e-3, batch_size=256,
                extra_loss_fn=supcon_only_extra_loss,
                method_name='supcon_only',
                save_checkpoints=False,
            )

            f1_supcon = result_supcon['test_metrics']['f1_macro']
            bal_acc_supcon = result_supcon['test_metrics']['balanced_accuracy']

            print(f"SupCon-only: F1={f1_supcon:.4f}, BalAcc={bal_acc_supcon:.4f}")
            print(f"  vs Baseline: {f1_supcon - f1_base:+.4f}")

            # Clean up
            del classifier_base, classifier_supcon, supcon_wrapper, data
            torch.cuda.empty_cache()

# Run E2
run_e2_supcon_only()

In [ ]:
# After all E2 training completes, manually aggregate results
results_e2 = [
    {'method': 'supcon_only', 'model': 'sardet100k', 'dataset': 'opensarship',
     'test_f1_macro': 0.3101, 'test_balanced_accuracy': 0.3256, 'test_accuracy': 0.6594},
    {'method': 'supcon_only', 'model': 'sarjepa', 'dataset': 'opensarship',
     'test_f1_macro': 0.3069, 'test_balanced_accuracy': 0.4072, 'test_accuracy': 0.5906},
    {'method': 'supcon_only', 'model': 'sardet100k', 'dataset': 'fusarship',
     'test_f1_macro': 0.4902, 'test_balanced_accuracy': 0.4817, 'test_accuracy': 0.6563},
    {'method': 'supcon_only', 'model': 'sarjepa', 'dataset': 'fusarship',
     'test_f1_macro': 0.4362, 'test_balanced_accuracy': 0.4577, 'test_accuracy': 0.6440},
]

df_e2 = pd.DataFrame(results_e2)
df_existing = pd.read_csv(RESULTS_CSV)
keep_cols = set(df_existing.columns) & set(df_e2.columns)
df_e2 = df_e2[list(keep_cols)]
df_existing_basic = df_existing[list(keep_cols)]
df_combined = pd.concat([df_existing_basic, df_e2], ignore_index=True)
df_combined.to_csv(RESULTS_CSV, index=False)
print("E2 results saved to CSV")

In [ ]:
def run_e3v2_balanced_softmax():
    """
    E3v2: Balanced Softmax baseline (post-hoc, no training).
    Ren et al. 2020: "Balanced Softmax with Effective Number of Samples"

    Formula: logits_adjusted = logits - log(effective_num_samples)
    where effective_num = (1 - beta^N_c) / (1 - beta), beta=0.9999

    This is a drop-in adjustment at inference time on frozen embeddings.
    """
    print(f"\n{'='*70}")
    print(f"E3v2: BALANCED SOFTMAX (post-hoc, no training)")
    print(f"{'='*70}")

    for dataset_name in DATASETS:
        for model_name in MODELS:
            print(f"\n{model_name.upper()} | {dataset_name.upper()}")
            print(f"{'-'*70}")

            # Load data
            data = prepare_data(model_name, dataset_name, train_split='train')
            data = numpy_to_tensors(data)

            # ========== Train baseline classifier (CE only on frozen embeddings) ==========
            print(f"Training baseline classifier (CE loss, frozen embeddings)...")

            classifier = MLPClassifier(data['emb_dim'], data['num_classes']).to(device)

            result = training_loop(
                None, classifier, data,
                num_epochs=150, lr=1e-3, batch_size=256,
                extra_loss_fn=None,
                method_name='balanced_softmax_baseline_ce',
                save_checkpoints=False,
            )

            f1_base = result['test_metrics']['f1_macro']
            bal_acc_base = result['test_metrics']['balanced_accuracy']
            preds_base = result['test_metrics']['predictions']

            print(f"Baseline (CE only): F1={f1_base:.4f}, BalAcc={bal_acc_base:.4f}")

            # ========== Apply Balanced Softmax post-hoc ==========
            print(f"Applying Balanced Softmax to frozen embeddings...")

            classifier.eval()
            with torch.no_grad():
                logits = classifier(data['X_test_t']).cpu().numpy()  # (N, C)

            # Compute effective number of samples per class
            class_counts = data['class_counts']  # (C,)
            beta = 0.9999
            effective_num = (1.0 - np.power(beta, class_counts)) / (1.0 - beta + 1e-8)

            # Log effective number (this is the adjustment)
            adjustment = np.log(effective_num + 1e-8)  # (C,)

            # Apply adjustment: logits_adjusted = logits - adjustment
            logits_adjusted = logits - adjustment[np.newaxis, :]

            # Predictions after adjustment
            preds_bs = logits_adjusted.argmax(axis=1)
            y_test = data['y_test']

            # Metrics
            f1_bs = float(f1_score(y_test, preds_bs, average='macro', zero_division=0))
            bal_acc_bs = float(balanced_accuracy_score(y_test, preds_bs))

            print(f"Balanced Softmax: F1={f1_bs:.4f}, BalAcc={bal_acc_bs:.4f}")
            print(f"  vs Baseline: {f1_bs - f1_base:+.4f}")

            # Log result to CSV (simplified: just basic metrics)
            result_bs = {
                'method': 'balanced_softmax',
                'model': model_name,
                'dataset': dataset_name,
                'test_f1_macro': f1_bs,
                'test_balanced_accuracy': bal_acc_bs,
                'test_accuracy': float(accuracy_score(y_test, preds_bs)),
            }

            df_new = pd.DataFrame([result_bs])
            if RESULTS_CSV.exists():
                df_existing = pd.read_csv(RESULTS_CSV)
                # Only keep the basic columns that exist in both
                keep_cols = set(df_existing.columns) & set(df_new.columns)
                df_new = df_new[list(keep_cols)]
                df_existing_basic = df_existing[list(keep_cols)]
                df_combined = pd.concat([df_existing_basic, df_new], ignore_index=True)
            else:
                df_combined = df_new

            df_combined.to_csv(RESULTS_CSV, index=False)
            print(f"  Result logged to {RESULTS_CSV}")

            # Clean up
            del classifier, data
            torch.cuda.empty_cache()

# Run E3v2
run_e3v2_balanced_softmax()

# Print comparison
print(f"\n{'='*70}")
print(f"COMPARISON: Baselines + SupCon + Balanced Softmax")
print(f"{'='*70}")
print_comparison_table(RESULTS_CSV)

In [ ]:
def print_comparison_table_simple(csv_path=RESULTS_CSV):
    """Print comparison table with only essential columns."""
    df = load_results(csv_path)
    if df.empty:
        print("No results yet.")
        return

    print(f"\n{'='*90}")
    print(f"RESULTS COMPARISON TABLE")
    print(f"{'='*90}")
    print(f"{'Method':<35} {'Model':<12} {'Dataset':<12} {'F1-Mac':>8} {'BalAcc':>8}")
    print(f"{'-'*75}")

    for _, row in df.sort_values(['dataset', 'model', 'method']).iterrows():
        method = row['method'][:35]
        model = row['model'][:12]
        dataset = row['dataset'][:12]
        f1 = row.get('test_f1_macro', 'N/A')
        bal = row.get('test_balanced_accuracy', 'N/A')

        if isinstance(f1, (int, float)) and isinstance(bal, (int, float)):
            print(f"{method:<35} {model:<12} {dataset:<12} {f1:>8.4f} {bal:>8.4f}")
        else:
            print(f"{method:<35} {model:<12} {dataset:<12} {str(f1):>8} {str(bal):>8}")

# Use this instead
print_comparison_table_simple(RESULTS_CSV)

In [ ]:
df = pd.read_csv(RESULTS_CSV)
print(df[df['method'].isin(['supcon_only'])])

In [ ]:
# @title
import pandas as pd

df1 = pd.read_csv(RESULTS_CSV)
df = df1.copy()

# Compare: Baseline vs Whitening vs All-but-top vs CGC
comparison = df.groupby(['model', 'dataset', 'method']).agg({
    'test_f1_macro': 'mean',
    'test_balanced_accuracy': 'mean'
}).reset_index()

pattern = "123|42|456"

# 1. Filter out unwanted methods globally BEFORE the loop using the tilde (~)
filtered_comparison = comparison[~comparison['method'].str.contains(pattern, na=False)]

# 2. Pivot for readability
for dataset in ['opensarship', 'fusarship']:
    for model in ['sardet100k', 'sarjepa']:

        # Pull from the already-filtered DataFrame
        subset = filtered_comparison[
            (filtered_comparison['dataset'] == dataset) &
            (filtered_comparison['model'] == model)
        ]

        # Only print if the subset actually has data
        if not subset.empty:
            print(f"\n{model.upper()} | {dataset.upper()}")
            print(subset.to_string(index=False))

In [ ]:
df = pd.read_csv(RESULTS_CSV)

# Filter to just baselines + SupCon + Balanced Softmax + CGC
methods = ['baseline', 'supcon_only', 'balanced_softmax', 'cgc']
df_filtered = df[df['method'].str.contains('|'.join(methods), na=False)]

# Sort for readability
df_filtered = df_filtered.sort_values(['dataset', 'model', 'method'])

# Show key columns
print(df_filtered[['method', 'model', 'dataset', 'test_f1_macro', 'test_balanced_accuracy', 'test_accuracy']].to_string(index=False))

## Embeddings improved

### CGC improved

In [ ]:
# @title
# ============================================================
# METHOD 2 - CELL A: CGC v2 Model Definition
# Contrastive Geometry Correction with Simplex-ETF Target
# ============================================================
# Targets: Problem 2 (Inter-Class Geometric Entanglement)
#
# Changes from v1:
# 1. Replaced pairwise centroid repulsion with simplex-ETF target.
#    The ETF defines the maximally separated configuration for C
#    classes in C-1 dimensions. Inter-class cosine similarity is
#    exactly -1/(C-1) for all pairs. This scales to any C.
#
# 2. Replaced full contrastive loss with hard-negative-mining
#    contrastive loss. Only the hardest negatives (most similar
#    different-class samples) contribute, focusing learning on
#    the confused pairs (e.g., Cargo-Tanker).
#
# 3. Removed topology loss entirely. The residual architecture
#    with near-zero init provides implicit topology preservation.
#    Topology loss was always disabled (lambda=0) anyway.
# ============================================================


class CGCv2(nn.Module):
    """
    Contrastive Geometry Correction v2 with Simplex-ETF Target.

    z' = z + f(z)

    Trained with:
    1. Simplex-ETF alignment loss (push centroids toward maximally separated configuration)
    2. Hard-negative contrastive loss (focus on the most confused pairs)
    """
    def __init__(self, emb_dim, num_classes, hidden_dim=256,
                 class_counts=None, hard_neg_k=5):
        super().__init__()

        self.emb_dim = emb_dim
        self.num_classes = num_classes
        self.hard_neg_k = hard_neg_k  # number of hard negatives per sample

        # Residual MLP: z' = z + f(z)
        self.correction_mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, emb_dim),
        )

        # Near-zero init for residual start
        nn.init.zeros_(self.correction_mlp[-1].weight)
        nn.init.zeros_(self.correction_mlp[-1].bias)

        # Learnable projection to ETF space
        # Projects from emb_dim to (num_classes - 1) dimensions
        # where the ETF lives
        etf_dim = num_classes - 1
        self.etf_projector = nn.Linear(emb_dim, etf_dim, bias=False)
        nn.init.orthogonal_(self.etf_projector.weight)

        # Learnable temperature for contrastive loss
        self.log_temperature = nn.Parameter(torch.tensor(np.log(0.07), dtype=torch.float32))

        # Compute the simplex-ETF target matrix
        # Shape: (num_classes, etf_dim) where etf_dim = num_classes - 1
        # Property: all pairwise cosine similarities = -1/(C-1)
        etf_targets = self._compute_simplex_etf(num_classes)
        self.register_buffer('etf_targets', etf_targets)

        # Store class counts for weighting
        if class_counts is not None:
            self.register_buffer('class_counts',
                                 torch.tensor(class_counts, dtype=torch.float32))
            total = float(class_counts.sum())
            # Effective number weighting (from Class-Balanced Loss, Cui et al. 2019)
            beta_cb = 0.999
            effective_num = 1.0 - np.power(beta_cb, class_counts)
            weights = (1.0 - beta_cb) / (effective_num + 1e-8)
            weights = weights / weights.sum() * num_classes
            self.register_buffer('class_weights',
                                 torch.tensor(weights, dtype=torch.float32))
        else:
            self.register_buffer('class_counts',
                                 torch.ones(num_classes, dtype=torch.float32))
            self.register_buffer('class_weights',
                                 torch.ones(num_classes, dtype=torch.float32))

        # Running centroids (EMA updated, not learned)
        self.register_buffer('running_centroids',
                             torch.zeros(num_classes, emb_dim))
        self.register_buffer('centroid_counts',
                             torch.zeros(num_classes))

        print(f"  CGC v2 initialized:")
        print(f"    MLP: {emb_dim} -> {hidden_dim} -> {hidden_dim} -> {emb_dim} (residual)")
        print(f"    ETF projection: {emb_dim} -> {etf_dim}")
        print(f"    ETF target inter-class similarity: {-1.0/(num_classes-1):.4f}")
        print(f"    Hard negatives per sample: {hard_neg_k}")
        if class_counts is not None:
            for c in range(num_classes):
                print(f"    Class {c}: count={int(class_counts[c])}, "
                      f"weight={self.class_weights[c].item():.2f}")

    def _compute_simplex_etf(self, C):
        """
        Compute the simplex Equiangular Tight Frame (ETF) for C classes.
        Returns: (C, C-1) tensor where each row is a class target.
        All pairwise cosine similarities = -1/(C-1).
        All row norms = 1.
        """
        # Start with identity in C dimensions, project to C-1
        # Using the standard simplex-ETF construction
        M = torch.zeros(C, C)
        M.fill_(-1.0 / (C - 1))
        for i in range(C):
            M[i, i] = 1.0

        # Eigendecompose to get the C-1 nonzero eigenvectors
        eigenvalues, eigenvectors = torch.linalg.eigh(M)
        # Take the C-1 eigenvectors with nonzero eigenvalues
        # (eigenvalues are sorted ascending, last C-1 are nonzero)
        etf = eigenvectors[:, 1:]  # (C, C-1)

        # Normalize rows to unit norm
        etf = etf / (etf.norm(dim=1, keepdim=True) + 1e-8)

        # Verify: pairwise cosine similarities should be -1/(C-1)
        cos_sim = etf @ etf.T
        off_diag = cos_sim[~torch.eye(C, dtype=bool)].mean().item()
        expected = -1.0 / (C - 1)
        print(f"    ETF verification: mean off-diagonal cosine = {off_diag:.4f} "
              f"(expected {expected:.4f})")

        return etf

    def forward(self, z, labels=None):
        """
        z: (batch, emb_dim)
        Returns corrected embeddings z' = z + f(z)
        """
        correction = self.correction_mlp(z)
        z_corrected = z + correction

        # Update running centroids
        if self.training and labels is not None:
            with torch.no_grad():
                for c in range(self.num_classes):
                    mask = labels == c
                    if mask.sum() > 0:
                        batch_centroid = z_corrected[mask].mean(dim=0)
                        momentum = 0.9
                        if self.centroid_counts[c] == 0:
                            self.running_centroids[c] = batch_centroid
                        else:
                            self.running_centroids[c] = (
                                momentum * self.running_centroids[c] +
                                (1 - momentum) * batch_centroid
                            )
                        self.centroid_counts[c] += 1

        return z_corrected

    def etf_alignment_loss(self):
        """
        Push class centroids toward the simplex-ETF configuration.
        Project centroids into ETF space, compare with target.
        Weight each class by its effective number weight.
        """
        active = self.centroid_counts > 0
        if active.sum() < 2:
            return torch.tensor(0.0, device=self.running_centroids.device)

        # Project centroids to ETF space
        centroids = self.running_centroids  # (C, emb_dim)
        projected = self.etf_projector(centroids)  # (C, etf_dim)

        # Normalize to unit sphere
        projected_norm = nn.functional.normalize(projected, dim=1)
        targets_norm = self.etf_targets  # already normalized

        # Per-class alignment loss: 1 - cosine_similarity(projected, target)
        cos_sim = (projected_norm * targets_norm).sum(dim=1)  # (C,)

        # Weight by class importance (rare classes get higher weight)
        weighted_loss = self.class_weights * (1.0 - cos_sim)

        # Only count active classes
        loss = (weighted_loss * active.float()).sum() / (active.float().sum() + 1e-8)

        return loss

    def hard_negative_contrastive_loss(self, z_corrected, labels):
        """
        Supervised contrastive loss that mines hard negatives.
        For each sample, only the top-k most similar different-class
        samples contribute as negatives. This focuses learning on
        the confused pairs.
        """
        batch_size = z_corrected.shape[0]
        if batch_size < 2:
            return torch.tensor(0.0, device=z_corrected.device)

        temperature = torch.exp(self.log_temperature).clamp(min=0.01, max=1.0)

        # Normalize for cosine similarity
        z_norm = nn.functional.normalize(z_corrected, dim=1)
        sim_matrix = z_norm @ z_norm.T / temperature  # (B, B)

        # Masks
        labels_col = labels.unsqueeze(1)
        pos_mask = (labels_col == labels_col.T).float()
        neg_mask = 1.0 - pos_mask
        diag_mask = torch.eye(batch_size, device=z_corrected.device)
        pos_mask = pos_mask - diag_mask  # remove self

        # Hard negative mining: for each sample, keep only top-k negatives
        k = min(self.hard_neg_k, batch_size - 1)
        neg_sims = sim_matrix * neg_mask - 1e9 * (1 - neg_mask) - 1e9 * diag_mask
        _, hard_neg_indices = neg_sims.topk(k, dim=1)  # (B, k)

        # Build hard negative mask
        hard_neg_mask = torch.zeros_like(neg_mask)
        hard_neg_mask.scatter_(1, hard_neg_indices, 1.0)
        hard_neg_mask = hard_neg_mask * neg_mask  # safety

        # Denominator: positives + hard negatives
        denom_mask = pos_mask + hard_neg_mask
        denom_mask = denom_mask.clamp(max=1.0)

        # Log-sum-exp stability
        logits_max, _ = sim_matrix.max(dim=1, keepdim=True)
        logits = sim_matrix - logits_max.detach()

        exp_logits = torch.exp(logits) * denom_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

        # Mean log-prob over positive pairs
        # Weight by class importance
        sample_weights = self.class_weights[labels]  # (B,)
        pos_log_prob = (pos_mask * log_prob).sum(dim=1) / pos_mask.sum(dim=1).clamp(min=1)
        loss = -(sample_weights * pos_log_prob).mean()

        return loss


def cgc_extra_loss(z_corrected, labels, correction_model,
                   lambda_etf=2.0, lambda_con=1.0):
    """
    Combined extra loss for CGC v2.
    """
    etf_loss = correction_model.etf_alignment_loss()
    con_loss = correction_model.hard_negative_contrastive_loss(z_corrected, labels)

    total = lambda_etf * etf_loss + lambda_con * con_loss
    return total


def build_cgc(data_dict):
    """
    Factory function for run_method_sweep.
    """
    hidden_dim = min(256, data_dict['emb_dim'] // 2)
    hidden_dim = max(128, hidden_dim)

    model = CGCv2(
        emb_dim=data_dict['emb_dim'],
        num_classes=data_dict['num_classes'],
        hidden_dim=hidden_dim,
        class_counts=data_dict['class_counts'],
        hard_neg_k=5,
    )
    return model


print("Method 2 (CGC v2 - Simplex-ETF) model definition loaded.")

In [ ]:
# @title
# ============================================================
# METHOD 2 - CELL B: CGC v2 Training (3 seeds)
# ============================================================

METHOD_NAME = 'cgc_orig'

LAMBDA_ETF = 2.0    # simplex-ETF alignment (core novelty)
LAMBDA_CON = 1.0    # hard-negative contrastive

def cgc_loss_fn(z_corrected, labels, correction_model):
    """Wrapper with configured lambdas."""
    return cgc_extra_loss(
        z_corrected, labels, correction_model,
        lambda_etf=LAMBDA_ETF,
        lambda_con=LAMBDA_CON,
    )

SEEDS = [42, 123, 456]

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_method_name = f"{METHOD_NAME}_improved_s{seed}"

    print(f"\n\n{'#'*70}")
    print(f"# {METHOD_NAME} | SEED {seed}")
    print(f"{'#'*70}")

    results = run_method_sweep(
        method_name=seed_method_name,
        build_correction_fn=build_cgc,
        extra_loss_fn=cgc_loss_fn,
        num_epochs=100,
        lr=5e-4,
        batch_size=256,
        train_split='train',
    )

    print(f"\nSeed {seed} complete. {len(results)} runs.")

print_comparison_table()

In [ ]:
# @title
# ============================================================
# METHOD 2 - CELL B: CGC v2 Training (3 seeds)
# ============================================================

METHOD_NAME = 'cgc_ab_etf'

LAMBDA_ETF = 2.0    # simplex-ETF alignment (core novelty)
LAMBDA_CON = 1.0    # hard-negative contrastive

def cgc_etf_loss(z_corrected, labels, correction_model,
                   lambda_etf=2.0, lambda_con=1.0):
    """
    Combined extra loss for CGC v2.
    """
    etf_loss = correction_model.etf_alignment_loss()
    return lambda_etf * etf_loss
    # con_loss = correction_model.hard_negative_contrastive_loss(z_corrected, labels)

    # total = lambda_etf * etf_loss + lambda_con * con_loss
    # return total

SEEDS = [42, 123, 456]

print("#### ETF ONLY")

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_method_name = f"{METHOD_NAME}_improved_s{seed}"

    print(f"\n\n{'#'*70}")
    print(f"# {METHOD_NAME} | SEED {seed}")
    print(f"{'#'*70}")

    results = run_method_sweep(
        method_name=seed_method_name,
        build_correction_fn=build_cgc,
        extra_loss_fn=cgc_etf_loss,
        num_epochs=100,
        lr=5e-4,
        batch_size=256,
        train_split='train',
    )

    print(f"\nSeed {seed} complete. {len(results)} runs.")

print_comparison_table()

In [ ]:
# @title
# ============================================================
# METHOD 2 - CELL B: CGC v2 Training (3 seeds)
# ============================================================

METHOD_NAME = 'cgc_ab_con'

LAMBDA_ETF = 2.0    # simplex-ETF alignment (core novelty)
LAMBDA_CON = 1.0    # hard-negative contrastive

def cgc_con_loss(z_corrected, labels, correction_model,
                   lambda_etf=2.0, lambda_con=1.0):
    """
    Combined extra loss for CGC v2.
    """
    # etf_loss = correction_model.etf_alignment_loss()
    con_loss = correction_model.hard_negative_contrastive_loss(z_corrected, labels)

    return lambda_con * con_loss

    # total = lambda_etf * etf_loss + lambda_con * con_loss
    # return total

SEEDS = [42, 123, 456]

print("#### Contrastive ONLY")

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_method_name = f"{METHOD_NAME}_improved_s{seed}"

    print(f"\n\n{'#'*70}")
    print(f"# {METHOD_NAME} | SEED {seed}")
    print(f"{'#'*70}")

    results = run_method_sweep(
        method_name=seed_method_name,
        build_correction_fn=build_cgc,
        extra_loss_fn=cgc_con_loss,
        num_epochs=100,
        lr=5e-4,
        batch_size=256,
        train_split='train',
    )

    print(f"\nSeed {seed} complete. {len(results)} runs.")

print_comparison_table()

In [ ]:
# @title
# ============================================================
# METHOD 2 - CELL B: CGC v2 Training (3 seeds)
# ============================================================

METHOD_NAME = 'cgc_ab_cls'

LAMBDA_ETF = 2.0    # simplex-ETF alignment (core novelty)
LAMBDA_CON = 1.0    # hard-negative contrastive

SEEDS = [42, 123, 456]

print("#### CLS ONLY")

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_method_name = f"{METHOD_NAME}_improved_s{seed}"

    print(f"\n\n{'#'*70}")
    print(f"# {METHOD_NAME} | SEED {seed}")
    print(f"{'#'*70}")

    results = run_method_sweep(
        method_name=seed_method_name,
        build_correction_fn=build_cgc,
        extra_loss_fn=None,
        num_epochs=100,
        lr=5e-4,
        batch_size=256,
        train_split='train',
    )

    print(f"\nSeed {seed} complete. {len(results)} runs.")

print_comparison_table()

In [ ]:
# @title
# ============================================================
# METHOD 2 - CELL B: CGC v2 Training (3 seeds)
# ============================================================

METHOD_NAME = 'baseline'

LAMBDA_ETF = 2.0    # simplex-ETF alignment (core novelty)
LAMBDA_CON = 1.0    # hard-negative contrastive

def cgc_loss_fn(z_corrected, labels, correction_model):
    """Wrapper with configured lambdas."""
    return cgc_extra_loss(
        z_corrected, labels, correction_model,
        lambda_etf=LAMBDA_ETF,
        lambda_con=LAMBDA_CON,
    )

SEEDS = [42, 123, 456]

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    seed_method_name = f"{METHOD_NAME}_improved_s{seed}"

    print(f"\n\n{'#'*70}")
    print(f"# {METHOD_NAME} | SEED {seed}")
    print(f"{'#'*70}")

    results = run_method_sweep(
        method_name=seed_method_name,
        build_correction_fn=None,
        extra_loss_fn=None,
        num_epochs=100,
        lr=5e-4,
        batch_size=256,
        train_split='train',
    )

    print(f"\nSeed {seed} complete. {len(results)} runs.")

print_comparison_table()

In [ ]:
## Seeded average and STD

import pandas as pd
import numpy as np
import re
from pathlib import Path

# 1. Load the results
csv_path = Path('results.csv')
df = pd.read_csv(csv_path)

# 2. Extract Base Method and Seed
# Handles: 'cgc_ab_etf_improved_s123' -> 'cgc_ab_etf_improved' and '123'
# Handles: 'baseline_improved_s42' -> 'baseline_improved' and '42'
def extract_info(method_string):
    match = re.search(r'(.*)_s(\d+)$', str(method_string))
    if match:
        return match.group(1), match.group(2)
    return method_string, np.nan

df[['base_method', 'seed']] = df['method'].apply(lambda x: pd.Series(extract_info(x)))

# 3. Identify Metrics
# Define exactly what we want to calculate stats for
global_metrics = [
    'test_accuracy', 'test_f1_macro', 'test_f1_weighted', 'test_balanced_accuracy',
    'geo_tightness_corr_delta', 'geo_inter_sim_min_delta', 'geo_inter_sim_max_delta',
    'geo_inter_sim_mean_delta', 'geo_pca_1_delta', 'geo_pca_5_delta', 'geo_pca_10_delta'
]

# Dynamically find all per-class performance and intra-sim delta columns
per_class_metrics = [c for c in df.columns if 'test_acc_class_' in c or 'intra_sim_class_' in c and '_delta' in c]
all_target_metrics = global_metrics + per_class_metrics

# 4. Grouped Analysis
# We group by Dataset and Model first to create the 4 separate analyses
for (dataset_name, model_name), group_df in df.groupby(['dataset', 'model']):
    print(f"\n" + "="*80)
    print(f"ANALYSIS FOR: Dataset: {dataset_name} | Model: {model_name}")
    print("="*80)

    # Calculate Mean and Std across the 3 seeds (s42, s123, s456)
    summary = group_df.groupby('base_method')[all_target_metrics].agg(['mean', 'std'])

    # Flatten column names for CSV readability
    summary.columns = [f"{col[0]}_{col[1]}" for col in summary.columns]
    summary = summary.reset_index()

    # Drop any per-class columns that are empty for THIS specific dataset (e.g. class 6-8 for 6-class data)
    summary = summary.dropna(axis=1, how='all')

    # 5. Save the detailed CSV for this specific Model/Dataset combo
    output_fn = f"summary_{dataset_name}_{model_name}.csv"
    summary.to_csv(output_fn, index=False)
    print(f"Detailed CSV saved: {output_fn}")

    # 6. Print a clean summary table for the terminal
    # We'll show Baseline vs. Our Method vs. Ablations
    print(f"{'Method':<30} | {'Acc (Mean±Std)':<18} | {'F1 (Mean±Std)':<18} | {'PCA-1 Δ (Mean)'}")
    print("-" * 90)

    # Sort to ensure baseline is at the top or bottom for comparison
    summary = summary.sort_values('base_method')

    for _, row in summary.iterrows():
        acc_str = f"{row['test_accuracy_mean']:.3f}±{row['test_accuracy_std']:.3f}"
        f1_str = f"{row['test_f1_macro_mean']:.3f}±{row['test_f1_macro_std']:.3f}"
        pca_val = f"{row['geo_pca_1_delta_mean']:.4f}"
        print(f"{row['base_method']:<30} | {acc_str:<18} | {f1_str:<18} | {pca_val}")

### Visualization

In [ ]:
# @title
# ============================================================
# FINAL CELL: Complete Cross-Method Analysis (3-Seed Version)
# ============================================================
# Reads seeded results (e.g., ccsd_s42, ccsd_s123, ccsd_s456)
# Aggregates into mean +/- std per base method.
# Compares against external baselines (original frozen, oversampling).
# ============================================================

import scipy.stats as sp_stats

# ============================================================
# PART 0: Helper functions for seed aggregation
# ============================================================

def extract_base_method(method_name):
    """Extract base method name from seeded name.
    'ccsd_s42' -> 'ccsd'
    'baseline_s123' -> 'baseline'
    """
    for suffix in ['_s42', '_s123', '_s456']:
        if method_name.endswith(suffix):
            return method_name[:-len(suffix)]
    return method_name


def aggregate_seeds(df):
    """
    Aggregate seeded runs into mean +/- std per base method,
    model, dataset. Returns a DataFrame with one row per
    (base_method, model, dataset) combination.
    """
    df = df.copy()
    df['base_method'] = df['method'].apply(extract_base_method)

    metrics = ['test_accuracy', 'test_f1_macro', 'test_f1_weighted',
               'test_balanced_accuracy']

    # Find per-class accuracy columns
    acc_class_cols = [c for c in df.columns if c.startswith('test_acc_class_')]

    rows = []
    for (base_method, model, dataset), group in df.groupby(
            ['base_method', 'model', 'dataset']):

        row = {
            'method': base_method,
            'model': model,
            'dataset': dataset,
            'n_seeds': len(group),
        }

        for m in metrics:
            vals = group[m].dropna().values
            row[f'{m}_mean'] = np.mean(vals) if len(vals) > 0 else np.nan
            row[f'{m}_std'] = np.std(vals) if len(vals) > 1 else 0.0

        # Per-class accuracy
        for col in acc_class_cols:
            vals = group[col].dropna().values
            if len(vals) > 0:
                row[f'{col}_mean'] = np.mean(vals)
                row[f'{col}_std'] = np.std(vals) if len(vals) > 1 else 0.0

        rows.append(row)

    return pd.DataFrame(rows)


def fmt_mean_std(mean, std, decimals=4):
    """Format as mean+/-std."""
    return f"{mean:.{decimals}f}+/-{std:.{decimals}f}"


def fmt_delta(val):
    """Format signed delta."""
    return f"+{val:.4f}" if val >= 0 else f"{val:.4f}"


# ============================================================
# PART 1: Load and aggregate
# ============================================================

print(f"{'#'*80}")
print(f"# STEP 1: LOADING AND AGGREGATING RESULTS")
print(f"{'#'*80}")

df_raw = load_results()

if df_raw.empty:
    print("ERROR: No results found. Run method cells first.")
else:
    # Show raw method names
    raw_methods = sorted(df_raw['method'].unique())
    print(f"Raw methods found: {raw_methods}")
    print(f"Total raw runs: {len(df_raw)}")

    for m in raw_methods:
        print(f"  {m}: {len(df_raw[df_raw['method'] == m])} runs")

    # Aggregate
    df = aggregate_seeds(df_raw)
    base_methods = sorted(df['method'].unique())
    print(f"\nAggregated methods: {base_methods}")
    print(f"Aggregated rows: {len(df)}")

    for m in base_methods:
        mdf = df[df['method'] == m]
        seeds = mdf['n_seeds'].iloc[0] if not mdf.empty else 0
        print(f"  {m}: {len(mdf)} model-dataset combos, {seeds} seeds each")

    # ============================================================
    # External references (from prior experiments, not recomputed)
    # ============================================================

    # Original frozen-embedding baseline (2-layer MLP, no BatchNorm)
    ORIGINAL_BASELINES = {
        ('dofa', 'opensarship'):       {'accuracy': 0.5992, 'f1_macro': 0.3212, 'f1_weighted': 0.6335},
        ('ssl4eo', 'opensarship'):     {'accuracy': 0.5675, 'f1_macro': 0.3167, 'f1_weighted': 0.6193},
        ('scalemae', 'opensarship'):   {'accuracy': 0.5020, 'f1_macro': 0.2419, 'f1_weighted': 0.5591},
        ('prithvi', 'opensarship'):    {'accuracy': 0.3499, 'f1_macro': 0.2009, 'f1_weighted': 0.4433},
        ('sardet100k', 'opensarship'): {'accuracy': 0.6892, 'f1_macro': 0.3337, 'f1_weighted': 0.6939},
        ('sarjepa', 'opensarship'):    {'accuracy': 0.5218, 'f1_macro': 0.2696, 'f1_weighted': 0.5886},
        ('dofa', 'fusarship'):         {'accuracy': 0.6316, 'f1_macro': 0.3698, 'f1_weighted': 0.6362},
        ('ssl4eo', 'fusarship'):       {'accuracy': 0.6006, 'f1_macro': 0.4243, 'f1_weighted': 0.6036},
        ('scalemae', 'fusarship'):     {'accuracy': 0.4799, 'f1_macro': 0.3436, 'f1_weighted': 0.4954},
        ('prithvi', 'fusarship'):      {'accuracy': 0.4799, 'f1_macro': 0.3319, 'f1_weighted': 0.5015},
        ('sardet100k', 'fusarship'):   {'accuracy': 0.6316, 'f1_macro': 0.4770, 'f1_weighted': 0.6287},
        ('sarjepa', 'fusarship'):      {'accuracy': 0.6068, 'f1_macro': 0.4145, 'f1_weighted': 0.6233},
    }

    BEST_OVERSAMPLING = {
        ('dofa', 'opensarship'):       {'f1_macro': 0.4170, 'method': 'kmeans_smote'},
        ('ssl4eo', 'opensarship'):     {'f1_macro': 0.3765, 'method': 'svmsmote'},
        ('scalemae', 'opensarship'):   {'f1_macro': 0.2413, 'method': 'svmsmote'},
        ('prithvi', 'opensarship'):    {'f1_macro': 0.2320, 'method': 'kmeans_smote'},
        ('sardet100k', 'opensarship'): {'f1_macro': 0.3423, 'method': 'kmeans_smote'},
        ('sarjepa', 'opensarship'):    {'f1_macro': 0.2925, 'method': 'svmsmote'},
        ('dofa', 'fusarship'):         {'f1_macro': 0.3845, 'method': 'kmeans_smote'},
        ('ssl4eo', 'fusarship'):       {'f1_macro': 0.4243, 'method': 'original'},
        ('scalemae', 'fusarship'):     {'f1_macro': 0.3436, 'method': 'original'},
        ('prithvi', 'fusarship'):      {'f1_macro': 0.3555, 'method': 'adasyn'},
        ('sardet100k', 'fusarship'):   {'f1_macro': 0.4873, 'method': 'svmsmote'},
        ('sarjepa', 'fusarship'):      {'f1_macro': 0.4345, 'method': 'svmsmote'},
    }

    KNN_BALACC_ORIGINAL = {
        ('dofa', 'opensarship'):       0.2722,
        ('ssl4eo', 'opensarship'):     0.2405,
        ('scalemae', 'opensarship'):   0.2107,
        ('prithvi', 'opensarship'):    0.1944,
        ('sardet100k', 'opensarship'): 0.2348,
        ('sarjepa', 'opensarship'):    0.2152,
        ('dofa', 'fusarship'):         0.3537,
        ('ssl4eo', 'fusarship'):       0.3481,
        ('scalemae', 'fusarship'):     0.2382,
        ('prithvi', 'fusarship'):      0.2730,
        ('sardet100k', 'fusarship'):   0.3872,
        ('sarjepa', 'fusarship'):      0.3330,
    }

    KNN_BALACC_SMOTEENN = {
        ('dofa', 'opensarship'):       0.4238,
        ('ssl4eo', 'opensarship'):     0.3635,
        ('scalemae', 'opensarship'):   0.2729,
        ('prithvi', 'opensarship'):    0.2528,
        ('sardet100k', 'opensarship'): 0.3756,
        ('sarjepa', 'opensarship'):    0.3096,
        ('dofa', 'fusarship'):         0.3535,
        ('ssl4eo', 'fusarship'):       0.4429,
        ('scalemae', 'fusarship'):     0.2925,
        ('prithvi', 'fusarship'):      0.3144,
        ('sardet100k', 'fusarship'):   0.3999,
        ('sarjepa', 'fusarship'):      0.3861,
    }

    PERCLASS_KNN_BASELINE = {
        'opensarship': {
            'dofa':       {'Cargo': 0.909, 'Tanker': 0.337, 'Dredging': 0.069, 'Fishing': 0.241, 'Passenger': 0.000, 'Tug': 0.077},
            'ssl4eo':     {'Cargo': 0.905, 'Tanker': 0.332, 'Dredging': 0.000, 'Fishing': 0.207, 'Passenger': 0.000, 'Tug': 0.000},
            'scalemae':   {'Cargo': 0.895, 'Tanker': 0.162, 'Dredging': 0.000, 'Fishing': 0.207, 'Passenger': 0.000, 'Tug': 0.000},
            'prithvi':    {'Cargo': 0.870, 'Tanker': 0.227, 'Dredging': 0.000, 'Fishing': 0.069, 'Passenger': 0.000, 'Tug': 0.000},
            'sardet100k': {'Cargo': 0.896, 'Tanker': 0.271, 'Dredging': 0.000, 'Fishing': 0.241, 'Passenger': 0.000, 'Tug': 0.000},
            'sarjepa':    {'Cargo': 0.904, 'Tanker': 0.241, 'Dredging': 0.000, 'Fishing': 0.069, 'Passenger': 0.000, 'Tug': 0.077},
        },
        'fusarship': {
            'dofa':       {'Cargo': 0.800, 'Fishing': 0.342, 'Bulk': 0.786, 'Tanker': 0.375, 'Container': 0.571, 'Dredging': 0.143, 'Tug': 0.167, 'GeneralCargo': 0.000, 'Passenger': 0.000},
            'ssl4eo':     {'Cargo': 0.835, 'Fishing': 0.392, 'Bulk': 0.821, 'Tanker': 0.312, 'Container': 0.429, 'Dredging': 0.143, 'Tug': 0.000, 'GeneralCargo': 0.000, 'Passenger': 0.200},
            'scalemae':   {'Cargo': 0.759, 'Fishing': 0.367, 'Bulk': 0.607, 'Tanker': 0.125, 'Container': 0.286, 'Dredging': 0.000, 'Tug': 0.000, 'GeneralCargo': 0.000, 'Passenger': 0.000},
            'prithvi':    {'Cargo': 0.782, 'Fishing': 0.380, 'Bulk': 0.571, 'Tanker': 0.438, 'Container': 0.286, 'Dredging': 0.000, 'Tug': 0.000, 'GeneralCargo': 0.000, 'Passenger': 0.000},
            'sardet100k': {'Cargo': 0.771, 'Fishing': 0.532, 'Bulk': 0.750, 'Tanker': 0.375, 'Container': 0.571, 'Dredging': 0.286, 'Tug': 0.000, 'GeneralCargo': 0.000, 'Passenger': 0.200},
            'sarjepa':    {'Cargo': 0.824, 'Fishing': 0.405, 'Bulk': 0.786, 'Tanker': 0.188, 'Container': 0.286, 'Dredging': 0.143, 'Tug': 0.167, 'GeneralCargo': 0.000, 'Passenger': 0.200},
        },
    }

    # Novel methods only (exclude baseline for some tables)
    NOVEL_METHODS = [m for m in base_methods if m != 'baseline']

    # ============================================================
    # Compute trained baseline from aggregated seeds
    # ============================================================
    TRAINED_BASELINE = {}
    baseline_df = df[df['method'] == 'baseline']
    for _, row in baseline_df.iterrows():
        key = (row['model'], row['dataset'])
        TRAINED_BASELINE[key] = {
            'accuracy_mean': row['test_accuracy_mean'],
            'accuracy_std': row['test_accuracy_std'],
            'f1_macro_mean': row['test_f1_macro_mean'],
            'f1_macro_std': row['test_f1_macro_std'],
            'balanced_accuracy_mean': row['test_balanced_accuracy_mean'],
            'balanced_accuracy_std': row['test_balanced_accuracy_std'],
        }

    # ============================================================
    # TABLE A: Main Results (Mean +/- Std) per Dataset
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE A: MAIN RESULTS (mean+/-std) - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12} {'Method':<12} {'Seeds':>5} {'Accuracy':>16} "
              f"{'F1-Macro':>16} {'BalAcc':>16}")
        print(f"{'-'*85}")

        for model_name in MODELS:
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if row.empty:
                    continue
                r = row.iloc[0]
                acc = fmt_mean_std(r['test_accuracy_mean'], r['test_accuracy_std'])
                f1m = fmt_mean_std(r['test_f1_macro_mean'], r['test_f1_macro_std'])
                ba = fmt_mean_std(r['test_balanced_accuracy_mean'], r['test_balanced_accuracy_std'])
                print(f"{model_name:<12} {m:<12} {int(r['n_seeds']):>5} "
                      f"{acc:>16} {f1m:>16} {ba:>16}")
            print()  # blank line between models

    # ============================================================
    # TABLE B: F1-Macro Pivot (Mean only, compact)
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE B: F1-Macro PIVOT (mean) - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12} {'OrigBase':>8}", end="")
        for m in methods_present:
            print(f" {m:>10}", end="")
        print(f" {'BestOS':>10}")
        print(f"{'-'*(14 + 10 + len(methods_present)*11 + 11)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            orig_f1m = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', float('nan'))
            best_os_f1m = BEST_OVERSAMPLING.get(key, {}).get('f1_macro', float('nan'))

            print(f"{model_name:<12} {orig_f1m:>8.4f}", end="")
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    print(f" {row.iloc[0]['test_f1_macro_mean']:>10.4f}", end="")
                else:
                    print(f" {'---':>10}", end="")
            print(f" {best_os_f1m:>10.4f}")

        # Averages
        print(f"{'AVERAGE':<12} ", end="")
        avg_orig = np.mean([ORIGINAL_BASELINES.get((m, dataset_name), {}).get('f1_macro', 0)
                            for m in MODELS])
        print(f"{avg_orig:>7.4f}", end="")
        for m in methods_present:
            mdf = subset[subset['method'] == m]
            if not mdf.empty:
                print(f" {mdf['test_f1_macro_mean'].mean():>10.4f}", end="")
            else:
                print(f" {'---':>10}", end="")
        avg_os = np.mean([BEST_OVERSAMPLING.get((m, dataset_name), {}).get('f1_macro', 0)
                          for m in MODELS])
        print(f" {avg_os:>10.4f}")

    # ============================================================
    # TABLE C: F1-Macro Pivot (Mean +/- Std, paper-ready)
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE C: F1-Macro (mean+/-std) - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12} {'OrigBase':>8}", end="")
        for m in methods_present:
            print(f" {m:>18}", end="")
        print()
        print(f"{'-'*(14 + 8 + len(methods_present)*19)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            orig_f1m = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', float('nan'))
            print(f"{model_name:<12} {orig_f1m:>8.4f}", end="")
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    r = row.iloc[0]
                    val = fmt_mean_std(r['test_f1_macro_mean'], r['test_f1_macro_std'])
                    print(f" {val:>18}", end="")
                else:
                    print(f" {'---':>18}", end="")
            print()

    # ============================================================
    # TABLE D: Delta vs Original Baseline (Mean F1-Macro)
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE D: F1-Macro DELTA vs ORIGINAL BASELINE - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12}", end="")
        for m in methods_present:
            print(f" {m:>10}", end="")
        print(f" {'BestOS d':>10}")
        print(f"{'-'*(14 + len(methods_present)*11 + 11)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            orig_f1m = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', 0)
            best_os_delta = BEST_OVERSAMPLING.get(key, {}).get('f1_macro', 0) - orig_f1m

            print(f"{model_name:<12}", end="")
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    delta = row.iloc[0]['test_f1_macro_mean'] - orig_f1m
                    print(f" {fmt_delta(delta):>10}", end="")
                else:
                    print(f" {'---':>10}", end="")
            print(f" {fmt_delta(best_os_delta):>10}")

        # Averages
        print(f"{'AVERAGE':<12}", end="")
        for m in methods_present:
            mdf = subset[subset['method'] == m]
            deltas = []
            for _, row in mdf.iterrows():
                key = (row['model'], row['dataset'])
                orig = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', 0)
                deltas.append(row['test_f1_macro_mean'] - orig)
            if deltas:
                print(f" {fmt_delta(np.mean(deltas)):>10}", end="")
            else:
                print(f" {'---':>10}", end="")
        avg_os_delta = np.mean([
            BEST_OVERSAMPLING.get((m, dataset_name), {}).get('f1_macro', 0) -
            ORIGINAL_BASELINES.get((m, dataset_name), {}).get('f1_macro', 0)
            for m in MODELS
        ])
        print(f" {fmt_delta(avg_os_delta):>10}")

    # ============================================================
    # TABLE E: Delta vs Trained Baseline (Mean F1-Macro)
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE E: F1-Macro DELTA vs TRAINED BASELINE (3-seed) - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted([m for m in subset['method'].unique() if m != 'baseline'])
        print(f"{'Model':<12} {'TrainBase':>9}", end="")
        for m in methods_present:
            print(f" {m:>10}", end="")
        print()
        print(f"{'-'*(14 + 10 + len(methods_present)*11)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            tb = TRAINED_BASELINE.get(key, {})
            tb_f1m = tb.get('f1_macro_mean', 0)

            print(f"{model_name:<12} {tb_f1m:>9.4f}", end="")
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    delta = row.iloc[0]['test_f1_macro_mean'] - tb_f1m
                    print(f" {fmt_delta(delta):>10}", end="")
                else:
                    print(f" {'---':>10}", end="")
            print()

        # Averages
        print(f"{'AVERAGE':<12} ", end="")
        avg_tb = np.mean([TRAINED_BASELINE.get((m, dataset_name), {}).get('f1_macro_mean', 0)
                          for m in MODELS])
        print(f"{avg_tb:>8.4f}", end="")
        for m in methods_present:
            mdf = subset[subset['method'] == m]
            deltas = []
            for _, row in mdf.iterrows():
                key = (row['model'], row['dataset'])
                tb = TRAINED_BASELINE.get(key, {}).get('f1_macro_mean', 0)
                deltas.append(row['test_f1_macro_mean'] - tb)
            if deltas:
                print(f" {fmt_delta(np.mean(deltas)):>10}", end="")
            else:
                print(f" {'---':>10}", end="")
        print()

    # ============================================================
    # TABLE F: Delta vs Best Oversampling
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE F: F1-Macro DELTA vs BEST OVERSAMPLING - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12} {'BestOS':>8} {'OSMethod':>12}", end="")
        for m in methods_present:
            print(f" {m:>10}", end="")
        print()
        print(f"{'-'*(14 + 22 + len(methods_present)*11)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            best_os = BEST_OVERSAMPLING.get(key, {})
            best_os_f1 = best_os.get('f1_macro', 0)
            best_os_method = best_os.get('method', '---')

            print(f"{model_name:<12} {best_os_f1:>8.4f} {best_os_method:>12}", end="")
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    delta = row.iloc[0]['test_f1_macro_mean'] - best_os_f1
                    print(f" {fmt_delta(delta):>10}", end="")
                else:
                    print(f" {'---':>10}", end="")
            print()

    # ============================================================
    # TABLE G: Balanced Accuracy Pivot
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE G: BALANCED ACCURACY PIVOT - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12} {'kNN Orig':>8} {'kNN SENN':>8}", end="")
        for m in methods_present:
            print(f" {m:>10}", end="")
        print()
        print(f"{'-'*(14 + 18 + len(methods_present)*11)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            knn_orig = KNN_BALACC_ORIGINAL.get(key, float('nan'))
            knn_senn = KNN_BALACC_SMOTEENN.get(key, float('nan'))
            print(f"{model_name:<12} {knn_orig:>8.4f} {knn_senn:>8.4f}", end="")
            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    print(f" {row.iloc[0]['test_balanced_accuracy_mean']:>10.4f}", end="")
                else:
                    print(f" {'---':>10}", end="")
            print()

    # ============================================================
    # TABLE H: Per-Class Accuracy (Mean across seeds)
    # ============================================================
    for dataset_name in DATASETS:
        print(f"\n\n{'#'*80}")
        print(f"# TABLE H: PER-CLASS TEST ACCURACY (mean) - {dataset_name.upper()}")
        print(f"{'#'*80}")

        for model_name in MODELS:
            print(f"\n  --- {model_name} | {dataset_name} ---")

            baseline_perclass = PERCLASS_KNN_BASELINE.get(dataset_name, {}).get(model_name, {})
            class_names = sorted(baseline_perclass.keys()) if baseline_perclass else []
            if not class_names:
                print(f"  No baseline per-class data available.")
                continue

            try:
                data = prepare_data(model_name, dataset_name)
                idx_to_name = data['label_names']
                name_to_idx = {v: k for k, v in idx_to_name.items()}
                del data
            except:
                continue

            print(f"  {'Method':<12}", end="")
            for cn in class_names:
                print(f" {cn[:8]:>8}", end="")
            print()
            print(f"  {'-'*(14 + len(class_names)*9)}")

            # kNN baseline row
            print(f"  {'kNN-orig':<12}", end="")
            for cn in class_names:
                print(f" {baseline_perclass.get(cn, float('nan')):>8.3f}", end="")
            print()

            # Each method
            subset = df[(df['dataset'] == dataset_name) & (df['model'] == model_name)]
            for _, row in subset.sort_values('method').iterrows():
                print(f"  {row['method']:<12}", end="")
                for cn in class_names:
                    c_idx = name_to_idx.get(cn, None)
                    col = f'test_acc_class_{c_idx}_mean' if c_idx is not None else None
                    if col and col in row.index and pd.notna(row[col]):
                        print(f" {row[col]:>8.3f}", end="")
                    else:
                        print(f" {'---':>8}", end="")
                print()

    # ============================================================
    # TABLE I: Zero-Accuracy Class Recovery
    # ============================================================
    print(f"\n\n{'#'*80}")
    print(f"# TABLE I: ZERO-ACCURACY CLASS RECOVERY (mean across seeds)")
    print(f"{'#'*80}")

    ZERO_CLASSES = {
        'opensarship': ['Passenger', 'Tug', 'Dredging'],
        'fusarship': ['Passenger', 'GeneralCargo', 'Tug'],
    }

    for dataset_name in DATASETS:
        target_classes = ZERO_CLASSES.get(dataset_name, [])
        if not target_classes:
            continue

        print(f"\n  --- {dataset_name.upper()} ---")
        print(f"  Target classes: {target_classes}")

        for model_name in MODELS:
            baseline_perclass = PERCLASS_KNN_BASELINE.get(
                dataset_name, {}).get(model_name, {})

            try:
                data = prepare_data(model_name, dataset_name)
                name_to_idx = {v: k for k, v in data['label_names'].items()}
                del data
            except:
                continue

            print(f"\n  {model_name}:")
            print(f"  {'Method':<12}", end="")
            for cn in target_classes:
                print(f" {cn:>12}", end="")
            print()
            print(f"  {'-'*(14 + len(target_classes)*13)}")

            # kNN baseline
            print(f"  {'kNN-base':<12}", end="")
            for cn in target_classes:
                print(f" {baseline_perclass.get(cn, float('nan')):>12.3f}", end="")
            print()

            # Each method
            subset = df[(df['dataset'] == dataset_name) & (df['model'] == model_name)]
            for _, row in subset.sort_values('method').iterrows():
                print(f"  {row['method']:<12}", end="")
                for cn in target_classes:
                    c_idx = name_to_idx.get(cn, None)
                    col = f'test_acc_class_{c_idx}_mean' if c_idx is not None else None
                    if col and col in row.index and pd.notna(row[col]):
                        val = row[col]
                        base_val = baseline_perclass.get(cn, 0)
                        flag = " *" if val > 0.01 and base_val < 0.01 else "  "
                        print(f" {val:>8.3f}{flag}  ", end="")
                    else:
                        print(f" {'---':>12}", end="")
                print()
            print(f"  (* = recovered from 0%)")

    # ============================================================
    # TABLE J: Method Ranking (by mean F1-Macro)
    # ============================================================
    print(f"\n\n{'#'*80}")
    print(f"# TABLE J: METHOD RANKING BY MEAN F1-MACRO")
    print(f"{'#'*80}")

    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n  --- {dataset_name.upper()} ---")
        methods_present = sorted(subset['method'].unique())

        method_stats = []
        for m in methods_present:
            mdf = subset[subset['method'] == m]
            avg_f1m = mdf['test_f1_macro_mean'].mean()
            avg_ba = mdf['test_balanced_accuracy_mean'].mean()
            avg_acc = mdf['test_accuracy_mean'].mean()

            deltas_orig = []
            deltas_os = []
            deltas_tb = []
            for _, row in mdf.iterrows():
                key = (row['model'], row['dataset'])
                orig = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', 0)
                os_best = BEST_OVERSAMPLING.get(key, {}).get('f1_macro', 0)
                tb = TRAINED_BASELINE.get(key, {}).get('f1_macro_mean', 0)
                deltas_orig.append(row['test_f1_macro_mean'] - orig)
                deltas_os.append(row['test_f1_macro_mean'] - os_best)
                deltas_tb.append(row['test_f1_macro_mean'] - tb)

            # Wins
            wins = 0
            for model_name in MODELS:
                model_rows = subset[subset['model'] == model_name]
                if model_rows.empty:
                    continue
                best_idx = model_rows['test_f1_macro_mean'].idxmax()
                if model_rows.loc[best_idx, 'method'] == m:
                    wins += 1

            method_stats.append({
                'method': m,
                'avg_f1m': avg_f1m,
                'avg_ba': avg_ba,
                'avg_acc': avg_acc,
                'delta_orig': np.mean(deltas_orig),
                'delta_os': np.mean(deltas_os),
                'delta_tb': np.mean(deltas_tb),
                'wins': wins,
            })

        method_stats.sort(key=lambda x: x['avg_f1m'], reverse=True)

        print(f"  {'Rank':>4} {'Method':<12} {'Avg F1M':>8} {'Avg BA':>8} "
              f"{'d/Orig':>8} {'d/OS':>8} {'d/Train':>8} {'Wins':>5}")
        print(f"  {'-'*75}")

        for rank, ms in enumerate(method_stats, 1):
            print(f"  {rank:>4} {ms['method']:<12} {ms['avg_f1m']:>8.4f} "
                  f"{ms['avg_ba']:>8.4f} "
                  f"{fmt_delta(ms['delta_orig']):>8} "
                  f"{fmt_delta(ms['delta_os']):>8} "
                  f"{fmt_delta(ms['delta_tb']):>8} "
                  f"{ms['wins']:>5}/{len(MODELS)}")

    # ============================================================
    # TABLE K: Accuracy Preservation
    # ============================================================
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n\n{'#'*80}")
        print(f"# TABLE K: ACCURACY PRESERVATION - {dataset_name.upper()}")
        print(f"{'#'*80}")

        methods_present = sorted(subset['method'].unique())
        print(f"{'Model':<12} {'OrigAcc':>7}", end="")
        for m in methods_present:
            print(f" {m:>14}", end="")
        print()
        print(f"{'-'*(14 + 8 + len(methods_present)*15)}")

        for model_name in MODELS:
            key = (model_name, dataset_name)
            orig_acc = ORIGINAL_BASELINES.get(key, {}).get('accuracy', 0)
            print(f"{model_name:<12} {orig_acc:>7.4f}", end="")

            for m in methods_present:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    r = row.iloc[0]
                    acc_mean = r['test_accuracy_mean']
                    delta = acc_mean - orig_acc
                    print(f" {acc_mean:.4f}({fmt_delta(delta)})", end="")
                else:
                    print(f" {'---':>14}", end="")
            print()

    # ============================================================
    # TABLE L: Statistical Summary
    # ============================================================
    print(f"\n\n{'#'*80}")
    print(f"# TABLE L: STATISTICAL SUMMARY (mean +/- std across model-datasets)")
    print(f"{'#'*80}")

    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n  --- {dataset_name.upper()} ---")
        methods_present = sorted(subset['method'].unique())

        for m in methods_present:
            mdf = subset[subset['method'] == m]
            f1_means = mdf['test_f1_macro_mean'].values
            f1_stds = mdf['test_f1_macro_std'].values
            ba_means = mdf['test_balanced_accuracy_mean'].values
            acc_means = mdf['test_accuracy_mean'].values

            print(f"\n  {m} ({int(mdf.iloc[0]['n_seeds'])} seeds):")
            print(f"    F1-Macro:     avg_mean={np.mean(f1_means):.4f}  "
                  f"avg_seed_std={np.mean(f1_stds):.4f}  "
                  f"cross_model_std={np.std(f1_means):.4f}")
            print(f"    Balanced Acc: avg_mean={np.mean(ba_means):.4f}  "
                  f"cross_model_std={np.std(ba_means):.4f}")
            print(f"    Accuracy:     avg_mean={np.mean(acc_means):.4f}  "
                  f"cross_model_std={np.std(acc_means):.4f}")

    # ============================================================
    # TABLE M: Best Method Per Model-Dataset
    # ============================================================
    print(f"\n\n{'#'*80}")
    print(f"# TABLE M: BEST METHOD PER MODEL-DATASET")
    print(f"{'#'*80}")
    print(f"{'Model':<12} {'Dataset':<14} {'Best Method':<12} {'F1-Mac':>9} "
          f"{'OrigBase':>8} {'BestOS':>8} {'Improve':>9}")
    print(f"{'-'*80}")

    for dataset_name in DATASETS:
        for model_name in MODELS:
            subset = df[(df['dataset'] == dataset_name) & (df['model'] == model_name)]
            if subset.empty:
                continue

            best_row = subset.loc[subset['test_f1_macro_mean'].idxmax()]
            best_method = best_row['method']
            best_f1m = best_row['test_f1_macro_mean']
            best_std = best_row['test_f1_macro_std']

            key = (model_name, dataset_name)
            orig = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', 0)
            os_best = BEST_OVERSAMPLING.get(key, {}).get('f1_macro', 0)
            reference = max(orig, os_best)
            improvement = best_f1m - reference

            print(f"{model_name:<12} {dataset_name:<14} {best_method:<12} "
                  f"{best_f1m:.4f}+/-{best_std:.4f} "
                  f"{orig:>8.4f} {os_best:>8.4f} {fmt_delta(improvement):>9}")

    # ============================================================
    # PLOTS
    # ============================================================
    print(f"\n\n{'#'*80}")
    print(f"# GENERATING PLOTS")
    print(f"{'#'*80}")

    # Grouped bar chart with error bars
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        methods_present = sorted(subset['method'].unique())
        n_methods = len(methods_present)

        fig, ax = plt.subplots(figsize=(max(14, n_methods * 2), 6))
        x = np.arange(len(MODELS))
        width = 0.8 / (n_methods + 2)

        # Original baseline
        orig_vals = [ORIGINAL_BASELINES.get((m, dataset_name), {}).get('f1_macro', 0)
                     for m in MODELS]
        ax.bar(x - width * (n_methods + 1) / 2, orig_vals, width,
               label='Orig Baseline', color='lightgray', edgecolor='black', linewidth=0.5)

        # Best oversampling
        os_vals = [BEST_OVERSAMPLING.get((m, dataset_name), {}).get('f1_macro', 0)
                   for m in MODELS]
        ax.bar(x - width * (n_methods - 1) / 2, os_vals, width,
               label='Best Oversampling', color='darkgray', edgecolor='black', linewidth=0.5)

        # Methods with error bars
        colors = plt.cm.Set2(np.linspace(0, 1, n_methods))
        for i, m in enumerate(methods_present):
            means = []
            stds = []
            for model_name in MODELS:
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    means.append(row.iloc[0]['test_f1_macro_mean'])
                    stds.append(row.iloc[0]['test_f1_macro_std'])
                else:
                    means.append(0)
                    stds.append(0)
            offset = x + width * (i - (n_methods - 3) / 2)
            ax.bar(offset, means, width, yerr=stds, label=m, color=colors[i],
                   edgecolor='black', linewidth=0.5, capsize=2)

        ax.set_xlabel('Foundation Model')
        ax.set_ylabel('F1-Macro (mean +/- std)')
        ax.set_title(f'{dataset_name} - F1-Macro: All Methods (3 seeds)')
        ax.set_xticks(x)
        ax.set_xticklabels(MODELS, rotation=45)
        ax.legend(fontsize=7, ncol=3, loc='upper right')
        ax.set_ylim(0, 0.7)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()

        path = PLOTS_DIR / f"final_3seed_{dataset_name}_bar.png"
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"  Saved: {path}")

    # Delta chart with error bars
    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        methods_present = sorted(subset['method'].unique())

        fig, ax = plt.subplots(figsize=(12, 5))
        x = np.arange(len(MODELS))
        width = 0.8 / len(methods_present)
        colors = plt.cm.tab10(np.linspace(0, 1, len(methods_present)))

        for i, m in enumerate(methods_present):
            deltas = []
            stds = []
            for model_name in MODELS:
                key = (model_name, dataset_name)
                orig = ORIGINAL_BASELINES.get(key, {}).get('f1_macro', 0)
                row = subset[(subset['model'] == model_name) & (subset['method'] == m)]
                if not row.empty:
                    deltas.append(row.iloc[0]['test_f1_macro_mean'] - orig)
                    stds.append(row.iloc[0]['test_f1_macro_std'])
                else:
                    deltas.append(0)
                    stds.append(0)
            offset = x + width * (i - len(methods_present) / 2 + 0.5)
            ax.bar(offset, deltas, width, yerr=stds, label=m, color=colors[i],
                   edgecolor='black', linewidth=0.5, capsize=2)

        ax.axhline(y=0, color='black', linewidth=1)
        ax.set_xlabel('Foundation Model')
        ax.set_ylabel('F1-Macro Delta vs Original Baseline')
        ax.set_title(f'{dataset_name} - Improvement (mean +/- std, 3 seeds)')
        ax.set_xticks(x)
        ax.set_xticklabels(MODELS, rotation=45)
        ax.legend(fontsize=7, ncol=3)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()

        path = PLOTS_DIR / f"final_3seed_delta_{dataset_name}_bar.png"
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"  Saved: {path}")

    # ============================================================
    # FINAL SUMMARY
    # ============================================================
    print(f"\n\n{'#'*80}")
    print(f"# FINAL SUMMARY")
    print(f"{'#'*80}")

    print(f"\n  Total raw runs: {len(df_raw)}")
    print(f"  Seeds per method: {SEEDS}")
    print(f"  Aggregated methods: {base_methods}")
    print(f"  Models: {MODELS}")
    print(f"  Datasets: {DATASETS}")

    for dataset_name in DATASETS:
        subset = df[df['dataset'] == dataset_name]
        if subset.empty:
            continue

        print(f"\n  === {dataset_name.upper()} ===")

        best_row = subset.loc[subset['test_f1_macro_mean'].idxmax()]
        print(f"  Overall best: {best_row['method']} on {best_row['model']} "
              f"(F1-Macro={best_row['test_f1_macro_mean']:.4f}"
              f"+/-{best_row['test_f1_macro_std']:.4f})")

        method_avgs = subset.groupby('method')['test_f1_macro_mean'].mean().sort_values(ascending=False)
        print(f"  Method ranking by avg F1-Macro:")
        for rank, (method, avg) in enumerate(method_avgs.items(), 1):
            orig_avg = np.mean([ORIGINAL_BASELINES.get((m, dataset_name), {}).get('f1_macro', 0)
                                for m in MODELS])
            delta = avg - orig_avg
            print(f"    {rank}. {method:<12} avg={avg:.4f} (delta vs orig: {fmt_delta(delta)})")

    # Save aggregated results
    agg_path = OUTPUT_ROOT / 'aggregated_3seed_results.csv'
    df.to_csv(agg_path, index=False)
    print(f"\n  Aggregated results saved to: {agg_path}")

    print(f"\n{'#'*80}")
    print(f"# DONE. PASTE EVERYTHING ABOVE FOR ANALYSIS.")
    print(f"{'#'*80}")

# Understanding results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load your CSV
df = pd.read_csv('/content/drive/MyDrive/PhD Research/iclrFoundationExp/aggregated_3seed_results.csv')

print(f"Methods: {df['method'].unique()}")
print(f"Models: {df['model'].unique()}")
print(f"Datasets: {df['dataset'].unique()}")

# Groupings for baseline/OS comparison -- you should define these according to your process
BASELINE_METHOD = 'baseline'

# 1. AGGREGATE F1-MACRO BAR PLOT (Method vs. F1-Macro, by dataset)
for dataset in df['dataset'].unique():
    dd = df[df['dataset'] == dataset]
    plt.figure(figsize=(10,5))
    sns.barplot(data=dd, x='method', y='test_f1_macro_mean', ci=None)
    plt.title(f'Mean F1-Macro by Method ({dataset})')
    plt.xticks(rotation=45)
    plt.ylabel('F1-Macro (mean across models, seeds)')
    plt.tight_layout()
    plt.show()

# 2. DELTA VS BASELINE/BEST OS (for each dataset/model)
for dataset in df['dataset'].unique():
    plt.figure(figsize=(10,5))
    models = df['model'].unique()
    baseline = df[(df['method'] == BASELINE_METHOD) & (df['dataset'] == dataset)]
    b_f1s = baseline.set_index('model')['test_f1_macro_mean']
    for method in df['method'].unique():
        dd = df[(df['method'] == method) & (df['dataset'] == dataset)]
        f1s = []
        for m in models:
            mf1 = dd[dd['model'] == m]['test_f1_macro_mean']
            if len(mf1) > 0:
                f1s.append(mf1.iloc[0] - b_f1s.get(m, np.nan))
            else:
                f1s.append(np.nan)
        plt.plot(models, f1s, label=method)
    plt.ylabel('Δ F1-Macro vs Baseline')
    plt.xlabel('Model')
    plt.title(f'Delta F1-Macro vs Baseline ({dataset})')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.xticks(rotation=45)
    plt.show()

# 3. PER-CLASS ACCURACY HEATMAP (for every method/dataset/model)
n_classes = max(
    [int(c.split('_')[-1]) for c in df.columns if c.startswith('test_acc_class_')]
) + 1
class_cols = [f'test_acc_class_{i}_mean' for i in range(n_classes)]

for dataset in df['dataset'].unique():
    for model in df['model'].unique():
        subset = df[(df['dataset'] == dataset) & (df['model'] == model)]
        class_matrix = []
        for _, row in subset.iterrows():
            class_matrix.append([row[c] if c in row else np.nan for c in class_cols])
        plt.figure(figsize=(10,6))
        sns.heatmap(class_matrix, annot=True, fmt='.2f',
                    yticklabels=subset['method'].values, xticklabels=[f'Class{i}' for i in range(n_classes)],
                    cmap='YlGnBu', vmin=0, vmax=1)
        plt.title(f'Per-Class Test Acc by Method (Model: {model}, Dataset: {dataset})')
        plt.xlabel('Class')
        plt.ylabel('Method')
        plt.tight_layout()
        plt.show()
        # This is verbose, comment out some if too many methods/models.

# 4. FRACTION OF RARE/TAIL CLASSES "RECOVERED" (acc > 0 threshold)
# Define tail/rare classes as the ones with lowest class index per dataset info
TAIL_CLASSES = {'opensarship': [4,5], 'fusarship': [6,7,8]}  # example, adjust as appropriate!
RECOVERY_THRESH = 0.1

def fraction_tail_recovered(row, dataset, tail_classes=TAIL_CLASSES, thresh=RECOVERY_THRESH):
    count = 0
    total = 0
    for i in tail_classes.get(dataset, []):
        cval = row.get(f'test_acc_class_{i}_mean', 0.0)
        total += 1
        if cval > thresh:
            count += 1
    return count / total if total > 0 else np.nan
df['fraction_tail_recovered'] = df.apply(lambda r: fraction_tail_recovered(r, r['dataset']), axis=1)

for dataset in df['dataset'].unique():
    dd = df[df['dataset'] == dataset]
    plt.figure(figsize=(10,5))
    sns.barplot(data=dd, x='method', y='fraction_tail_recovered', ci=None)
    plt.title(f'Fraction of Tail Classes Recovered (Acc>{RECOVERY_THRESH}, {dataset})')
    plt.xticks(rotation=45)
    plt.ylabel('Fraction of Tail Classes with acc > %.2f' % RECOVERY_THRESH)
    plt.tight_layout()
    plt.show()

# 5. SEED ROBUSTNESS: F1-Macro MEAN +/- STD PER METHOD/MODEL
for dataset in df['dataset'].unique():
    for model in df['model'].unique():
        dd = df[(df['dataset'] == dataset) & (df['model'] == model)]
        if dd.empty: continue
        plt.figure(figsize=(10,5))
        plt.errorbar(dd['method'], dd['test_f1_macro_mean'], dd['test_f1_macro_std'], fmt='o')
        plt.title(f'F1-Macro mean +/- std by method (Model: {model}, Dataset: {dataset})')
        plt.xticks(rotation=45)
        plt.ylabel('F1-Macro')
        plt.tight_layout()
        plt.show()

# 6. PRINT LEADERBOARDS
for dataset in df['dataset'].unique():
    best = df[df['dataset'] == dataset].groupby('method')['test_f1_macro_mean'].mean().sort_values(ascending=False)
    print(f'Leaderboard for {dataset} (mean F1-Macro):')
    print(best)
    print()

# 7. TAIL CLASS RECOVERY PRINTOUT (Tabular, for report)
for dataset in df['dataset'].unique():
    tail = TAIL_CLASSES[dataset]
    print(f"\nTail class recovery for {dataset}:")
    for _, row in df[df['dataset'] == dataset].iterrows():
        vals = [row[f'test_acc_class_{i}_mean'] for i in tail]
        method = row['method']
        print(f"{method:15} : " + " | ".join([f"C{i}:{v:.2f}" for i,v in zip(tail,vals)]))


In [ ]:
import re
import pandas as pd

infile = "all_geom_comparison.txt"
outfile = "geom_comparison_metrics.csv"

with open(infile, 'r') as f:
    raw = f.read()

# Find all runs by scanning for 'Training:' then look for GEOMETRIC COMPARISON below them
run_blocks = re.split(r'\n=+\nTraining: +', raw)
run_records = []

for block in run_blocks[1:]:  # skip intro preamble
    lines = block.splitlines()
    # Get run_name from first line
    run_name = lines[0].strip()
    try:
        method, seed, model, dataset = re.match(r"([a-zA-Z0-9_]+)_s(\d+)_([a-zA-Z0-9]+)_([a-zA-Z0-9]+)", run_name).groups()
    except Exception:
        method = seed = model = dataset = None
    # Find the GEOMETRIC COMPARISON block for this run
    geom_match = re.search(r'GEOMETRIC COMPARISON:.*?\n=+\n(.*?)(\n=+\n|Training:|$)', block, re.DOTALL)
    if geom_match:
        section = geom_match.group(1)
        metrics = {}
        cur_section = "main"
        for line in section.splitlines():
            line = line.strip()
            if not line or "Metric" in line or line.startswith("-"):  # skip blank/headers
                continue
            if "Per-Class Intra-Sim:" in line:
                cur_section = "perclass"
                continue
            if cur_section == "main":
                parts = re.split(r'\s{2,}', line)
                if len(parts) == 4:
                    key, before, after, delta = parts
                    key = key.strip().replace(" ", "_").replace("-", "").replace("(", "").replace(")", "")
                    try:
                        metrics[key+"_before"] = float(before)
                        metrics[key+"_after"] = float(after)
                        metrics[key+"_delta"] = float(delta)
                    except ValueError:
                        continue
            elif cur_section == "perclass":
                pcs = re.split(r'\s{2,}', line)
                if len(pcs) == 4:
                    cls, before, after, delta = pcs
                    cls = cls.strip()
                    try:
                        metrics[f"PerClassIntraSim_{cls}_before"] = float(before)
                        metrics[f"PerClassIntraSim_{cls}_after"] = float(after)
                        metrics[f"PerClassIntraSim_{cls}_delta"] = float(delta)
                    except ValueError:
                        continue
        # Add a record only if there's anything in metrics (to filter incomplete runs)
        if metrics:
            record = dict(run_name=run_name, method=method, model=model, dataset=dataset, seed=seed)
            record.update(metrics)
            run_records.append(record)
    else:
        print(f"WARNING: Geom block not found for run {run_name}")

df = pd.DataFrame(run_records)
print(f"Parsed {len(df)} runs.")
print(df.columns)
df.to_csv(outfile, index=False)
print(f"Wrote to: {outfile}")
print(df.head())

In [ ]:
df.keys()

# Visualization CGC

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_delta_panel(ax, run_title, base_acc, cgc_acc, class_counts=None):
    # Align classes present in both
    classes = [c for c in base_acc.keys() if c in cgc_acc]
    if class_counts is not None:
        classes = sorted(classes, key=lambda c: class_counts.get(c, 0))
        xticklabels = [f"{c}\n(n={class_counts.get(c, 0)})" for c in classes]
    else:
        classes = sorted(classes, key=lambda c: base_acc[c])  # weakest first
        xticklabels = classes

    delta = np.array([cgc_acc[c] - base_acc[c] for c in classes], dtype=float)

    colors = ["#2C7FB8" if d >= 0 else "#D95F0E" for d in delta]
    ax.bar(np.arange(len(classes)), delta, color=colors, edgecolor="black", linewidth=0.4)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(run_title, fontsize=10)
    ax.set_xticks(np.arange(len(classes)))
    ax.set_xticklabels(xticklabels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel(r"$\Delta$ per-class test accuracy", fontsize=9)
    ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.6)

    return float(delta.min()), float(delta.max())

def make_figure1(runs, y_lim=None, out_path="figure1_per_class_delta.pdf"):
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
    axes = axes.flatten()

    dmins, dmaxs = [], []
    for ax, run in zip(axes, runs):
        dmin, dmax = plot_delta_panel(
            ax=ax,
            run_title=run["title"],
            base_acc=run["base_acc"],
            cgc_acc=run["cgc_acc"],
            class_counts=run.get("class_counts", None),
        )
        dmins.append(dmin); dmaxs.append(dmax)

    # Apply consistent y-limits across panels
    if y_lim is None:
        lo = min(dmins) - 0.02
        hi = max(dmaxs) + 0.02
        y_lim = (lo, hi)

    for ax in axes:
        ax.set_ylim(*y_lim)

    fig.suptitle("Per-class test accuracy change after CGC", fontsize=12)
    fig.savefig(out_path, dpi=300)
    print(f"Saved: {out_path}")

# Example usage: fill these dicts with your numbers
if __name__ == "__main__":
    runs = [
        {
            "title": "SARDet-100K | FUSARShip",
            "base_acc": {"Bulk": 0.7857, "Cargo": 0.7176, "Container": 0.3333, "Dredging": 0.5714,
                         "Fishing": 0.5063, "GeneralCargo": 0.2000, "Passenger": 0.1333, "Tanker": 0.6875, "Tug": 0.1667},
            "cgc_acc":  {"Bulk": 0.7500, "Cargo": 0.6820, "Container": 0.5710, "Dredging": 0.7140,
                         "Fishing": 0.6080, "GeneralCargo": 0.2000, "Passenger": 0.2000, "Tanker": 0.6250, "Tug": 0.3330},
        },
        {
            "title": "SAR-JEPA | FUSARShip",
            "base_acc": {"Bulk": 0.8571, "Cargo": 0.6000, "Container": 0.4286, "Dredging": 0.3810,
                         "Fishing": 0.6624, "GeneralCargo": 0.0667, "Passenger": 0.2000, "Tanker": 0.5208, "Tug": 0.5556},
            "cgc_acc":  {"Bulk": 0.8930, "Cargo": 0.6530, "Container": 0.4290, "Dredging": 0.4290,
                         "Fishing": 0.7340, "GeneralCargo": 0.2000, "Passenger": 0.2000, "Tanker": 0.5000, "Tug": 0.5000},
        },
        {
            "title": "SARDet-100K | OpenSARShip",
            "base_acc": {"Cargo": 0.7310, "Dredging": 0.0920, "Fishing": 0.3678, "Passenger": 0.1905, "Tanker": 0.5900, "Tug": 0.0256},
            "cgc_acc":  {"Cargo": 0.5850, "Dredging": 0.5170, "Fishing": 0.4480, "Passenger": 0.2140, "Tanker": 0.5100, "Tug": 0.2310},
        },
        {
            "title": "SAR-JEPA | OpenSARShip",
            "base_acc": {"Cargo": 0.5615, "Dredging": 0.5172, "Fishing": 0.4943, "Passenger": 0.2857, "Tanker": 0.4941, "Tug": 0.1538},
            "cgc_acc":  {"Cargo": 0.6150, "Dredging": 0.4480, "Fishing": 0.4830, "Passenger": 0.2860, "Tanker": 0.5590, "Tug": 0.2310},
        },
    ]

    make_figure1(runs, out_path="figure1_per_class_accuracy_delta.pdf")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

FUSAR_DELTAS = {
    "Bulk":        {"SARDet-100K": -0.0357, "SAR-JEPA":  0.0359},
    "Cargo":       {"SARDet-100K": -0.0356, "SAR-JEPA":  0.0530},
    "Container":   {"SARDet-100K":  0.2377, "SAR-JEPA":  0.0004},
    "Dredging":    {"SARDet-100K":  0.1426, "SAR-JEPA":  0.0480},
    "Fishing":     {"SARDet-100K":  0.1017, "SAR-JEPA":  0.0716},
    "GeneralCargo":{"SARDet-100K":  0.0000, "SAR-JEPA":  0.1333},
    "Passenger":   {"SARDet-100K":  0.0667, "SAR-JEPA":  0.0000},
    "Tanker":      {"SARDet-100K": -0.0625, "SAR-JEPA": -0.0208},
    "Tug":         {"SARDet-100K":  0.1663, "SAR-JEPA": -0.0556},
}

OPENSAR_DELTAS = {
    "Cargo":     {"SARDet-100K": -0.1460, "SAR-JEPA":  0.0535},
    "Dredging":  {"SARDet-100K":  0.4250, "SAR-JEPA": -0.0692},
    "Fishing":   {"SARDet-100K":  0.0802, "SAR-JEPA": -0.0113},
    "Passenger": {"SARDet-100K":  0.0235, "SAR-JEPA":  0.0003},
    "Tanker":    {"SARDet-100K": -0.0800, "SAR-JEPA":  0.0649},
    "Tug":       {"SARDet-100K":  0.2054, "SAR-JEPA":  0.0772},
}

def plot_grouped_delta(ax, title, deltas_dict, order=None):
    models = ["SARDet-100K", "SAR-JEPA"]
    if order is None:
        # Sort by average delta magnitude (largest changes first)
        order = sorted(
            deltas_dict.keys(),
            key=lambda c: abs(np.mean([deltas_dict[c][m] for m in models])),
            reverse=True,
        )

    x = np.arange(len(order))
    width = 0.38

    vals_a = np.array([deltas_dict[c][models[0]] for c in order], dtype=float)
    vals_b = np.array([deltas_dict[c][models[1]] for c in order], dtype=float)

    ax.bar(x - width/2, vals_a, width, label=models[0], color="#2C7FB8", edgecolor="black", linewidth=0.4)
    ax.bar(x + width/2, vals_b, width, label=models[1], color="#7FCDBB", edgecolor="black", linewidth=0.4)

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=45, ha="right")
    ax.set_title(title)
    ax.set_ylabel(r"$\Delta$ per-class test accuracy")
    ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.6)

    # Return min/max for global y-limits
    vmin = float(min(vals_a.min(), vals_b.min()))
    vmax = float(max(vals_a.max(), vals_b.max()))
    return vmin, vmax

def main():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

    vmins, vmaxs = [], []
    vmin, vmax = plot_grouped_delta(axes[0], "FUSARShip: per-class accuracy deltas", FUSAR_DELTAS)
    vmins.append(vmin); vmaxs.append(vmax)
    vmin, vmax = plot_grouped_delta(axes[1], "OpenSARShip: per-class accuracy deltas", OPENSAR_DELTAS)
    vmins.append(vmin); vmaxs.append(vmax)

    # Consistent y-limits across both datasets
    lo = min(vmins) - 0.03
    hi = max(vmaxs) + 0.03
    for ax in axes:
        ax.set_ylim(lo, hi)

    axes[0].legend(frameon=False)
    axes[1].legend(frameon=False)

    out = "per_class_delta_grouped_by_model.pdf"
    fig.savefig(out, dpi=300)
    print(f"Saved: {out}")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Test-set counts provided by you
counts_opensarship_test = {
    "Cargo": 1062,
    "Dredging": 29,
    "Fishing": 29,
    "Passenger": 14,
    "Tanker": 365,
    "Tug": 13,
}

counts_fusarship_test = {
    "Bulk": 28,
    "Cargo": 170,
    "Container": 7,
    "Dredging": 7,
    "Fishing": 79,
    "GeneralCargo": 5,
    "Passenger": 5,
    "Tanker": 16,
    "Tug": 6,
}

# Delta per-class test accuracy (CGC - baseline) you provided
deltas_fusarship = {
    "Bulk":        {"SARDet-100K": -0.0357, "SAR-JEPA":  0.0359},
    "Cargo":       {"SARDet-100K": -0.0356, "SAR-JEPA":  0.0530},
    "Container":   {"SARDet-100K":  0.2377, "SAR-JEPA":  0.0004},
    "Dredging":    {"SARDet-100K":  0.1426, "SAR-JEPA":  0.0480},
    "Fishing":     {"SARDet-100K":  0.1017, "SAR-JEPA":  0.0716},
    "GeneralCargo":{"SARDet-100K":  0.0000, "SAR-JEPA":  0.1333},
    "Passenger":   {"SARDet-100K":  0.0667, "SAR-JEPA":  0.0000},
    "Tanker":      {"SARDet-100K": -0.0625, "SAR-JEPA": -0.0208},
    "Tug":         {"SARDet-100K":  0.1663, "SAR-JEPA": -0.0556},
}

deltas_opensarship = {
    "Cargo":     {"SARDet-100K": -0.1460, "SAR-JEPA":  0.0535},
    "Dredging":  {"SARDet-100K":  0.4250, "SAR-JEPA": -0.0692},
    "Fishing":   {"SARDet-100K":  0.0802, "SAR-JEPA": -0.0113},
    "Passenger": {"SARDet-100K":  0.0235, "SAR-JEPA":  0.0003},
    "Tanker":    {"SARDet-100K": -0.0800, "SAR-JEPA":  0.0649},
    "Tug":       {"SARDet-100K":  0.2054, "SAR-JEPA":  0.0772},
}

def scatter(ax, title, counts, deltas_by_class, model, color, annotate=True):
    xs, ys, labels = [], [], []
    for cls, d in deltas_by_class.items():
        if cls not in counts:
            raise KeyError(f"Missing count for class '{cls}' in {title}")
        n = counts[cls]
        xs.append(np.log10(n))
        ys.append(d[model])
        labels.append(cls)

    xs = np.array(xs, dtype=float)
    ys = np.array(ys, dtype=float)

    ax.scatter(xs, ys, s=65, color=color, edgecolor="black", linewidth=0.4, label=model)

    if annotate:
        for x, y, cls in zip(xs, ys, labels):
            ax.text(x + 0.015, y, cls, fontsize=8, va="center")

def make_plot(out_path="logcount_test_vs_delta_accuracy.pdf", annotate=True):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    # Global y-limits for comparability
    all_ys = []
    for cls in deltas_fusarship:
        all_ys += [deltas_fusarship[cls]["SARDet-100K"], deltas_fusarship[cls]["SAR-JEPA"]]
    for cls in deltas_opensarship:
        all_ys += [deltas_opensarship[cls]["SARDet-100K"], deltas_opensarship[cls]["SAR-JEPA"]]
    ymin, ymax = min(all_ys), max(all_ys)
    pad = 0.05

    # FUSARShip
    ax = axes[0]
    ax.axhline(0, color="black", linewidth=0.8)
    scatter(ax, "FUSARShip", counts_fusarship_test, deltas_fusarship, "SARDet-100K", "#2C7FB8", annotate=annotate)
    scatter(ax, "FUSARShip", counts_fusarship_test, deltas_fusarship, "SAR-JEPA", "#7FCDBB", annotate=annotate)
    ax.set_title("FUSARShip: log10(test count) vs Δ per-class accuracy")
    ax.set_xlabel(r"$\log_{10}(n_{\mathrm{test}})$")
    ax.set_ylabel(r"$\Delta$ per-class test accuracy")
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.6)
    ax.legend(frameon=False)

    # OpenSARShip
    ax = axes[1]
    ax.axhline(0, color="black", linewidth=0.8)
    scatter(ax, "OpenSARShip", counts_opensarship_test, deltas_opensarship, "SARDet-100K", "#2C7FB8", annotate=annotate)
    scatter(ax, "OpenSARShip", counts_opensarship_test, deltas_opensarship, "SAR-JEPA", "#7FCDBB", annotate=annotate)
    ax.set_title("OpenSARShip: log10(test count) vs Δ per-class accuracy")
    ax.set_xlabel(r"$\log_{10}(n_{\mathrm{test}})$")
    ax.set_ylabel(r"$\Delta$ per-class test accuracy")
    ax.set_ylim(ymin - pad, ymax + pad)
    ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.6)
    ax.legend(frameon=False)

    fig.savefig(out_path, dpi=300)
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    make_plot(annotate=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Training counts + deltas (CGC - baseline) as provided
opensar = {
    "Cargo":     {"train": 8496, "SARDet-100K": -0.1460, "SAR-JEPA":  0.0535},
    "Dredging":  {"train":  232, "SARDet-100K":  0.4250, "SAR-JEPA": -0.0692},
    "Fishing":   {"train":  232, "SARDet-100K":  0.0802, "SAR-JEPA": -0.0113},
    "Passenger": {"train":  112, "SARDet-100K":  0.0235, "SAR-JEPA":  0.0003},
    "Tanker":    {"train": 2920, "SARDet-100K": -0.0800, "SAR-JEPA":  0.0649},
    "Tug":       {"train":  104, "SARDet-100K":  0.2054, "SAR-JEPA":  0.0772},
}

fusar = {
    "Bulk":        {"train":  224, "SARDet-100K": -0.0357, "SAR-JEPA":  0.0359},
    "Cargo":       {"train": 1360, "SARDet-100K": -0.0356, "SAR-JEPA":  0.0530},
    "Container":   {"train":   56, "SARDet-100K":  0.2377, "SAR-JEPA":  0.0004},
    "Dredging":    {"train":   56, "SARDet-100K":  0.1426, "SAR-JEPA":  0.0480},
    "Fishing":     {"train":  632, "SARDet-100K":  0.1017, "SAR-JEPA":  0.0716},
    "GeneralCargo":{"train":   40, "SARDet-100K":  0.0000, "SAR-JEPA":  0.1333},
    "Passenger":   {"train":   40, "SARDet-100K":  0.0667, "SAR-JEPA":  0.0000},
    "Tanker":      {"train":  128, "SARDet-100K": -0.0625, "SAR-JEPA": -0.0208},
    "Tug":         {"train":   48, "SARDet-100K":  0.1663, "SAR-JEPA": -0.0556},
}

def scatter_panel(ax, title, data, annotate=True):
    models = [("SARDet-100K", "#2C7FB8"), ("SAR-JEPA", "#7FCDBB")]

    ax.axhline(0.0, color="black", linewidth=0.8)

    for model, color in models:
        xs, ys, labels = [], [], []
        for cls, row in data.items():
            xs.append(np.log10(row["train"]))
            ys.append(row[model])
            labels.append(cls)

        xs = np.array(xs, dtype=float)
        ys = np.array(ys, dtype=float)

        ax.scatter(xs, ys, s=70, color=color, edgecolor="black", linewidth=0.4, label=model)

        if annotate:
            for x, y, cls in zip(xs, ys, labels):
                ax.text(x + 0.015, y, cls, fontsize=8, va="center")

    ax.set_title(title)
    ax.set_xlabel(r"$\log_{10}(n_{\mathrm{train}})$")
    ax.set_ylabel(r"$\Delta$ per-class test accuracy")
    ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.6)
    ax.legend(frameon=False)

def main(out_path="figure_log_traincount_vs_delta_accuracy.pdf", annotate=True):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

    scatter_panel(axes[0], "FUSARShip: frequency vs per-class gain", fusar, annotate=annotate)
    scatter_panel(axes[1], "OpenSARShip: frequency vs per-class gain", opensar, annotate=annotate)

    # Consistent y-limits across panels
    all_deltas = []
    for d in (fusar, opensar):
        for cls in d:
            all_deltas.append(d[cls]["SARDet-100K"])
            all_deltas.append(d[cls]["SAR-JEPA"])
    lo, hi = min(all_deltas), max(all_deltas)
    pad = 0.05
    for ax in axes:
        ax.set_ylim(lo - pad, hi + pad)

    fig.savefig(out_path, dpi=300)
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    main(annotate=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data: train counts and deltas (CGC - baseline)
opensar = {
    "Cargo":     {"train": 8496, "SARDet-100K": -0.1460, "SAR-JEPA":  0.0535},
    "Dredging":  {"train":  232, "SARDet-100K":  0.4250, "SAR-JEPA": -0.0692},
    "Fishing":   {"train":  232, "SARDet-100K":  0.0802, "SAR-JEPA": -0.0113},
    "Passenger": {"train":  112, "SARDet-100K":  0.0235, "SAR-JEPA":  0.0003},
    "Tanker":    {"train": 2920, "SARDet-100K": -0.0800, "SAR-JEPA":  0.0649},
    "Tug":       {"train":  104, "SARDet-100K":  0.2054, "SAR-JEPA":  0.0772},
}

fusar = {
    "Bulk":        {"train":  224, "SARDet-100K": -0.0357, "SAR-JEPA":  0.0359},
    "Cargo":       {"train": 1360, "SARDet-100K": -0.0356, "SAR-JEPA":  0.0530},
    "Container":   {"train":   56, "SARDet-100K":  0.2377, "SAR-JEPA":  0.0004},
    "Dredging":    {"train":   56, "SARDet-100K":  0.1426, "SAR-JEPA":  0.0480},
    "Fishing":     {"train":  632, "SARDet-100K":  0.1017, "SAR-JEPA":  0.0716},
    "GeneralCargo":{"train":   40, "SARDet-100K":  0.0000, "SAR-JEPA":  0.1333},
    "Passenger":   {"train":   40, "SARDet-100K":  0.0667, "SAR-JEPA":  0.0000},
    "Tanker":      {"train":  128, "SARDet-100K": -0.0625, "SAR-JEPA": -0.0208},
    "Tug":         {"train":   48, "SARDet-100K":  0.1663, "SAR-JEPA": -0.0556},
}

plt.rcParams.update({"font.size": 24})

def grouped_delta_bars(ax, title, data):
    # Sort by train count (ascending): minority first
    classes = sorted(data.keys(), key=lambda c: data[c]["train"])
    counts = [data[c]["train"] for c in classes]

    x = np.arange(len(classes))
    width = 0.38

    sardet = np.array([data[c]["SARDet-100K"] for c in classes], dtype=float)
    sarjepa = np.array([data[c]["SAR-JEPA"] for c in classes], dtype=float)

    # Colors chosen for print and colorblind-friendliness
    ax.bar(x - width/2, sardet, width, label="SARDET-100K", color="#2C7FB8", edgecolor="black", linewidth=0.4)
    ax.bar(x + width/2, sarjepa, width, label="SAR-JEPA", color="#7FCDBB", edgecolor="black", linewidth=0.4)

    ax.axhline(0.0, color="black", linewidth=0.9)
    ax.set_title(title)
    ax.set_ylabel(r"$\Delta$")
    ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.6)

    # Tick labels with counts on a second line
    xticks = [f"{c}\n(n={n})" for c, n in zip(classes, counts)]
    ax.set_xticks(x)
    ax.set_xticklabels(xticks, rotation=60, ha="right", fontsize=16)

    return float(min(sardet.min(), sarjepa.min())), float(max(sardet.max(), sarjepa.max()))

def main(out_path="per_class_performance.pdf"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

    vmin1, vmax1 = grouped_delta_bars(axes[0], "FUSARShip", fusar)
    vmin2, vmax2 = grouped_delta_bars(axes[1], "OpenSARShip", opensar)

    # Consistent y-limits across panels
    lo = min(vmin1, vmin2) - 0.03
    hi = max(vmax1, vmax2) + 0.03
    for ax in axes:
        ax.set_ylim(lo, hi)

    # One shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=2, frameon=True, prop={'family': 'monospace', 'size': 16, 'weight': 'bold', 'style': 'italic'})

    fig.savefig(out_path, dpi=300)
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Data organized by (dataset, model)
data = {
    ("FUSARShip", "SARDet-100K"): {
        "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9660},
        "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.9575},
        "Container":   {"train": 56, "before": 0.2140, "after": 0.9515},
        "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9681},
        "Fishing":     {"train": 632, "before": 0.0872, "after": 0.9539},
        "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9634},
        "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9765},
        "Tanker":      {"train": 128, "before": 0.6812, "after": 0.9731},
        "Tug":         {"train": 48, "before": 0.5122, "after": 0.9731},
    },
    ("FUSARShip", "SAR-JEPA"): {
        "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9084},
        "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.7959},
        "Container":   {"train": 56, "before": 0.2140, "after": 0.8931},
        "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9076},
        "Fishing":     {"train": 632, "before": 0.0872, "after": 0.8386},
        "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9128},
        "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9229},
        "Tanker":      {"train": 128, "before": 0.6812, "after": 0.8908},
        "Tug":         {"train": 48, "before": 0.5122, "after": 0.9041},
    },
    ("OpenSARShip", "SARDet-100K"): {
        "Cargo":     {"train": 8496, "before": 0.0116, "after": 0.9670},
        "Dredging":  {"train": 232, "before": 0.0424, "after": 0.9731},
        "Fishing":   {"train": 232, "before": 0.1905, "after": 0.9490},
        "Passenger": {"train": 112, "before": 0.0623, "after": 0.9654},
        "Tanker":    {"train": 2920, "before": 0.0238, "after": 0.9671},
        "Tug":       {"train": 104, "before": 0.1104, "after": 0.9586},
    },
    ("OpenSARShip", "SAR-JEPA"): {
        "Cargo":     {"train": 8496, "before": 0.0042, "after": 0.9793},
        "Dredging":  {"train": 232, "before": 0.0519, "after": 0.9820},
        "Fishing":   {"train": 232, "before": 0.3849, "after": 0.9712},
        "Passenger": {"train": 112, "before": 0.1952, "after": 0.9791},
        "Tanker":    {"train": 2920, "before": 0.0359, "after": 0.9800},
        "Tug":       {"train": 104, "before": 0.1211, "after": 0.9801},
    },
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("FUSARShip - SARDet-100K", "FUSARShip - SAR-JEPA",
                    "OpenSARShip - SARDet-100K", "OpenSARShip - SAR-JEPA"),
    specs=[[{}, {}], [{}, {}]]
)

datasets = [("FUSARShip", "SARDet-100K"), ("FUSARShip", "SAR-JEPA"),
            ("OpenSARShip", "SARDet-100K"), ("OpenSARShip", "SAR-JEPA")]
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (dataset, model), (row, col) in zip(datasets, positions):
    d = data[(dataset, model)]

    # Extract x (log train count), y_before, y_after
    x_log = np.array([np.log10(d[c]["train"]) for c in d.keys()], dtype=float)
    y_before = np.array([d[c]["before"] for c in d.keys()], dtype=float)
    y_after = np.array([d[c]["after"] for c in d.keys()], dtype=float)
    classes = list(d.keys())

    # Scatter: Before CGC
    fig.add_trace(
        go.Scatter(
            x=x_log, y=y_before,
            mode="markers",
            name="Before CGC",
            marker=dict(size=10, color="#E74C3C", symbol="circle", line=dict(width=1, color="darkred")),
            text=classes,
            hovertemplate="<b>%{text}</b><br>log10(n_train)=%{x:.2f}<br>Intra-Sim=%.3f<extra></extra>" % tuple([y_before]),
            legendgroup="before",
            showlegend=(row == 1 and col == 1),
        ),
        row=row, col=col
    )

    # Scatter: After CGC
    fig.add_trace(
        go.Scatter(
            x=x_log, y=y_after,
            mode="markers",
            name="After CGC",
            marker=dict(size=10, color="#27AE60", symbol="square", line=dict(width=1, color="darkgreen")),
            text=classes,
            hovertemplate="<b>%{text}</b><br>log10(n_train)=%{x:.2f}<br>Intra-Sim=%.3f<extra></extra>" % tuple([y_after]),
            legendgroup="after",
            showlegend=(row == 1 and col == 1),
        ),
        row=row, col=col
    )

    # Trend lines (linear fit in log space)
    z_before = np.polyfit(x_log, y_before, 1)
    z_after = np.polyfit(x_log, y_after, 1)
    x_trend = np.linspace(x_log.min() - 0.1, x_log.max() + 0.1, 100)
    y_trend_before = np.polyval(z_before, x_trend)
    y_trend_after = np.polyval(z_after, x_trend)

    fig.add_trace(
        go.Scatter(
            x=x_trend, y=y_trend_before,
            mode="lines",
            name="Trend Before",
            line=dict(color="#E74C3C", width=2, dash="dash"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row, col=col
    )

    fig.add_trace(
        go.Scatter(
            x=x_trend, y=y_trend_after,
            mode="lines",
            name="Trend After",
            line=dict(color="#27AE60", width=2, dash="dash"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row, col=col
    )

    # Correlation values (for reference)
    corr_before = np.corrcoef(x_log, y_before)[0, 1]
    corr_after = np.corrcoef(x_log, y_after)[0, 1]

# Update axes
fig.update_xaxes(title_text="log₁₀(training count)", row=2, col=1)
fig.update_xaxes(title_text="log₁₀(training count)", row=2, col=2)
fig.update_yaxes(title_text="Per-class intra-similarity", row=1, col=1)
fig.update_yaxes(title_text="Per-class intra-similarity", row=2, col=1)

for row in [1, 2]:
    for col in [1, 2]:
        fig.update_xaxes(range=[0.8, 4.0], row=row, col=col)
        fig.update_yaxes(range=[0, 1.05], row=row, col=col)
        fig.add_hline(y=0.5, line_dash="dot", line_color="gray", opacity=0.4, row=row, col=col)

fig.update_layout(
    height=900, width=1400,
    title_text="Per-class intra-class similarity vs training frequency (before and after CGC)",
    hovermode="closest",
    font=dict(size=12),
    showlegend=True,
    legend=dict(x=0.35, y=0.02, orientation="h", bgcolor="rgba(255,255,255,0.8)"),
)

fig.write_html("intra_similarity_vs_traincount.html")
fig.write_image("intra_similarity_vs_traincount.pdf", width=1400, height=900)
print("Saved: intra_similarity_vs_traincount.html and .pdf")

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

# Data: exact values, no changes
data = {
    ("FUSARShip", "SARDet-100K"): {
        "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9660},
        "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.9575},
        "Container":   {"train": 56, "before": 0.2140, "after": 0.9515},
        "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9681},
        "Fishing":     {"train": 632, "before": 0.0872, "after": 0.9539},
        "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9634},
        "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9765},
        "Tanker":      {"train": 128, "before": 0.6812, "after": 0.9731},
        "Tug":         {"train": 48, "before": 0.5122, "after": 0.9731},
    },
    ("FUSARShip", "SAR-JEPA"): {
        "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9084},
        "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.7959},
        "Container":   {"train": 56, "before": 0.2140, "after": 0.8931},
        "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9076},
        "Fishing":     {"train": 632, "before": 0.0872, "after": 0.8386},
        "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9128},
        "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9229},
        "Tanker":      {"train": 128, "before": 0.6812, "after": 0.8908},
        "Tug":         {"train": 48, "before": 0.5122, "after": 0.9041},
    },
    ("OpenSARShip", "SARDet-100K"): {
        "Cargo":     {"train": 8496, "before": 0.0116, "after": 0.9670},
        "Dredging":  {"train": 232, "before": 0.0424, "after": 0.9731},
        "Fishing":   {"train": 232, "before": 0.1905, "after": 0.9490},
        "Passenger": {"train": 112, "before": 0.0623, "after": 0.9654},
        "Tanker":    {"train": 2920, "before": 0.0238, "after": 0.9671},
        "Tug":       {"train": 104, "before": 0.1104, "after": 0.9586},
    },
    ("OpenSARShip", "SAR-JEPA"): {
        "Cargo":     {"train": 8496, "before": 0.0042, "after": 0.9793},
        "Dredging":  {"train": 232, "before": 0.0519, "after": 0.9820},
        "Fishing":   {"train": 232, "before": 0.3849, "after": 0.9712},
        "Passenger": {"train": 112, "before": 0.1952, "after": 0.9791},
        "Tanker":    {"train": 2920, "before": 0.0359, "after": 0.9800},
        "Tug":       {"train": 104, "before": 0.1211, "after": 0.9801},
    },
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("FUSARShip, SARDet-100K", "FUSARShip, SAR-JEPA",
                    "OpenSARShip, SARDet-100K", "OpenSARShip, SAR-JEPA"),
    specs=[[{}, {}], [{}, {}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
)

datasets = [("FUSARShip", "SARDet-100K"), ("FUSARShip", "SAR-JEPA"),
            ("OpenSARShip", "SARDet-100K"), ("OpenSARShip", "SAR-JEPA")]
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (dataset, model), (row, col) in zip(datasets, positions):
    d = data[(dataset, model)]

    x_log = np.array([np.log10(d[c]["train"]) for c in d.keys()], dtype=float)
    y_before = np.array([d[c]["before"] for c in d.keys()], dtype=float)
    y_after = np.array([d[c]["after"] for c in d.keys()], dtype=float)
    classes = list(d.keys())

    # Compute correlations (exact, unmodified)
    corr_before, pval_before = pearsonr(x_log, y_before)
    corr_after, pval_after = pearsonr(x_log, y_after)

    # Before CGC: red circles
    fig.add_trace(
        go.Scatter(
            x=x_log, y=y_before,
            mode="markers",
            name="Before CGC",
            marker=dict(size=11, color="#D62728", symbol="circle", line=dict(width=1.5, color="darkred")),
            text=[f"<b>{c}</b><br>train n={d[c]['train']}<br>intra-sim={d[c]['before']:.4f}" for c in classes],
            hovertemplate="%{text}<extra></extra>",
            legendgroup="before",
            showlegend=(row == 1 and col == 1),
        ),
        row=row, col=col
    )

    # After CGC: blue squares
    fig.add_trace(
        go.Scatter(
            x=x_log, y=y_after,
            mode="markers",
            name="After CGC",
            marker=dict(size=11, color="#1F77B4", symbol="square", line=dict(width=1.5, color="darkblue")),
            text=[f"<b>{c}</b><br>train n={d[c]['train']}<br>intra-sim={d[c]['after']:.4f}" for c in classes],
            hovertemplate="%{text}<extra></extra>",
            legendgroup="after",
            showlegend=(row == 1 and col == 1),
        ),
        row=row, col=col
    )

    # Add correlation text annotations (exact computed values)
    corr_text = f"r_before={corr_before:.3f}<br>r_after={corr_after:.3f}"
    fig.add_annotation(
        text=corr_text,
        xref=f"x{'' if row == 1 and col == 1 else f'{row}{col}'}",
        yref=f"y{'' if row == 1 and col == 1 else f'{row}{col}'}",
        x=0.95, y=0.05,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=0.5,
        borderpad=4,
        row=row, col=col
    )

# Update axes (consistent ranges for all subplots)
for row in [1, 2]:
    for col in [1, 2]:
        fig.update_xaxes(title_text="log₁₀(training count)", row=row, col=col, range=[0.5, 4.0], showgrid=True, gridwidth=0.5, gridcolor="lightgray")
        fig.update_yaxes(title_text="intra-class similarity", row=row, col=col, range=[-0.05, 1.1], showgrid=True, gridwidth=0.5, gridcolor="lightgray")

fig.update_layout(
    height=800, width=1200,
    title_text="Per-class intra-class similarity vs training frequency (before and after CGC)",
    font=dict(size=13, family="Arial"),
    hovermode="closest",
    showlegend=True,
    legend=dict(x=0.5, y=-0.08, orientation="h", bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=0.5, xanchor="center"),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.write_html("intra_similarity_vs_traincount_clean.html")
fig.write_image("intra_similarity_vs_traincount_clean.pdf", width=1200, height=800)
print("Saved: intra_similarity_vs_traincount_clean.html and .pdf")

In [ ]:
!pip install -U kaleido==0.2.1
!pip install plotly==5.24.1
# !kaleido_get_chrome

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

# Data: exact values, no changes
data = {
    ("FUSARShip", "SARDet-100K"): {
        "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9660},
        "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.9575},
        "Container":   {"train": 56, "before": 0.2140, "after": 0.9515},
        "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9681},
        "Fishing":     {"train": 632, "before": 0.0872, "after": 0.9539},
        "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9634},
        "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9765},
        "Tanker":      {"train": 128, "before": 0.6812, "after": 0.9731},
        "Tug":         {"train": 48, "before": 0.5122, "after": 0.9731},
    },
    ("FUSARShip", "SAR-JEPA"): {
        "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9084},
        "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.7959},
        "Container":   {"train": 56, "before": 0.2140, "after": 0.8931},
        "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9076},
        "Fishing":     {"train": 632, "before": 0.0872, "after": 0.8386},
        "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9128},
        "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9229},
        "Tanker":      {"train": 128, "before": 0.6812, "after": 0.8908},
        "Tug":         {"train": 48, "before": 0.5122, "after": 0.9041},
    },
    ("OpenSARShip", "SARDet-100K"): {
        "Cargo":     {"train": 8496, "before": 0.0116, "after": 0.9670},
        "Dredging":  {"train": 232, "before": 0.0424, "after": 0.9731},
        "Fishing":   {"train": 232, "before": 0.1905, "after": 0.9490},
        "Passenger": {"train": 112, "before": 0.0623, "after": 0.9654},
        "Tanker":    {"train": 2920, "before": 0.0238, "after": 0.9671},
        "Tug":       {"train": 104, "before": 0.1104, "after": 0.9586},
    },
    ("OpenSARShip", "SAR-JEPA"): {
        "Cargo":     {"train": 8496, "before": 0.0042, "after": 0.9793},
        "Dredging":  {"train": 232, "before": 0.0519, "after": 0.9820},
        "Fishing":   {"train": 232, "before": 0.3849, "after": 0.9712},
        "Passenger": {"train": 112, "before": 0.1952, "after": 0.9791},
        "Tanker":    {"train": 2920, "before": 0.0359, "after": 0.9800},
        "Tug":       {"train": 104, "before": 0.1211, "after": 0.9801},
    },
}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("FUSARShip, SARDet-100K", "FUSARShip, SAR-JEPA",
                    "OpenSARShip, SARDet-100K", "OpenSARShip, SAR-JEPA"),
    specs=[[{}, {}], [{}, {}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.12,
)

datasets = [("FUSARShip", "SARDet-100K"), ("FUSARShip", "SAR-JEPA"),
            ("OpenSARShip", "SARDet-100K"), ("OpenSARShip", "SAR-JEPA")]
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (dataset, model), (row, col) in zip(datasets, positions):
    d = data[(dataset, model)]

    x_log = np.array([np.log10(d[c]["train"]) for c in d.keys()], dtype=float)
    y_before = np.array([d[c]["before"] for c in d.keys()], dtype=float)
    y_after = np.array([d[c]["after"] for c in d.keys()], dtype=float)
    classes = list(d.keys())

    # Compute correlations (exact, unmodified)
    corr_before, pval_before = pearsonr(x_log, y_before)
    corr_after, pval_after = pearsonr(x_log, y_after)

    # Before CGC: red circles
    fig.add_trace(
        go.Scatter(
            x=x_log, y=y_before,
            mode="markers",
            name="Before CGC",
            marker=dict(size=11, color="#D62728", symbol="circle", line=dict(width=1.5, color="darkred")),
            text=[f"<b>{c}</b><br>train n={d[c]['train']}<br>intra-sim={d[c]['before']:.4f}" for c in classes],
            hovertemplate="%{text}<extra></extra>",
            legendgroup="before",
            showlegend=(row == 1 and col == 1),
        ),
        row=row, col=col
    )

    # After CGC: blue squares
    fig.add_trace(
        go.Scatter(
            x=x_log, y=y_after,
            mode="markers",
            name="After CGC",
            marker=dict(size=11, color="#1F77B4", symbol="square", line=dict(width=1.5, color="darkblue")),
            text=[f"<b>{c}</b><br>train n={d[c]['train']}<br>intra-sim={d[c]['after']:.4f}" for c in classes],
            hovertemplate="%{text}<extra></extra>",
            legendgroup="after",
            showlegend=(row == 1 and col == 1),
        ),
        row=row, col=col
    )

    # Add correlation text annotations (exact computed values)
    corr_text = f"r_before={corr_before:.3f}<br>r_after={corr_after:.3f}"
    fig.add_annotation(
        text=corr_text,
        xref=f"x{'' if row == 1 and col == 1 else f'{row}{col}'}",
        yref=f"y{'' if row == 1 and col == 1 else f'{row}{col}'}",
        x=0.95, y=0.05,
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=0.5,
        borderpad=4,
        row=row, col=col
    )

# Update axes (consistent ranges for all subplots)
for row in [1, 2]:
    for col in [1, 2]:
        fig.update_xaxes(title_text="log₁₀(training count)", row=row, col=col, range=[0.5, 4.0], showgrid=True, gridwidth=0.5, gridcolor="lightgray")
        fig.update_yaxes(title_text="intra-class similarity", row=row, col=col, range=[-0.05, 1.1], showgrid=True, gridwidth=0.5, gridcolor="lightgray")

fig.update_layout(
    height=800, width=1200,
    title_text="Per-class intra-class similarity vs training frequency (before and after CGC)",
    font=dict(size=13, family="Arial"),
    hovermode="closest",
    showlegend=True,
    legend=dict(x=0.5, y=-0.08, orientation="h", bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=0.5, xanchor="center"),
    plot_bgcolor="white",
    paper_bgcolor="white",
)

fig.write_html("intra_similarity_vs_traincount_clean.html")
fig.write_image("intra_similarity_vs_traincount_clean.pdf", width=1200, height=800)
print("Saved: intra_similarity_vs_traincount_clean.html and .pdf")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data: train counts and delta intra-similarity (After - Before)
fusar = {
    "Bulk":        {"train": 224, "SARDet-100K": 0.5661, "SAR-JEPA": 0.5085},
    "Cargo":       {"train": 1360, "SARDet-100K": 0.9355, "SAR-JEPA": 0.7740},
    "Container":   {"train": 56, "SARDet-100K": 0.7374, "SAR-JEPA": 0.6791},
    "Dredging":    {"train": 56, "SARDet-100K": 0.3803, "SAR-JEPA": 0.3198},
    "Fishing":     {"train": 632, "SARDet-100K": 0.8667, "SAR-JEPA": 0.7514},
    "GeneralCargo":{"train": 40, "SARDet-100K": 0.7652, "SAR-JEPA": 0.7146},
    "Passenger":   {"train": 40, "SARDet-100K": 0.3099, "SAR-JEPA": 0.2563},
    "Tanker":      {"train": 128, "SARDet-100K": 0.2919, "SAR-JEPA": 0.2096},
    "Tug":         {"train": 48, "SARDet-100K": 0.4609, "SAR-JEPA": 0.3919},
}

opensar = {
    "Cargo":     {"train": 8496, "SARDet-100K": 0.9555, "SAR-JEPA": 0.9751},
    "Dredging":  {"train": 232, "SARDet-100K": 0.9307, "SAR-JEPA": 0.9301},
    "Fishing":   {"train": 232, "SARDet-100K": 0.7586, "SAR-JEPA": 0.5863},
    "Passenger": {"train": 112, "SARDet-100K": 0.9031, "SAR-JEPA": 0.7840},
    "Tanker":    {"train": 2920, "SARDet-100K": 0.9432, "SAR-JEPA": 0.9441},
    "Tug":       {"train": 104, "SARDet-100K": 0.8482, "SAR-JEPA": 0.8589},
}

def scatter_delta(ax, title, data):
    models = ["SARDet-100K", "SAR-JEPA"]
    colors = {"SARDet-100K": "#2C7FB8", "SAR-JEPA": "#7FCDBB"}
    markers = {"SARDet-100K": "o", "SAR-JEPA": "s"}

    for model in models:
        x_log = np.array([np.log10(data[c]["train"]) for c in data.keys()], dtype=float)
        y_delta = np.array([data[c][model] for c in data.keys()], dtype=float)
        classes = list(data.keys())

        ax.scatter(x_log, y_delta, s=100, color=colors[model], marker=markers[model],
                   label=model, edgecolor="black", linewidth=1, alpha=0.8, zorder=3)

        # Label each point with class name
        for x, y, cls in zip(x_log, y_delta, classes):
            ax.text(x + 0.08, y, cls, fontsize=9, va="center")

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(r"$\log_{10}(n_{\mathrm{train}})$", fontsize=12)
    ax.set_ylabel(r"$\Delta$ intra-class similarity", fontsize=12)
    ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.5)
    ax.legend(frameon=True, fontsize=11)

def main(out_path="intra_similarity_delta_vs_traincount.pdf"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)

    scatter_delta(axes[0], "FUSARShip: intra-similarity gain vs frequency", fusar)
    scatter_delta(axes[1], "OpenSARShip: intra-similarity gain vs frequency", opensar)

    # Consistent y-limits
    all_deltas = []
    for d in (fusar, opensar):
        for cls in d:
            all_deltas.extend([d[cls]["SARDet-100K"], d[cls]["SAR-JEPA"]])
    lo, hi = min(all_deltas), max(all_deltas)
    for ax in axes:
        ax.set_ylim(lo - 0.05, hi + 0.05)

    fig.suptitle("Per-class intra-class similarity improvement after CGC", fontsize=14, fontweight="bold", y=1.02)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data: train counts and intra-similarity before and after
fusar = {
    "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9660},
    "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.9575},
    "Container":   {"train": 56, "before": 0.2140, "after": 0.9515},
    "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9681},
    "Fishing":     {"train": 632, "before": 0.0872, "after": 0.9539},
    "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9634},
    "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9765},
    "Tanker":      {"train": 128, "before": 0.6812, "after": 0.9731},
    "Tug":         {"train": 48, "before": 0.5122, "after": 0.9731},
}

opensar = {
    "Cargo":     {"train": 8496, "before": 0.0116, "after": 0.9670},
    "Dredging":  {"train": 232, "before": 0.0424, "after": 0.9731},
    "Fishing":   {"train": 232, "before": 0.1905, "after": 0.9490},
    "Passenger": {"train": 112, "before": 0.0623, "after": 0.9654},
    "Tanker":    {"train": 2920, "before": 0.0238, "after": 0.9671},
    "Tug":       {"train": 104, "before": 0.1104, "after": 0.9586},
}

# SAR-JEPA data
fusar_jepa = {
    "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9084},
    "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.7959},
    "Container":   {"train": 56, "before": 0.2140, "after": 0.8931},
    "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9076},
    "Fishing":     {"train": 632, "before": 0.0872, "after": 0.8386},
    "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9128},
    "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9229},
    "Tanker":      {"train": 128, "before": 0.6812, "after": 0.8908},
    "Tug":         {"train": 48, "before": 0.5122, "after": 0.9041},
}

opensar_jepa = {
    "Cargo":     {"train": 8496, "before": 0.0042, "after": 0.9793},
    "Dredging":  {"train": 232, "before": 0.0519, "after": 0.9820},
    "Fishing":   {"train": 232, "before": 0.3849, "after": 0.9712},
    "Passenger": {"train": 112, "before": 0.1952, "after": 0.9791},
    "Tanker":    {"train": 2920, "before": 0.0359, "after": 0.9800},
    "Tug":       {"train": 104, "before": 0.1211, "after": 0.9801},
}

def plot_before_after(ax_before, ax_after, title_prefix, data_before_after, model_name):
    """Plot before and after on separate axes with arrows connecting them."""
    classes = list(data_before_after.keys())
    x_log = np.array([np.log10(data_before_after[c]["train"]) for c in classes], dtype=float)
    y_before = np.array([data_before_after[c]["before"] for c in classes], dtype=float)
    y_after = np.array([data_before_after[c]["after"] for c in classes], dtype=float)

    # Before scatter
    ax_before.scatter(x_log, y_before, s=120, color="#E74C3C", edgecolor="darkred",
                      linewidth=1.5, alpha=0.7, zorder=3)
    for x, y, cls in zip(x_log, y_before, classes):
        ax_before.text(x + 0.08, y, cls, fontsize=9, va="center", fontweight="bold")

    ax_before.set_title(f"{title_prefix}\nBefore CGC", fontsize=12, fontweight="bold")
    ax_before.set_xlabel(r"$\log_{10}(n_{\mathrm{train}})$", fontsize=11)
    ax_before.set_ylabel("Intra-class similarity", fontsize=11)
    ax_before.set_ylim(-0.05, 1.1)
    ax_before.grid(True, linestyle=":", linewidth=0.5, alpha=0.5)

    # After scatter
    ax_after.scatter(x_log, y_after, s=120, color="#27AE60", edgecolor="darkgreen",
                     linewidth=1.5, alpha=0.7, zorder=3)
    for x, y, cls in zip(x_log, y_after, classes):
        ax_after.text(x + 0.08, y, cls, fontsize=9, va="center", fontweight="bold")

    # Draw arrows from before to after
    for x, yb, ya in zip(x_log, y_before, y_after):
        ax_after.annotate("", xy=(x, ya), xytext=(x, yb),
                          xycoords=("data", "data"),
                          textcoords=("data", "data"),
                          arrowprops=dict(arrowstyle="->", lw=1.5, color="gray",
                                          connectionstyle="arc3,rad=0.3", alpha=0.6))

    ax_after.set_title(f"{title_prefix}\nAfter CGC", fontsize=12, fontweight="bold")
    ax_after.set_xlabel(r"$\log_{10}(n_{\mathrm{train}})$", fontsize=11)
    ax_after.set_ylabel("Intra-class similarity", fontsize=11)
    ax_after.set_ylim(-0.05, 1.1)
    ax_after.grid(True, linestyle=":", linewidth=0.5, alpha=0.5)

def main(out_path="intra_similarity_before_after_comparison.pdf"):
    fig = plt.figure(figsize=(16, 10), constrained_layout=True)
    gs = fig.add_gridspec(2, 4, hspace=0.3, wspace=0.3)

    # FUSARShip - SARDet-100K
    ax_b1 = fig.add_subplot(gs[0, 0])
    ax_a1 = fig.add_subplot(gs[0, 1])
    plot_before_after(ax_b1, ax_a1, "FUSARShip - SARDet-100K", fusar, "SARDet-100K")

    # FUSARShip - SAR-JEPA
    ax_b2 = fig.add_subplot(gs[0, 2])
    ax_a2 = fig.add_subplot(gs[0, 3])
    plot_before_after(ax_b2, ax_a2, "FUSARShip - SAR-JEPA", fusar_jepa, "SAR-JEPA")

    # OpenSARShip - SARDet-100K
    ax_b3 = fig.add_subplot(gs[1, 0])
    ax_a3 = fig.add_subplot(gs[1, 1])
    plot_before_after(ax_b3, ax_a3, "OpenSARShip - SARDet-100K", opensar, "SARDet-100K")

    # OpenSARShip - SAR-JEPA
    ax_b4 = fig.add_subplot(gs[1, 2])
    ax_a4 = fig.add_subplot(gs[1, 3])
    plot_before_after(ax_b4, ax_a4, "OpenSARShip - SAR-JEPA", opensar_jepa, "SAR-JEPA")

    fig.suptitle("Per-class intra-class similarity: before vs after CGC\n(arrows show per-class improvement)",
                 fontsize=14, fontweight="bold", y=0.995)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data: train counts and intra-similarity before and after
fusar_sardet = {
    "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9660},
    "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.9575},
    "Container":   {"train": 56, "before": 0.2140, "after": 0.9515},
    "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9681},
    "Fishing":     {"train": 632, "before": 0.0872, "after": 0.9539},
    "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9634},
    "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9765},
    "Tanker":      {"train": 128, "before": 0.6812, "after": 0.9731},
    "Tug":         {"train": 48, "before": 0.5122, "after": 0.9731},
}

fusar_jepa = {
    "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9084},
    "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.7959},
    "Container":   {"train": 56, "before": 0.2140, "after": 0.8931},
    "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9076},
    "Fishing":     {"train": 632, "before": 0.0872, "after": 0.8386},
    "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9128},
    "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9229},
    "Tanker":      {"train": 128, "before": 0.6812, "after": 0.8908},
    "Tug":         {"train": 48, "before": 0.5122, "after": 0.9041},
}

opensar_sardet = {
    "Cargo":     {"train": 8496, "before": 0.0116, "after": 0.9670},
    "Dredging":  {"train": 232, "before": 0.0424, "after": 0.9731},
    "Fishing":   {"train": 232, "before": 0.1905, "after": 0.9490},
    "Passenger": {"train": 112, "before": 0.0623, "after": 0.9654},
    "Tanker":    {"train": 2920, "before": 0.0238, "after": 0.9671},
    "Tug":       {"train": 104, "before": 0.1104, "after": 0.9586},
}

opensar_jepa = {
    "Cargo":     {"train": 8496, "before": 0.0042, "after": 0.9793},
    "Dredging":  {"train": 232, "before": 0.0519, "after": 0.9820},
    "Fishing":   {"train": 232, "before": 0.3849, "after": 0.9712},
    "Passenger": {"train": 112, "before": 0.1952, "after": 0.9791},
    "Tanker":    {"train": 2920, "before": 0.0359, "after": 0.9800},
    "Tug":       {"train": 104, "before": 0.1211, "after": 0.9801},
}

def plot_before_after_overlay(ax, title, data):
    """Plot before and after on same panel with arrows."""
    classes = list(data.keys())
    x_log = np.array([np.log10(data[c]["train"]) for c in classes], dtype=float)
    y_before = np.array([data[c]["before"] for c in classes], dtype=float)
    y_after = np.array([data[c]["after"] for c in classes], dtype=float)

    # Before scatter (red)
    ax.scatter(x_log, y_before, s=130, color="#E74C3C", edgecolor="darkred",
               linewidth=1.5, alpha=0.7, zorder=3, label="Before CGC", marker="o")

    # After scatter (green)
    ax.scatter(x_log, y_after, s=130, color="#27AE60", edgecolor="darkgreen",
               linewidth=1.5, alpha=0.7, zorder=3, label="After CGC", marker="s")

    # Draw arrows from before to after
    for x, yb, ya in zip(x_log, y_before, y_after):
        ax.annotate("", xy=(x, ya), xytext=(x, yb),
                    xycoords="data",
                    arrowprops=dict(arrowstyle="->", lw=1.2, color="gray", alpha=0.5))

    # Label classes
    for x, yb, ya, cls in zip(x_log, y_before, y_after, classes):
        y_mid = (yb + ya) / 2
        ax.text(x + 0.10, y_mid, cls, fontsize=10, va="center", fontweight="bold")

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel(r"$\log_{10}(n_{\mathrm{train}})$", fontsize=11)
    ax.set_ylabel("Intra-class similarity", fontsize=11)
    ax.set_ylim(-0.05, 1.1)
    ax.set_xlim(0.4, 4.0)
    ax.grid(True, linestyle=":", linewidth=0.5, alpha=0.5)
    ax.legend(frameon=True, fontsize=11, loc="lower right")

def main(out_path="intra_similarity_before_after_4plots.pdf"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    axes = axes.flatten()

    plot_before_after_overlay(axes[0], "FUSARShip - SARDet-100K", fusar_sardet)
    plot_before_after_overlay(axes[1], "FUSARShip - SAR-JEPA", fusar_jepa)
    plot_before_after_overlay(axes[2], "OpenSARShip - SARDet-100K", opensar_sardet)
    plot_before_after_overlay(axes[3], "OpenSARShip - SAR-JEPA", opensar_jepa)

    # fig.suptitle("Per-class intra-class similarity: before vs after CGC",
                #  fontsize=14, fontweight="bold", y=0.995)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import io

# 1. Load the data
data = """Dataset	Class	Before CGC	After CGC	Delta	Train Count
Fusarship-SARDet-100k	Bulk	0.3999	0.9660	0.5661	224
Fusarship-SARDet-100k	Cargo	0.0219	0.9575	0.9355	1360
Fusarship-SARDet-100k	Container	0.2140	0.9515	0.7374	56
Fusarship-SARDet-100k	Dredging	0.5878	0.9681	0.3803	56
Fusarship-SARDet-100k	Fishing	0.0872	0.9539	0.8667	632
Fusarship-SARDet-100k	GeneralCargo	0.1982	0.9634	0.7652	40
Fusarship-SARDet-100k	Passenger	0.6667	0.9765	0.3099	40
Fusarship-SARDet-100k	Tanker	0.6812	0.9731	0.2919	128
Fusarship-SARDet-100k	Tug	0.5122	0.9731	0.4609	48
Fusarship-sarjepa	Bulk	0.3999	0.9084	0.5085	224
Fusarship-sarjepa	Cargo	0.0219	0.7959	0.7740	1360
Fusarship-sarjepa	Container	0.2140	0.8931	0.6791	56
Fusarship-sarjepa	Dredging	0.5878	0.9076	0.3198	56
Fusarship-sarjepa	Fishing	0.0872	0.8386	0.7514	632
Fusarship-sarjepa	GeneralCargo	0.1982	0.9128	0.7146	40
Fusarship-sarjepa	Passenger	0.6667	0.9229	0.2563	40
Fusarship-sarjepa	Tanker	0.6812	0.8908	0.2096	128
Fusarship-sarjepa	Tug	0.5122	0.9041	0.3919	48
OpenSARShip-SARDet-100k	Cargo	0.0116	0.9670	0.9555	8496
OpenSARShip-SARDet-100k	Dredging	0.0424	0.9731	0.9307	232
OpenSARShip-SARDet-100k	Fishing	0.1905	0.9490	0.7586	232
OpenSARShip-SARDet-100k	Passenger	0.0623	0.9654	0.9031	112
OpenSARShip-SARDet-100k	Tanker	0.0238	0.9671	0.9432	2920
OpenSARShip-SARDet-100k	Tug	0.1104	0.9586	0.8482	104
OpenSARShip-sarjepa	Cargo	0.0042	0.9793	0.9751	8496
OpenSARShip-sarjepa	Dredging	0.0519	0.9820	0.9301	232
OpenSARShip-sarjepa	Fishing	0.3849	0.9712	0.5863	232
OpenSARShip-sarjepa	Passenger	0.1952	0.9791	0.7840	112
OpenSARShip-sarjepa	Tanker	0.0359	0.9800	0.9441	2920
OpenSARShip-sarjepa	Tug	0.1211	0.9801	0.8589	104"""

df = pd.read_csv(io.StringIO(data), sep='\t')
datasets = df['Dataset'].unique()

# Set a beautiful, readable theme
sns.set_theme(style="whitegrid", context="talk")

# ==========================================
# PLOT 1: Dumbbell Plot (Highly Recommended)
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharex=True)
fig.suptitle("Effect of CGC on Per-Class Intra-Sim (Before vs After)", fontsize=22, fontweight='bold', y=1.02)
axes = axes.flatten()

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    # Sort values so the longest improvements look organized
    df_sub = df[df['Dataset'] == ds].sort_values('After CGC')

    # Draw line connecting before and after (The 'Delta')
    ax.hlines(y=df_sub['Class'], xmin=df_sub['Before CGC'], xmax=df_sub['After CGC'],
              color='grey', alpha=0.5, linewidth=3, zorder=1)

    # Draw Before Dots
    ax.scatter(df_sub['Before CGC'], df_sub['Class'], color='crimson', label='Before CGC', s=150, zorder=2)
    # Draw After Dots
    ax.scatter(df_sub['After CGC'], df_sub['Class'], color='mediumseagreen', label='After CGC', s=150, zorder=2)

    # Formatting
    ax.set_title(ds, fontsize=16, fontweight='bold')
    ax.set_xlabel("Intra-Sim Score" if idx >= 2 else "")
    ax.set_ylabel("")
    ax.set_xlim(0, 1.05)

    # Only add legend to the first plot to avoid clutter
    if idx == 0:
        ax.legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.savefig("dumbbell_plot.png", bbox_inches='tight', dpi=300)
plt.show()

# ==========================================
# PLOT 2: Horizontal Grouped Bar Chart
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharey=False)
fig.suptitle("Comparison of Intra-Sim Scores Before and After CGC", fontsize=22, fontweight='bold', y=1.02)
axes = axes.flatten()

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    # Melt dataframe to make it suitable for seaborn barplot
    df_sub = df[df['Dataset'] == ds].melt(id_vars=['Class'],
                                          value_vars=['Before CGC', 'After CGC'],
                                          var_name='Condition', value_name='Score')

    sns.barplot(data=df_sub, x='Score', y='Class', hue='Condition', ax=ax,
                palette=['crimson', 'mediumseagreen'])

    ax.set_title(ds, fontsize=16, fontweight='bold')
    ax.set_xlabel("Intra-Sim Score" if idx >= 2 else "")
    ax.set_ylabel("")
    ax.set_xlim(0, 1.05)

    # Manage legends
    if idx != 0:
        ax.get_legend().remove()
    else:
        ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig("grouped_barplot.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
import os

# ── Change this to your preferred output folder ───────────────────────
SAVE_DIR = "."          # current directory  (change to e.g. "/content/" on Colab)
os.makedirs(SAVE_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, name + ".pdf"), bbox_inches="tight", dpi=200)
    fig.savefig(os.path.join(SAVE_DIR, name + ".png"), bbox_inches="tight", dpi=200)
    print(f"  Saved {name}.pdf / .png  →  {SAVE_DIR}/")
    plt.close(fig)

# ── Colour palette ────────────────────────────────────────────────────
C_BEFORE       = "#9DB4C0"
C_AFTER        = "#2D6A4F"
C_ARROW        = "#52B788"
C_DELTA_ACCENT = "#E76F51"
BG             = "#FAFAFA"
GRID_COL       = "#E0E0E0"

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.facecolor":   BG,
    "figure.facecolor": BG,
    "axes.grid":        True,
    "grid.color":       GRID_COL,
    "grid.linewidth":   0.6,
    "axes.edgecolor":   "#AAAAAA",
    "xtick.color":      "#555555",
    "ytick.color":      "#555555",
    "axes.labelcolor":  "#333333",
    "text.color":       "#333333",
})

# ── Data ──────────────────────────────────────────────────────────────
datasets = {
    "FuSARShip — SARDet-100k": {
        "class":       ["Bulk","Cargo","Container","Dredging","Fishing","GeneralCargo","Passenger","Tanker","Tug"],
        "before":      [0.3999,0.0219,0.2140,0.5878,0.0872,0.1982,0.6667,0.6812,0.5122],
        "after":       [0.9660,0.9575,0.9515,0.9681,0.9539,0.9634,0.9765,0.9731,0.9731],
        "train_count": [224,1360,56,56,632,40,40,128,48],
        "test_count":  [28,170,7,7,79,5,5,16,6],
    },
    "FuSARShip — SARJepa": {
        "class":       ["Bulk","Cargo","Container","Dredging","Fishing","GeneralCargo","Passenger","Tanker","Tug"],
        "before":      [0.3999,0.0219,0.2140,0.5878,0.0872,0.1982,0.6667,0.6812,0.5122],
        "after":       [0.9084,0.7959,0.8931,0.9076,0.8386,0.9128,0.9229,0.8908,0.9041],
        "train_count": [224,1360,56,56,632,40,40,128,48],
        "test_count":  [28,170,7,7,79,5,5,16,6],
    },
    "OpenSARShip — SARDet-100k": {
        "class":       ["Cargo","Dredging","Fishing","Passenger","Tanker","Tug"],
        "before":      [0.0116,0.0424,0.1905,0.0623,0.0238,0.1104],
        "after":       [0.9670,0.9731,0.9490,0.9654,0.9671,0.9586],
        "train_count": [8496,232,232,112,2920,104],
        "test_count":  [1062,29,29,14,365,13],
    },
    "OpenSARShip — SARJepa": {
        "class":       ["Cargo","Dredging","Fishing","Passenger","Tanker","Tug"],
        "before":      [0.0042,0.0519,0.3849,0.1952,0.0359,0.1211],
        "after":       [0.9793,0.9820,0.9712,0.9791,0.9800,0.9801],
        "train_count": [8496,232,232,112,2920,104],
        "test_count":  [1062,29,29,14,365,13],
    },
}

# ══════════════════════════════════════════════════════════════════════
# FIGURE 1 — Dumbbell chart  (2 × 2 grid)
# ══════════════════════════════════════════════════════════════════════
fig1, axes = plt.subplots(2, 2, figsize=(16, 13))
fig1.patch.set_facecolor(BG)
axes = axes.flatten()

for ax_idx, (title, d) in enumerate(datasets.items()):
    ax = axes[ax_idx]
    ax.set_facecolor(BG)

    classes     = d["class"]
    before_vals = d["before"]
    after_vals  = d["after"]
    counts      = d["train_count"]
    n           = len(classes)

    # sort ascending by train count so minority classes appear at top
    order     = np.argsort(counts)
    classes_s = [classes[i] for i in order]
    before_s  = [before_vals[i] for i in order]
    after_s   = [after_vals[i] for i in order]
    counts_s  = [counts[i] for i in order]

    y_pos = np.arange(n)

    for y, b, a in zip(y_pos, before_s, after_s):
        ax.plot([b, a], [y, y],
                color=C_ARROW, linewidth=2.2, alpha=0.7,
                solid_capstyle="round", zorder=2)

    ax.scatter(before_s, y_pos, s=90, color=C_BEFORE, zorder=4,
               linewidths=1.2, edgecolors="white")
    ax.scatter(after_s,  y_pos, s=90, color=C_AFTER,  zorder=4,
               linewidths=1.2, edgecolors="white")

    for y, b, a, cnt in zip(y_pos, before_s, after_s, counts_s):
        delta = a - b
        ax.text((b + a) / 2, y + 0.28, f"+{delta:.2f}",
                ha="center", va="bottom", fontsize=7.5,
                color=C_DELTA_ACCENT, fontweight="bold")
        ax.text(1.04, y, f"n={cnt}",
                ha="left", va="center", fontsize=7, color="#888888",
                transform=ax.get_yaxis_transform())

    ax.set_yticks(y_pos)
    ax.set_yticklabels(classes_s, fontsize=9)
    ax.set_xlim(-0.05, 1.10)
    ax.set_xlabel("Intra-Class Similarity", fontsize=10)
    ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}"))
    ax.tick_params(axis="both", length=0)

    if ax_idx == 0:
        ax.legend(
            handles=[
                mlines.Line2D([], [], color=C_BEFORE, marker="o", linestyle="None",
                              markersize=8, label="Before CGC"),
                mlines.Line2D([], [], color=C_AFTER,  marker="o", linestyle="None",
                              markersize=8, label="After CGC"),
                mlines.Line2D([], [], color=C_ARROW, linewidth=2,
                              label="Improvement (+Δ)"),
            ],
            loc="lower right", fontsize=8.5,
            framealpha=0.85, edgecolor=GRID_COL,
        )

fig1.suptitle(
    "Per-Class Intra-Similarity Before and After CGC\n"
    "Classes ordered by training count — fewest at top",
    fontsize=13, fontweight="bold", y=1.01,
)
fig1.tight_layout(pad=2.5, h_pad=3.5, w_pad=3.5)
save(fig1, "fig1_dumbbell")


# # ══════════════════════════════════════════════════════════════════════
# # FIGURE 2 — Scatter: Δ vs baseline  |  Δ vs log(train count)
# # ══════════════════════════════════════════════════════════════════════
# rows = []
# for title, d in datasets.items():
#     for cls, b, a, tr, te in zip(d["class"], d["before"], d["after"],
#                                   d["train_count"], d["test_count"]):
#         rows.append({"Dataset": title, "Class": cls,
#                      "Before": b, "After": a, "Delta": a - b,
#                      "TrainCount": tr, "TestCount": te})
# df = pd.DataFrame(rows)

# ds_colors = {
#     "FuSARShip — SARDet-100k":  "#264653",
#     "FuSARShip — SARJepa":      "#E9A820",
#     "OpenSARShip — SARDet-100k":"#E76F51",
#     "OpenSARShip — SARJepa":    "#2A9D8F",
# }

# fig2, (axA, axB) = plt.subplots(1, 2, figsize=(15, 6))
# fig2.patch.set_facecolor(BG)

# # Panel A — Δ vs before
# for title, grp in df.groupby("Dataset"):
#     col = ds_colors[title]
#     axA.scatter(grp["Before"], grp["Delta"],
#                 s=80, color=col, alpha=0.85,
#                 edgecolors="white", linewidths=0.8, zorder=3,
#                 label=title)
#     for _, row in grp.iterrows():
#         axA.annotate(row["Class"],
#                      (row["Before"], row["Delta"]),
#                      textcoords="offset points", xytext=(5, 3),
#                      fontsize=7.5, color=col, alpha=0.9)

# axA.set_xlabel("Intra-Similarity  Before CGC", fontsize=11)
# axA.set_ylabel("Δ Intra-Similarity  (After − Before)", fontsize=11)
# axA.set_title("(a)  Improvement vs. Baseline Similarity", fontsize=11, fontweight="bold")
# axA.tick_params(length=0)
# axA.legend(fontsize=8, framealpha=0.9, edgecolor=GRID_COL,
#            title="Dataset — Model", title_fontsize=8)

# # Panel B — Δ vs log10(train count)
# for title, grp in df.groupby("Dataset"):
#     col = ds_colors[title]
#     axB.scatter(np.log10(grp["TrainCount"]), grp["Delta"],
#                 s=80, color=col, alpha=0.85,
#                 edgecolors="white", linewidths=0.8, zorder=3,
#                 label=title)
#     for _, row in grp.iterrows():
#         axB.annotate(row["Class"],
#                      (np.log10(row["TrainCount"]), row["Delta"]),
#                      textcoords="offset points", xytext=(5, 3),
#                      fontsize=7.5, color=col, alpha=0.9)

# # trend line
# lc = np.log10(df["TrainCount"])
# z  = np.polyfit(lc, df["Delta"], 1)
# p  = np.poly1d(z)
# xs = np.linspace(lc.min(), lc.max(), 200)
# axB.plot(xs, p(xs), "--", color="#555555", linewidth=1.4,
#          label=f"Trend  (slope = {z[0]:+.3f})", zorder=2)

# xt = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
# axB.set_xticks(xt)
# axB.set_xticklabels([f"$10^{{{v}}}$" for v in xt], fontsize=9)
# axB.set_xlabel("Training Samples  (log scale)", fontsize=11)
# axB.set_ylabel("Δ Intra-Similarity  (After − Before)", fontsize=11)
# axB.set_title("(b)  Improvement vs. Training Set Size", fontsize=11, fontweight="bold")
# axB.tick_params(length=0)
# axB.legend(fontsize=8, framealpha=0.9, edgecolor=GRID_COL,
#            title="Dataset — Model", title_fontsize=8)

# fig2.suptitle(
#     "CGC Improvement as a Function of Baseline Quality and Class Frequency",
#     fontsize=13, fontweight="bold",
# )
# fig2.tight_layout(pad=2.5, w_pad=4.0)
# save(fig2, "fig2_scatter")


# # ══════════════════════════════════════════════════════════════════════
# # FIGURE 3 — Before / After heatmaps  (2 rows × 2 cols)
# # ══════════════════════════════════════════════════════════════════════
# fusar_classes = ["Bulk","Cargo","Container","Dredging","Fishing",
#                  "GeneralCargo","Passenger","Tanker","Tug"]
# open_classes  = ["Cargo","Dredging","Fishing","Passenger","Tanker","Tug"]

# combo_keys    = list(datasets.keys())
# fusar_combos  = combo_keys[:2]
# open_combos   = combo_keys[2:]

# short_labels = {
#     "FuSARShip — SARDet-100k":  "SARDet-100k",
#     "FuSARShip — SARJepa":      "SARJepa",
#     "OpenSARShip — SARDet-100k":"SARDet-100k",
#     "OpenSARShip — SARJepa":    "SARJepa",
# }

# def build_matrix(class_list, combo_list, value_key):
#     mat = np.full((len(class_list), len(combo_list)), np.nan)
#     for j, combo in enumerate(combo_list):
#         d = datasets[combo]
#         for i, cls in enumerate(class_list):
#             if cls in d["class"]:
#                 idx = d["class"].index(cls)
#                 mat[i, j] = d[value_key][idx]
#     return mat

# fig3, axs = plt.subplots(2, 2, figsize=(12, 10))
# fig3.patch.set_facecolor(BG)

# sub_configs = [
#     (0, 0, fusar_classes, fusar_combos,  "before", "FuSARShip — Before CGC",  plt.cm.YlOrRd),
#     (0, 1, fusar_classes, fusar_combos,  "after",  "FuSARShip — After CGC",   plt.cm.YlGn),
#     (1, 0, open_classes,  open_combos,   "before", "OpenSARShip — Before CGC",plt.cm.YlOrRd),
#     (1, 1, open_classes,  open_combos,   "after",  "OpenSARShip — After CGC", plt.cm.YlGn),
# ]

# for r, c, cls_list, combo_list, vkey, subtitle, cmap in sub_configs:
#     ax  = axs[r, c]
#     mat = build_matrix(cls_list, combo_list, vkey)
#     im  = ax.imshow(mat, cmap=cmap, vmin=0.0, vmax=1.0,
#                     aspect="auto", interpolation="nearest")

#     for i in range(mat.shape[0]):
#         for j in range(mat.shape[1]):
#             val = mat[i, j]
#             if not np.isnan(val):
#                 txt_col = "white" if val > 0.60 else "#333333"
#                 ax.text(j, i, f"{val:.2f}", ha="center", va="center",
#                         fontsize=10, fontweight="bold", color=txt_col)
#             else:
#                 ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
#                              fill=True, color="#EEEEEE", zorder=2))
#                 ax.text(j, i, "—", ha="center", va="center",
#                         fontsize=10, color="#BBBBBB")

#     ax.set_xticks(range(len(combo_list)))
#     ax.set_xticklabels([short_labels[x] for x in combo_list], fontsize=10)
#     ax.set_yticks(range(len(cls_list)))
#     ax.set_yticklabels(cls_list, fontsize=9)
#     ax.set_title(subtitle, fontsize=10.5, fontweight="bold", pad=6)
#     ax.tick_params(length=0)
#     plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
#                  label="Intra-Similarity")

# fig3.suptitle(
#     "Per-Class Intra-Similarity Before and After CGC",
#     fontsize=13, fontweight="bold", y=1.01,
# )
# fig3.tight_layout(pad=2.5, h_pad=3.5, w_pad=3.5)
# save(fig3, "fig3_heatmap")

print("\nDone — all three figures saved to:", os.path.abspath(SAVE_DIR))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data: train counts and intra-similarity before and after
fusar_sardet = {
    "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9660},
    "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.9575},
    "Container":   {"train": 56, "before": 0.2140, "after": 0.9515},
    "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9681},
    "Fishing":     {"train": 632, "before": 0.0872, "after": 0.9539},
    "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9634},
    "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9765},
    "Tanker":      {"train": 128, "before": 0.6812, "after": 0.9731},
    "Tug":         {"train": 48, "before": 0.5122, "after": 0.9731},
}

fusar_jepa = {
    "Bulk":        {"train": 224, "before": 0.3999, "after": 0.9084},
    "Cargo":       {"train": 1360, "before": 0.0219, "after": 0.7959},
    "Container":   {"train": 56, "before": 0.2140, "after": 0.8931},
    "Dredging":    {"train": 56, "before": 0.5878, "after": 0.9076},
    "Fishing":     {"train": 632, "before": 0.0872, "after": 0.8386},
    "GeneralCargo":{"train": 40, "before": 0.1982, "after": 0.9128},
    "Passenger":   {"train": 40, "before": 0.6667, "after": 0.9229},
    "Tanker":      {"train": 128, "before": 0.6812, "after": 0.8908},
    "Tug":         {"train": 48, "before": 0.5122, "after": 0.9041},
}

opensar_sardet = {
    "Cargo":     {"train": 8496, "before": 0.0116, "after": 0.9670},
    "Dredging":  {"train": 232, "before": 0.0424, "after": 0.9731},
    "Fishing":   {"train": 232, "before": 0.1905, "after": 0.9490},
    "Passenger": {"train": 112, "before": 0.0623, "after": 0.9654},
    "Tanker":    {"train": 2920, "before": 0.0238, "after": 0.9671},
    "Tug":       {"train": 104, "before": 0.1104, "after": 0.9586},
}

opensar_jepa = {
    "Cargo":     {"train": 8496, "before": 0.0042, "after": 0.9793},
    "Dredging":  {"train": 232, "before": 0.0519, "after": 0.9820},
    "Fishing":   {"train": 232, "before": 0.3849, "after": 0.9712},
    "Passenger": {"train": 112, "before": 0.1952, "after": 0.9791},
    "Tanker":    {"train": 2920, "before": 0.0359, "after": 0.9800},
    "Tug":       {"train": 104, "before": 0.1211, "after": 0.9801},
}

def dumbbell_plot(ax, title, data):
    """Create a dumbbell (connected dot) plot."""
    classes = list(data.keys())
    # Sort by training count (minority first)
    classes = sorted(classes, key=lambda c: data[c]["train"])

    y_pos = np.arange(len(classes))

    before = np.array([data[c]["before"] for c in classes], dtype=float)
    after = np.array([data[c]["after"] for c in classes], dtype=float)

    # Draw connecting lines
    for i, (b, a) in enumerate(zip(before, after)):
        ax.plot([b, a], [i, i], color="gray", linewidth=2.5, zorder=1, alpha=0.6)

    # Plot before dots (red, left)
    ax.scatter(before, y_pos, s=140, color="#E74C3C", edgecolor="darkred",
               linewidth=1.5, zorder=3, label="Before CGC", alpha=0.8)

    # Plot after dots (green, right)
    ax.scatter(after, y_pos, s=140, color="#27AE60", edgecolor="darkgreen",
               linewidth=1.5, zorder=3, label="After CGC", alpha=0.8)

    # Add class labels on the left
    ax.set_yticks(y_pos)
    ax.set_yticklabels(classes, fontsize=11)

    # Add value labels on the dots
    for i, (b, a) in enumerate(zip(before, after)):
        ax.text(b - 0.04, i, f"{b:.3f}", ha="right", va="center", fontsize=9, fontweight="bold")
        ax.text(a + 0.04, i, f"{a:.3f}", ha="left", va="center", fontsize=9, fontweight="bold")

    ax.set_xlabel("Intra-class similarity", fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlim(-0.15, 1.15)
    ax.grid(True, axis="x", linestyle=":", linewidth=0.5, alpha=0.5)
    ax.legend(frameon=True, fontsize=11, loc="lower right")

def main(out_path="intra_similarity_dumbbell_4plots.pdf"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    axes = axes.flatten()

    dumbbell_plot(axes[0], "FUSARShip - SARDet-100K", fusar_sardet)
    dumbbell_plot(axes[1], "FUSARShip - SAR-JEPA", fusar_jepa)
    dumbbell_plot(axes[2], "OpenSARShip - SARDet-100K", opensar_sardet)
    dumbbell_plot(axes[3], "OpenSARShip - SAR-JEPA", opensar_jepa)

    fig.suptitle("Per-class intra-class similarity: before vs after CGC",
                 fontsize=14, fontweight="bold", y=0.995)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {out_path}")

if __name__ == "__main__":
    main()

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np
import pandas as pd
import os

# ── Change this to your preferred output folder ───────────────────────
SAVE_DIR = "."
os.makedirs(SAVE_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, name + ".pdf"), bbox_inches="tight", dpi=300, transparent=True)
    fig.savefig(os.path.join(SAVE_DIR, name + ".png"), bbox_inches="tight", dpi=300, transparent=True)
    print(f"  Saved {name}.pdf / .png  →  {SAVE_DIR}/")
    plt.close(fig)

# ── Colour palette ────────────────────────────────────────────────────
C_BEFORE       = "#9DB4C0"
C_AFTER        = "#2D6A4F"
C_ARROW        = "#52B788"
C_DELTA_ACCENT = "#E76F51"
BG             = "#FAFAFA"
GRID_COL       = "#E0E0E0"

plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    # "axes.facecolor":   BG,
    # "figure.facecolor": BG,
    "axes.grid":        True,
    "grid.color":       GRID_COL,
    "grid.linewidth":   0.6,
    "axes.edgecolor":   "#AAAAAA",
    "xtick.color":      "#555555",
    "ytick.color":      "#555555",
    "axes.labelcolor":  "#333333",
    "text.color":       "#333333",
})

# ── Data ──────────────────────────────────────────────────────────────
datasets = {
    "FuSARShip — SARDet-100k": {
        "class":       ["Bulk","Cargo","Container","Dredging","Fishing","GeneralCargo","Passenger","Tanker","Tug"],
        "before":      [0.2417,	0.0069,	0.1170,	0.2061,	0.0717,	0.1307,	0.2398,	0.2906,	0.1791],
        "after":       [0.9342,	0.8555,	0.9113,	0.8537,	0.8983,	0.9370,	0.8799,	0.8924,	0.8785],
        "train_count": [224,1360,56,56,632,40,40,128,48],
        "test_count":  [28,170,7,7,79,5,5,16,6],
    },
    "FuSARShip — SARJepa": {
        "class":       ["Bulk","Cargo","Container","Dredging","Fishing","GeneralCargo","Passenger","Tanker","Tug"],
        "before":      [0.3999,	0.0456,	0.2140,	0.5878,	0.0966,	0.1982,	0.6667,	0.6812,	0.5122],
        "after":       [0.9655,	0.9569,	0.9507,	0.9678,	0.9529,	0.9639,	0.9770,	0.9741,	0.9736],
        "train_count": [224,1360,56,56,632,40,40,128,48],
        "test_count":  [28,170,7,7,79,5,5,16,6],
    },
    "OpenSARShip — SARDet-100k": {
        "class":       ["Cargo","Dredging","Fishing","Passenger","Tanker","Tug"],
        "before":      [0.0071,0.0424,0.1905,0.0623,0.0223,0.1104],
        "after":       [0.9654,0.9726,0.9476,0.9638,0.9662,0.9576],
        "train_count": [8496,232,232,112,2920,104],
        "test_count":  [1062,29,29,14,365,13],
    },
    "OpenSARShip — SARJepa": {
        "class":       ["Cargo","Dredging","Fishing","Passenger","Tanker","Tug"],
        "before":      [0.0069,0.0519,0.3849,0.1952,0.0240,0.1211],
        "after":       [0.9781,0.9816,0.9711,0.9787,0.9795,0.9797],
        "train_count": [8496,232,232,112,2920,104],
        "test_count":  [1062,29,29,14,365,13],
    },
}

def dumbbell_grid(datasets, titles, fig_name, suptitle, figsize=(16, 13)):
    """
    Create a 2x2 dumbbell grid with flexible titles.

    Parameters:
    -----------
    datasets : dict
        Dictionary of dataset data (as defined above)
    titles : list or dict
        List of 4 titles (order: same as dict insertion order)
        or dict mapping dataset keys to custom titles
    fig_name : str
        Output filename (without extension)
    suptitle : str
        Figure super title
    figsize : tuple
        Figure size (width, height)
    """

    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.patch.set_facecolor(BG)
    axes = axes.flatten()

    # Convert titles to list if it's a dict
    if isinstance(titles, dict):
        titles_list = [titles.get(k, k) for k in datasets.keys()]
    else:
        titles_list = titles

    for ax_idx, ((dataset_key, d), custom_title) in enumerate(zip(datasets.items(), titles_list)):
        ax = axes[ax_idx]
        ax.set_facecolor(BG)

        classes     = d["class"]
        before_vals = d["before"]
        after_vals  = d["after"]
        counts      = d["train_count"]
        n           = len(classes)

        # Sort by train count (minority first)
        order     = np.argsort(counts)
        classes_s = [classes[i] for i in order]
        before_s  = [before_vals[i] for i in order]
        after_s   = [after_vals[i] for i in order]
        counts_s  = [counts[i] for i in order]

        y_pos = np.arange(n)

        # Draw connecting lines
        for y, b, a in zip(y_pos, before_s, after_s):
            ax.plot([b, a], [y, y],
                    color=C_ARROW, linewidth=2.2, alpha=0.7,
                    solid_capstyle="round", zorder=2)

        # Draw scatter points
        ax.scatter(before_s, y_pos, s=90, color=C_BEFORE, zorder=4,
                   linewidths=1.2, edgecolors="white")
        ax.scatter(after_s,  y_pos, s=90, color=C_AFTER,  zorder=4,
                   linewidths=1.2, edgecolors="white")

        # Add delta labels and train counts
        for y, b, a, cnt in zip(y_pos, before_s, after_s, counts_s):
            delta = a - b
            ax.text((b + a) / 2, y + 0.18, f"+{delta:.2f}",
                    ha="center", va="bottom", fontsize=14,
                    color=C_DELTA_ACCENT, fontweight="bold")
            # ax.text(1.04, y, f"n={cnt}",
            #         ha="right", va="center", fontsize=7, color="#888888",
            #         transform=ax.get_yaxis_transform())

        ax.set_yticks(y_pos)
        ax.set_yticklabels(classes_s, fontsize=16)
        ax.set_xlim(-0.15, 1.10)
        ax.set_xlabel("Intra-Class Similarity", fontsize=16)
        ax.set_title(custom_title, fontsize=18, fontweight="bold", pad=12)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.1f}"))
        ax.tick_params(axis="both", length=0)

        # Legend only on first subplot
        if ax_idx == 0:
            ax.legend(
                handles=[
                    mlines.Line2D([], [], color=C_BEFORE, marker="o", linestyle="None",
                                  markersize=8, label="Before CGC"),
                    mlines.Line2D([], [], color=C_AFTER,  marker="o", linestyle="None",
                                  markersize=8, label="After CGC"),
                    mlines.Line2D([], [], color=C_ARROW, linewidth=2,
                                  label="Improvement (+Δ)"),
                ],
                loc="lower left", fontsize=14,
                framealpha=0.65, edgecolor=GRID_COL,
            )

    # fig.suptitle(suptitle, fontsize=13, fontweight="bold", y=1.01)
    # fig.tight_layout(pad=2.5, h_pad=2.5, w_pad=3.5)
    save(fig, fig_name)
    # plt.show()

# ══════════════════════════════════════════════════════════════════════
# Call with custom titles
# ══════════════════════════════════════════════════════════════════════

# Option 1: Pass titles as a list (in same order as dict keys)
custom_titles = [
    "(a) FS + SARDet-100K",
    "(b) FS + SAR-JEPA",
    "(c) OS + SARDet-100K",
    "(d) OS + SAR-JEPA",
]

dumbbell_grid(
    datasets,
    titles=custom_titles,
    fig_name="fig_intra_similarity_dumbbell",
    suptitle="Per-Class Intra-Similarity Before and After CGC\nClasses ordered by training count — fewest at top"
)

# Option 2 (alternative): Pass titles as a dict mapping dataset keys to custom titles
# custom_titles_dict = {
#     "FuSARShip — SARDet-100k": "(a) Custom Title 1",
#     "FuSARShip — SARJepa": "(b) Custom Title 2",
#     "OpenSARShip — SARDet-100k": "(c) Custom Title 3",
#     "OpenSARShip — SARJepa": "(d) Custom Title 4",
# }
# dumbbell_grid(datasets, titles=custom_titles_dict, fig_name="fig_intra_similarity_dumbbell", suptitle="...")import io
